# External validation, calibration, and transportability

**Public role:** Core SMDG/PAPILA external data-policy and inference notebook.

**Relation to the manuscript:** Supports source-wise SMDG transportability, label harmonization, and frozen temperature-scaling analyses.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `13_external_validation_dataset_selection_and_preparation.ipynb`


In [ ]:
# Cell 1: AQPR-SMDG direct setup, fixed metadata schema, robust path indexing, and external audit contract

import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
AI_ROOT = Path(os.environ.get("AQPR_DATA_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma_Data"))

RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"
MODELS_ROOT = JOURNAL_ROOT / "models"

RESULTS_07_DIR = RESULTS_ROOT / "07_train_or_load_hygd_glaucoma_classifiers_patient_safe"
RESULTS_08_DIR = RESULTS_ROOT / "08_anatomical_explainability_validation"
RESULTS_09_DIR = RESULTS_ROOT / "09_anatomy_ablation_and_shortcut_controls"
RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
RESULTS_13_DIR.mkdir(parents=True, exist_ok=True)

SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"
SMDG_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

SMDG_IMAGE_CACHE_DIR = SMDG_PROJECT_DATA_DIR / "external_inputs_384"
SMDG_IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Direct SMDG dataset location
# ---------------------------------------------------------------------

SMDG_SELECTED_ROOT = (
    JOURNAL_ROOT
    / "project_data"
    / "external_datasets"
    / "SMDG"
)

SMDG_METADATA_CANDIDATES = [
    SMDG_SELECTED_ROOT / "metadata - standardized.csv",
    SMDG_SELECTED_ROOT / "metada - standardized.csv",
    SMDG_SELECTED_ROOT / "metadata.csv",
]

existing_smdg_metadata_candidates = [
    path
    for path in SMDG_METADATA_CANDIDATES
    if path.exists()
]

if not SMDG_SELECTED_ROOT.exists():
    raise FileNotFoundError(
        f"SMDG root directory does not exist: {SMDG_SELECTED_ROOT}"
    )

if len(existing_smdg_metadata_candidates) == 0:
    raise FileNotFoundError(
        "No SMDG standardized metadata file found. Checked: "
        f"{[str(path) for path in SMDG_METADATA_CANDIDATES]}"
    )

SMDG_SELECTED_METADATA_CSV = existing_smdg_metadata_candidates[0]


# ---------------------------------------------------------------------
# Fixed SMDG metadata schema after direct CSV inspection
# ---------------------------------------------------------------------

SMDG_ID_COLUMN = "names"
SMDG_TYPE_COLUMN = "types"
SMDG_TYPE_EXPANDED_COLUMN = "type_expanded"
SMDG_FUNDUS_COLUMN = "fundus"
SMDG_OD_MASK_COLUMN = "fundus_od_seg"
SMDG_OC_MASK_COLUMN = "fundus_oc_seg"
SMDG_PATIENT_ID_COLUMN = "patient_id"
SMDG_EYE_COLUMN = "eye"

SMDG_DERIVED_SOURCE_COLUMN = "smdg_source_dataset"

SMDG_KNOWN_SOURCE_PREFIXES = [
    "OIA-ODIR-TEST-OFFLINE",
    "OIA-ODIR-TEST-ONLINE",
    "OIA-ODIR-TRAIN",
    "EyePACS-Glaucoma",
    "REFUGE1-train",
    "REFUGE1-val",
    "DRISHTI-GS1",
    "sjchoi86-HRF",
    "JSIEC-1000",
    "DR-HAGIS",
    "CRFO-v4",
    "LES-AV",
    "G1020",
    "ORIGA",
    "BEH",
    "PAPILA",
    "FIVES",
    "HRF",
]


# ---------------------------------------------------------------------
# AQPR frozen model panel and upstream notebook inputs
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_AQPR_MODEL_NAMES = [
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]

AQPR_PANEL_MODEL_NAMES = [
    PRIMARY_MODEL_NAME,
    *SECONDARY_AQPR_MODEL_NAMES,
]

AQPR_PANEL_ROLES = {
    PRIMARY_MODEL_NAME: "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}

FROZEN_PANEL_DIR = (
    MODELS_ROOT
    / "airogs_light_glaucoma_classifiers"
    / "frozen_aqpr_model_panel_notebook07"
)

NOTEBOOK07_AQPR_MODEL_PANEL_CSV = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.csv"
)

NOTEBOOK07_AQPR_MODEL_PANEL_JSON = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.json"
)

NOTEBOOK08_FINAL_STATUS_JSON = (
    RESULTS_08_DIR
    / "notebook08_final_status.json"
)

NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON = (
    RESULTS_08_DIR
    / "notebook08_downstream_handoff.json"
)

NOTEBOOK09_FINAL_STATUS_JSON = (
    RESULTS_09_DIR
    / "notebook09_final_status.json"
)

NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_downstream_handoff.json"
)


# ---------------------------------------------------------------------
# Notebook 13 outputs
# ---------------------------------------------------------------------

SMDG_CANDIDATE_DIRS_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_candidate_directories.csv"
)

SMDG_CANDIDATE_METADATA_FILES_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_candidate_metadata_files.csv"
)

SMDG_SELECTED_METADATA_PREVIEW_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_selected_metadata_preview.csv"
)

SMDG_METADATA_COLUMN_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_metadata_column_audit.csv"
)

SMDG_LABEL_POLICY_PRELIMINARY_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_preliminary_label_policy_audit.csv"
)

SMDG_SOURCE_DISTRIBUTION_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_distribution.csv"
)

SMDG_SOURCE_LABEL_DISTRIBUTION_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_label_distribution.csv"
)

SMDG_FILE_INVENTORY_SUMMARY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_file_inventory_summary.csv"
)

SMDG_PATH_EXISTENCE_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_path_existence_audit.csv"
)

SMDG_MISSING_PATH_PREVIEW_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_missing_path_preview.csv"
)

SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_metadata_with_derived_columns.csv"
)

SMDG_EXTERNAL_AUDIT_CONTRACT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_external_audit_contract.csv"
)

SMDG_SETUP_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_setup_summary.json"
)


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

SEED = 42
np.random.seed(SEED)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
}

TABLE_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".xlsx",
    ".xls",
    ".json",
    ".txt",
}

PRUNE_DIR_NAMES = {
    ".git",
    "__pycache__",
    "results",
    "models",
    "checkpoints",
    "runs",
    "logs",
    "figures",
    "wandb",
    "gradcam",
    "gradcamplusplus",
    "integrated_gradients",
    "probability_maps",
    "overlays",
}


def normalize_text(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def normalize_relative_path_key(value):
    value_str = str(value).strip().replace("\\", "/")
    value_str = value_str.lstrip("/")
    value_str = re.sub(r"^\./", "", value_str)
    value_str = re.sub(r"/+", "/", value_str)
    return value_str


def as_bool_series(series):
    if series.dtype == bool:
        return series

    lowered = series.astype(str).str.strip().str.lower()

    return lowered.isin(
        [
            "true",
            "1",
            "yes",
            "y",
            "sim",
            "si",
            "pass",
            "passed",
        ]
    )


def require_columns(df, required_columns, table_name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def read_json(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_table_flexible(path, nrows=None):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path, nrows=nrows, low_memory=False)

    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", nrows=nrows, low_memory=False)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, nrows=nrows)

    if suffix == ".json":
        data = pd.read_json(path)

        if nrows is not None:
            data = data.head(nrows)

        return data

    if suffix == ".txt":
        try:
            return pd.read_csv(path, sep=None, engine="python", nrows=nrows)
        except Exception:
            return pd.read_csv(path, sep="\t", nrows=nrows)

    raise ValueError(f"Unsupported table format: {path}")


def safe_sample_values(series, max_values=12):
    values = (
        series
        .dropna()
        .astype(str)
        .map(lambda x: x[:120])
        .unique()
        .tolist()
    )

    return values[:max_values]


def is_missing_path_value(value):
    if pd.isna(value):
        return True

    value_str = str(value).strip()

    if value_str == "":
        return True

    if value_str.lower() in [
        "nan",
        "none",
        "null",
        "not visible",
        "not_visible",
        "not available",
        "not_available",
        "missing",
    ]:
        return True

    return False


def extract_smdg_source_from_name(name):
    name_str = str(name)

    for prefix in SMDG_KNOWN_SOURCE_PREFIXES:
        if name_str.startswith(prefix + "-") or name_str == prefix:
            return prefix

    fallback = re.sub(r"-\d+$", "", name_str)

    return fallback


def preliminary_binary_label_from_types(value):
    if pd.isna(value):
        return np.nan

    try:
        value_int = int(value)
    except Exception:
        return np.nan

    if value_int == 1:
        return 1

    if value_int == 0:
        return 0

    return np.nan


def preliminary_label_policy_status(value):
    if pd.isna(value):
        return "missing_type"

    try:
        value_int = int(value)
    except Exception:
        return "non_numeric_type"

    if value_int == 1:
        return "binary_positive_candidate"

    if value_int == 0:
        return "binary_negative_candidate"

    if value_int == -1:
        return "ambiguous_or_suspect_excluded_candidate"

    return "unexpected_type_value"


def label_expanded_warning(value):
    if pd.isna(value):
        return "type_expanded_missing"

    value_norm = normalize_text(value)

    if "suspect" in value_norm:
        return "suspect_label_requires_exclusion_or_separate_policy"

    if "unknown" in value_norm:
        return "unknown_glaucoma_subtype_requires_policy_review"

    if "referable" in value_norm:
        return "referable_glaucoma_positive"

    if "poag" in value_norm or "ntg" in value_norm:
        return "clinical_glaucoma_positive"

    if "simple_chronic" in value_norm or "chronic" in value_norm:
        return "clinical_glaucoma_positive"

    if "healthy" in value_norm:
        return "healthy_negative"

    return "unclassified_expanded_text_review"


def bounded_smdg_inventory(root, max_files=100000):
    rows = []
    n_files = 0

    for current_root, dirnames, filenames in os.walk(root):
        dirnames[:] = [
            dirname
            for dirname in dirnames
            if dirname not in PRUNE_DIR_NAMES
            and not dirname.startswith(".")
        ]

        current_root_path = Path(current_root)

        for filename in filenames:
            n_files += 1

            path = current_root_path / filename
            suffix = path.suffix.lower()
            rel_path = path.relative_to(root)
            rel_norm = normalize_relative_path_key(rel_path)
            parent_norm = normalize_text(str(path.parent.relative_to(root)))
            name_norm = normalize_text(path.name)

            if suffix in IMAGE_EXTENSIONS:
                file_family = "image"
            elif suffix in TABLE_EXTENSIONS:
                file_family = "table_or_metadata"
            else:
                file_family = "other"

            mask_like = bool(
                any(
                    term in parent_norm or term in name_norm
                    for term in [
                        "optic_disc",
                        "optic_cup",
                        "optic_disc",
                        "optic_cup",
                        "disc",
                        "cup",
                        "seg",
                        "segmentation",
                        "mask",
                        "contour",
                    ]
                )
            )

            metadata_like = bool(
                file_family == "table_or_metadata"
                and any(
                    term in name_norm
                    for term in [
                        "metadata",
                        "metada",
                        "standardized",
                        "label",
                        "diagnosis",
                        "smdg",
                    ]
                )
            )

            rows.append(
                {
                    "path": str(path),
                    "relative_path": str(rel_path),
                    "relative_path_normalized": rel_norm,
                    "basename": path.name,
                    "parent_dir": str(path.parent),
                    "suffix": suffix,
                    "file_family": file_family,
                    "mask_like": mask_like,
                    "metadata_like": metadata_like,
                    "file_size_bytes": int(path.stat().st_size) if path.exists() else np.nan,
                }
            )

            if n_files >= max_files:
                break

        if n_files >= max_files:
            break

    return pd.DataFrame(rows)


def build_smdg_path_indexes(file_inventory):
    path_by_relative = {}
    path_by_basename = {}

    for _, row in file_inventory.iterrows():
        rel_key = str(row["relative_path_normalized"])
        abs_path = str(row["path"])
        basename = str(row["basename"])

        path_by_relative[rel_key] = abs_path
        path_by_basename.setdefault(basename, []).append(abs_path)

    return path_by_relative, path_by_basename


def filter_basename_candidates_by_family(candidates, expected_family):
    if len(candidates) == 0:
        return []

    expected_family = str(expected_family)

    filtered = []

    for candidate in candidates:
        candidate_norm = normalize_text(candidate)

        if expected_family == "fundus":
            if (
                "full_fundus" in candidate_norm
                or "fundus" in candidate_norm
            ):
                filtered.append(candidate)

        elif expected_family == "od_mask":
            if (
                "optic_disc" in candidate_norm
                or "od_seg" in candidate_norm
                or "disc" in candidate_norm
            ):
                filtered.append(candidate)

        elif expected_family == "oc_mask":
            if (
                "optic_cup" in candidate_norm
                or "oc_seg" in candidate_norm
                or "cup" in candidate_norm
            ):
                filtered.append(candidate)

    return filtered


def resolve_smdg_relative_path(value, expected_family):
    if is_missing_path_value(value):
        return None

    value_str = str(value).strip().replace("\\", "/")

    raw_candidate = Path(value_str)

    if raw_candidate.is_absolute() and raw_candidate.exists():
        return raw_candidate

    relative_key = normalize_relative_path_key(value_str)

    direct_candidate = SMDG_SELECTED_ROOT / relative_key

    if direct_candidate.exists():
        return direct_candidate

    # Some metadata values are absolute-looking paths such as /full-fundus/x.png.
    # The normalized key above already removes the leading slash.
    path_by_relative = globals().get("SMDG_PATH_BY_RELATIVE", {})
    path_by_basename = globals().get("SMDG_PATH_BY_BASENAME", {})

    if relative_key in path_by_relative:
        return Path(path_by_relative[relative_key])

    basename = Path(relative_key).name

    basename_candidates = path_by_basename.get(basename, [])
    family_candidates = filter_basename_candidates_by_family(
        candidates=basename_candidates,
        expected_family=expected_family,
    )

    if len(family_candidates) == 1:
        return Path(family_candidates[0])

    if len(basename_candidates) == 1:
        return Path(basename_candidates[0])

    return direct_candidate


def resolve_smdg_path_to_string(value, expected_family):
    resolved = resolve_smdg_relative_path(
        value=value,
        expected_family=expected_family,
    )

    return str(resolved) if resolved is not None else ""


# ---------------------------------------------------------------------
# Validate upstream AQPR context
# ---------------------------------------------------------------------

required_upstream_inputs = [
    NOTEBOOK07_AQPR_MODEL_PANEL_CSV,
    NOTEBOOK08_FINAL_STATUS_JSON,
    NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON,
    NOTEBOOK09_FINAL_STATUS_JSON,
    NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON,
]

missing_upstream_inputs = [
    str(path)
    for path in required_upstream_inputs
    if not Path(path).exists()
]

if missing_upstream_inputs:
    raise FileNotFoundError(
        "Missing upstream AQPR outputs. Notebook 13 must start after Notebooks 07, 08, and 09 are closed. "
        f"Missing: {missing_upstream_inputs}"
    )

frozen_aqpr_panel_manifest = pd.read_csv(NOTEBOOK07_AQPR_MODEL_PANEL_CSV)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
    ],
    "Notebook 07 frozen AQPR model panel manifest",
)

frozen_aqpr_panel_manifest["model_name"] = frozen_aqpr_panel_manifest["model_name"].astype(str)

observed_panel_models = frozen_aqpr_panel_manifest["model_name"].tolist()

if observed_panel_models != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR model panel order/content. "
        f"Expected {AQPR_PANEL_MODEL_NAMES}, found {observed_panel_models}"
    )

for _, row in frozen_aqpr_panel_manifest.iterrows():
    checkpoint_path = Path(str(row["checkpoint_path"]))

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing frozen AQPR checkpoint for {row['model_name']}: {checkpoint_path}"
        )

notebook08_status = read_json(NOTEBOOK08_FINAL_STATUS_JSON)
notebook08_handoff = read_json(NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON)
notebook09_status = read_json(NOTEBOOK09_FINAL_STATUS_JSON)
notebook09_handoff = read_json(NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON)

for label, payload in [
    ("Notebook 08", notebook08_status),
    ("Notebook 09", notebook09_status),
]:
    status = str(payload.get("status", "")).upper()

    if status != "PASS":
        raise RuntimeError(f"{label} does not report PASS. Found status={status}")


# ---------------------------------------------------------------------
# Load and validate SMDG metadata
# ---------------------------------------------------------------------

start_time = time.time()

smdg_metadata_raw = read_table_flexible(SMDG_SELECTED_METADATA_CSV)
smdg_metadata_raw = smdg_metadata_raw.copy()

if len(smdg_metadata_raw) == 0:
    raise RuntimeError(f"Selected SMDG metadata is empty: {SMDG_SELECTED_METADATA_CSV}")

required_smdg_columns = [
    SMDG_ID_COLUMN,
    SMDG_TYPE_COLUMN,
    SMDG_TYPE_EXPANDED_COLUMN,
    SMDG_FUNDUS_COLUMN,
    SMDG_OD_MASK_COLUMN,
    SMDG_OC_MASK_COLUMN,
    SMDG_PATIENT_ID_COLUMN,
    SMDG_EYE_COLUMN,
]

require_columns(
    smdg_metadata_raw,
    required_smdg_columns,
    "SMDG standardized metadata",
)

smdg_metadata_raw.head(100).to_csv(SMDG_SELECTED_METADATA_PREVIEW_CSV, index=False)


# ---------------------------------------------------------------------
# Direct SMDG file inventory and path indexes
# ---------------------------------------------------------------------

smdg_file_inventory = bounded_smdg_inventory(SMDG_SELECTED_ROOT)

if len(smdg_file_inventory) == 0:
    raise RuntimeError(f"SMDG root contains no files: {SMDG_SELECTED_ROOT}")

SMDG_PATH_BY_RELATIVE, SMDG_PATH_BY_BASENAME = build_smdg_path_indexes(
    smdg_file_inventory
)

print(f"SMDG path index entries: {len(SMDG_PATH_BY_RELATIVE)}")
print(f"SMDG basename index entries: {len(SMDG_PATH_BY_BASENAME)}")

file_inventory_summary = (
    smdg_file_inventory
    .groupby(["file_family", "suffix"], dropna=False)
    .agg(
        n_files=("path", "count"),
        n_mask_like=("mask_like", "sum"),
        n_metadata_like=("metadata_like", "sum"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
    .reset_index()
    .sort_values(["file_family", "suffix"])
)

file_inventory_summary.to_csv(SMDG_FILE_INVENTORY_SUMMARY_CSV, index=False)

candidate_dirs = pd.DataFrame(
    [
        {
            "dataset_alias": "SMDG",
            "candidate_dir": str(SMDG_SELECTED_ROOT),
            "search_root": str(SMDG_SELECTED_ROOT.parent),
            "path_depth": len(SMDG_SELECTED_ROOT.parts),
            "exists": SMDG_SELECTED_ROOT.exists(),
            "discovery_reason": "direct_user_confirmed_path",
            "n_files": int(len(smdg_file_inventory)),
            "n_images": int((smdg_file_inventory["file_family"] == "image").sum()),
            "n_tables": int((smdg_file_inventory["file_family"] == "table_or_metadata").sum()),
            "n_mask_like_files": int(smdg_file_inventory["mask_like"].sum()),
            "n_metadata_like_files": int(smdg_file_inventory["metadata_like"].sum()),
        }
    ]
)

candidate_dirs.to_csv(SMDG_CANDIDATE_DIRS_CSV, index=False)

candidate_metadata_files = pd.DataFrame(
    [
        {
            "metadata_path": str(path),
            "parent_dir": str(path.parent),
            "exists": path.exists(),
            "file_name": path.name,
            "suffix": path.suffix.lower(),
            "discovery_reason": "direct_smdg_metadata_candidate",
            "selected": bool(path == SMDG_SELECTED_METADATA_CSV),
        }
        for path in SMDG_METADATA_CANDIDATES
    ]
)

candidate_metadata_files.to_csv(SMDG_CANDIDATE_METADATA_FILES_CSV, index=False)


# ---------------------------------------------------------------------
# Column audit
# ---------------------------------------------------------------------

FIXED_SEMANTIC_ROLE_BY_COLUMN = {
    SMDG_ID_COLUMN: "standardized_image_id",
    SMDG_TYPE_COLUMN: "primary_numeric_label_column",
    SMDG_TYPE_EXPANDED_COLUMN: "expanded_label_text_column",
    "isColor": "image_modality_flag",
    "original_name": "original_source_filename_or_identifier",
    SMDG_PATIENT_ID_COLUMN: "partial_patient_identifier",
    SMDG_FUNDUS_COLUMN: "relative_fundus_image_path",
    SMDG_OD_MASK_COLUMN: "relative_native_od_mask_path",
    SMDG_OC_MASK_COLUMN: "relative_native_oc_mask_path",
    "bv_seg": "relative_blood_vessel_mask_path",
    "artery_seg": "relative_artery_mask_path",
    "vein_seg": "relative_vein_mask_path",
    SMDG_EYE_COLUMN: "eye_laterality",
    "vcdr": "vertical_cup_disc_ratio",
    "cdr_avg": "cup_disc_ratio_average",
}

column_audit_rows = []

for col in smdg_metadata_raw.columns:
    series = smdg_metadata_raw[col]

    column_audit_rows.append(
        {
            "column_name": col,
            "normalized_column_name": normalize_text(col),
            "dtype": str(series.dtype),
            "n_rows": int(len(series)),
            "n_non_null": int(series.notna().sum()),
            "missing_fraction": float(series.isna().mean()),
            "n_unique_non_null": int(series.dropna().astype(str).nunique()),
            "fixed_semantic_role": FIXED_SEMANTIC_ROLE_BY_COLUMN.get(col, "not_assigned_in_cell1"),
            "sample_values": json.dumps(safe_sample_values(series, max_values=12), ensure_ascii=False),
        }
    )

metadata_column_audit = pd.DataFrame(column_audit_rows)
metadata_column_audit.to_csv(SMDG_METADATA_COLUMN_AUDIT_CSV, index=False)


# ---------------------------------------------------------------------
# Derived SMDG metadata columns for downstream AQPR cells
# ---------------------------------------------------------------------

smdg_metadata = smdg_metadata_raw.copy()

smdg_metadata["smdg_row_index"] = np.arange(len(smdg_metadata), dtype=int)
smdg_metadata["smdg_sample_id"] = smdg_metadata[SMDG_ID_COLUMN].astype(str)

smdg_metadata[SMDG_DERIVED_SOURCE_COLUMN] = smdg_metadata[SMDG_ID_COLUMN].apply(
    extract_smdg_source_from_name
)

smdg_metadata["smdg_type_numeric"] = pd.to_numeric(
    smdg_metadata[SMDG_TYPE_COLUMN],
    errors="coerce",
)

smdg_metadata["smdg_binary_label_candidate"] = smdg_metadata[SMDG_TYPE_COLUMN].apply(
    preliminary_binary_label_from_types
)

smdg_metadata["smdg_label_policy_status_preliminary"] = smdg_metadata[SMDG_TYPE_COLUMN].apply(
    preliminary_label_policy_status
)

smdg_metadata["smdg_type_expanded_warning"] = smdg_metadata[SMDG_TYPE_EXPANDED_COLUMN].apply(
    label_expanded_warning
)

smdg_metadata["smdg_fundus_abs_path"] = smdg_metadata[SMDG_FUNDUS_COLUMN].apply(
    lambda value: resolve_smdg_path_to_string(value, expected_family="fundus")
)

smdg_metadata["smdg_od_mask_abs_path"] = smdg_metadata[SMDG_OD_MASK_COLUMN].apply(
    lambda value: resolve_smdg_path_to_string(value, expected_family="od_mask")
)

smdg_metadata["smdg_oc_mask_abs_path"] = smdg_metadata[SMDG_OC_MASK_COLUMN].apply(
    lambda value: resolve_smdg_path_to_string(value, expected_family="oc_mask")
)

smdg_metadata["smdg_fundus_exists"] = smdg_metadata["smdg_fundus_abs_path"].apply(
    lambda value: Path(value).exists() if str(value).strip() != "" else False
)

smdg_metadata["smdg_od_mask_exists"] = smdg_metadata["smdg_od_mask_abs_path"].apply(
    lambda value: Path(value).exists() if str(value).strip() != "" else False
)

smdg_metadata["smdg_oc_mask_exists"] = smdg_metadata["smdg_oc_mask_abs_path"].apply(
    lambda value: Path(value).exists() if str(value).strip() != "" else False
)

smdg_metadata["smdg_has_native_od_oc_masks"] = (
    smdg_metadata["smdg_od_mask_exists"].astype(bool)
    & smdg_metadata["smdg_oc_mask_exists"].astype(bool)
)

smdg_metadata["smdg_patient_id_available"] = smdg_metadata[SMDG_PATIENT_ID_COLUMN].notna()
smdg_metadata["smdg_eye_available"] = smdg_metadata[SMDG_EYE_COLUMN].notna()

smdg_metadata.to_csv(SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV, index=False)


# ---------------------------------------------------------------------
# Preliminary source, label, and path audits
# ---------------------------------------------------------------------

source_distribution = (
    smdg_metadata[SMDG_DERIVED_SOURCE_COLUMN]
    .fillna("__MISSING__")
    .astype(str)
    .value_counts(dropna=False)
    .reset_index()
)

source_distribution.columns = [
    "smdg_source_dataset",
    "n_rows",
]

source_distribution["fraction"] = (
    source_distribution["n_rows"]
    / source_distribution["n_rows"].sum()
)

source_distribution.to_csv(SMDG_SOURCE_DISTRIBUTION_CSV, index=False)

label_policy_audit = (
    smdg_metadata
    .groupby(
        [
            "smdg_type_numeric",
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_label_policy_status_preliminary",
            "smdg_type_expanded_warning",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_sources=(SMDG_DERIVED_SOURCE_COLUMN, "nunique"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_native_od_oc_masks=("smdg_has_native_od_oc_masks", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "smdg_type_numeric",
            "n_rows",
        ],
        ascending=[True, False],
    )
)

label_policy_audit["fraction"] = (
    label_policy_audit["n_rows"]
    / len(smdg_metadata)
)

label_policy_audit.to_csv(SMDG_LABEL_POLICY_PRELIMINARY_AUDIT_CSV, index=False)

source_label_distribution = (
    smdg_metadata
    .groupby(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            "smdg_type_numeric",
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_label_policy_status_preliminary",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_native_od_oc_masks=("smdg_has_native_od_oc_masks", "sum"),
        n_patient_id_available=("smdg_patient_id_available", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            "smdg_type_numeric",
            "n_rows",
        ],
        ascending=[True, True, False],
    )
)

source_label_distribution.to_csv(SMDG_SOURCE_LABEL_DISTRIBUTION_CSV, index=False)

path_existence_audit = pd.DataFrame(
    [
        {
            "path_type": "fundus_image",
            "metadata_column": SMDG_FUNDUS_COLUMN,
            "n_non_null_references": int(smdg_metadata[SMDG_FUNDUS_COLUMN].notna().sum()),
            "n_existing_files": int(smdg_metadata["smdg_fundus_exists"].sum()),
            "n_missing_files": int(
                smdg_metadata[SMDG_FUNDUS_COLUMN].notna().sum()
                - smdg_metadata["smdg_fundus_exists"].sum()
            ),
            "existence_fraction": float(
                smdg_metadata["smdg_fundus_exists"].sum()
                / max(1, smdg_metadata[SMDG_FUNDUS_COLUMN].notna().sum())
            ),
        },
        {
            "path_type": "native_od_mask",
            "metadata_column": SMDG_OD_MASK_COLUMN,
            "n_non_null_references": int(smdg_metadata[SMDG_OD_MASK_COLUMN].notna().sum()),
            "n_existing_files": int(smdg_metadata["smdg_od_mask_exists"].sum()),
            "n_missing_files": int(
                smdg_metadata[SMDG_OD_MASK_COLUMN].notna().sum()
                - smdg_metadata["smdg_od_mask_exists"].sum()
            ),
            "existence_fraction": float(
                smdg_metadata["smdg_od_mask_exists"].sum()
                / max(1, smdg_metadata[SMDG_OD_MASK_COLUMN].notna().sum())
            ),
        },
        {
            "path_type": "native_oc_mask",
            "metadata_column": SMDG_OC_MASK_COLUMN,
            "n_non_null_references": int(smdg_metadata[SMDG_OC_MASK_COLUMN].notna().sum()),
            "n_existing_files": int(smdg_metadata["smdg_oc_mask_exists"].sum()),
            "n_missing_files": int(
                smdg_metadata[SMDG_OC_MASK_COLUMN].notna().sum()
                - smdg_metadata["smdg_oc_mask_exists"].sum()
            ),
            "existence_fraction": float(
                smdg_metadata["smdg_oc_mask_exists"].sum()
                / max(1, smdg_metadata[SMDG_OC_MASK_COLUMN].notna().sum())
            ),
        },
    ]
)

path_existence_audit.to_csv(SMDG_PATH_EXISTENCE_AUDIT_CSV, index=False)

missing_path_rows = []

missing_fundus = smdg_metadata.loc[
    ~smdg_metadata["smdg_fundus_exists"],
    [
        "smdg_sample_id",
        SMDG_DERIVED_SOURCE_COLUMN,
        SMDG_FUNDUS_COLUMN,
        "smdg_fundus_abs_path",
    ],
].head(50).copy()

if len(missing_fundus) > 0:
    missing_fundus["path_type"] = "fundus_image"
    missing_path_rows.append(missing_fundus)

missing_od = smdg_metadata.loc[
    smdg_metadata[SMDG_OD_MASK_COLUMN].notna()
    & (~smdg_metadata["smdg_od_mask_exists"]),
    [
        "smdg_sample_id",
        SMDG_DERIVED_SOURCE_COLUMN,
        SMDG_OD_MASK_COLUMN,
        "smdg_od_mask_abs_path",
    ],
].head(50).copy()

if len(missing_od) > 0:
    missing_od["path_type"] = "native_od_mask"
    missing_path_rows.append(missing_od)

missing_oc = smdg_metadata.loc[
    smdg_metadata[SMDG_OC_MASK_COLUMN].notna()
    & (~smdg_metadata["smdg_oc_mask_exists"]),
    [
        "smdg_sample_id",
        SMDG_DERIVED_SOURCE_COLUMN,
        SMDG_OC_MASK_COLUMN,
        "smdg_oc_mask_abs_path",
    ],
].head(50).copy()

if len(missing_oc) > 0:
    missing_oc["path_type"] = "native_oc_mask"
    missing_path_rows.append(missing_oc)

if len(missing_path_rows) > 0:
    missing_path_preview = pd.concat(
        missing_path_rows,
        axis=0,
        ignore_index=True,
        sort=False,
    )
else:
    missing_path_preview = pd.DataFrame(
        columns=[
            "smdg_sample_id",
            SMDG_DERIVED_SOURCE_COLUMN,
            "path_type",
        ]
    )

missing_path_preview.to_csv(SMDG_MISSING_PATH_PREVIEW_CSV, index=False)


# ---------------------------------------------------------------------
# Patient-level permissibility
# ---------------------------------------------------------------------

patient_non_null_fraction = float(
    smdg_metadata[SMDG_PATIENT_ID_COLUMN].notna().mean()
)

patient_unique_count = int(
    smdg_metadata[SMDG_PATIENT_ID_COLUMN].dropna().astype(str).nunique()
)

n_patient_id_available = int(smdg_metadata["smdg_patient_id_available"].sum())

patient_level_smdg_global_allowed = False

patient_level_smdg_subset_possible = bool(
    n_patient_id_available > 0
    and patient_unique_count > 1
)

patient_level_smdg_boundary = (
    "Global SMDG patient-level analysis is disallowed because patient_id is sparse. "
    "A source-restricted patient-level sensitivity analysis may be considered only for rows with validated patient_id."
)


# ---------------------------------------------------------------------
# External AQPR audit contract
# ---------------------------------------------------------------------

external_audit_contract = pd.DataFrame(
    [
        {
            "contract_item": "training_policy",
            "status": "locked",
            "rule": (
                "SMDG is external only. No model training, architecture selection, threshold tuning, "
                "augmentation selection, or temperature fitting may use SMDG."
            ),
        },
        {
            "contract_item": "primary_use",
            "status": "locked",
            "rule": (
                "SMDG is used for external image-level generalization, source-level domain shift, "
                "calibration stress testing, and label-policy audit."
            ),
        },
        {
            "contract_item": "binary_label_policy_preliminary",
            "status": "requires_cell2_lock",
            "rule": (
                "Preliminary mapping is types=1 as positive and types=0 as negative; types=-1 is excluded "
                "from binary evaluation and retained for label-policy audit. Cell 2 must lock this policy by source."
            ),
        },
        {
            "contract_item": "patient_level_scope",
            "status": "restricted",
            "rule": patient_level_smdg_boundary,
        },
        {
            "contract_item": "anatomical_analysis_scope",
            "status": "conditional",
            "rule": (
                "SMDG anatomical/XAI analysis is allowed only for images with valid native OD/OC masks or with "
                "segmentor-generated OD/OC proxies that pass QC."
            ),
        },
        {
            "contract_item": "label_policy_scope",
            "status": "required",
            "rule": (
                "SMDG labels must be audited by source/subdataset before binary mapping. Ambiguous, suspect, "
                "or unknown labels must not be silently mapped without explicit policy justification."
            ),
        },
        {
            "contract_item": "claim_boundary",
            "status": "locked",
            "rule": (
                "SMDG supports AQPR external reliability validation. It must not be presented as evidence "
                "that the frozen models were optimized for SMDG."
            ),
        },
    ]
)

external_audit_contract.to_csv(SMDG_EXTERNAL_AUDIT_CONTRACT_CSV, index=False)


# ---------------------------------------------------------------------
# Setup summary
# ---------------------------------------------------------------------

n_binary_positive_candidate = int(
    (smdg_metadata["smdg_binary_label_candidate"] == 1).sum()
)

n_binary_negative_candidate = int(
    (smdg_metadata["smdg_binary_label_candidate"] == 0).sum()
)

n_binary_excluded_candidate = int(
    smdg_metadata["smdg_binary_label_candidate"].isna().sum()
)

n_unexpected_type_values = int(
    (~smdg_metadata["smdg_type_numeric"].isin([-1, 0, 1])).sum()
)

elapsed_sec = round(time.time() - start_time, 3)

setup_summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 1",
    "status": "PASS",
    "analysis": "AQPR-SMDG direct setup with fixed standardized metadata schema and robust path resolution",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_dataset": "SMDG",
    "external_dataset_root": str(SMDG_SELECTED_ROOT),
    "external_dataset_metadata": str(SMDG_SELECTED_METADATA_CSV),
    "external_dataset_use": "external_validation_domain_shift_calibration_label_policy_audit",
    "model_tuning_on_smdg_allowed": False,
    "patient_level_smdg_global_analysis_allowed": patient_level_smdg_global_allowed,
    "patient_level_smdg_subset_possible_after_validation": patient_level_smdg_subset_possible,
    "patient_level_hygd_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_aqpr_panel_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "notebook08_status": notebook08_status.get("status", "unknown"),
    "notebook09_status": notebook09_status.get("status", "unknown"),
    "smdg_fixed_id_column": SMDG_ID_COLUMN,
    "smdg_fixed_source_column": SMDG_DERIVED_SOURCE_COLUMN,
    "smdg_fixed_label_column": SMDG_TYPE_COLUMN,
    "smdg_fixed_label_expanded_column": SMDG_TYPE_EXPANDED_COLUMN,
    "smdg_fixed_fundus_column": SMDG_FUNDUS_COLUMN,
    "smdg_fixed_od_mask_column": SMDG_OD_MASK_COLUMN,
    "smdg_fixed_oc_mask_column": SMDG_OC_MASK_COLUMN,
    "smdg_fixed_patient_id_column": SMDG_PATIENT_ID_COLUMN,
    "smdg_fixed_eye_column": SMDG_EYE_COLUMN,
    "n_smdg_files_inventory": int(len(smdg_file_inventory)),
    "n_smdg_images_inventory": int((smdg_file_inventory["file_family"] == "image").sum()),
    "n_smdg_table_files_inventory": int((smdg_file_inventory["file_family"] == "table_or_metadata").sum()),
    "n_smdg_mask_like_files_inventory": int(smdg_file_inventory["mask_like"].sum()),
    "n_smdg_metadata_like_files_inventory": int(smdg_file_inventory["metadata_like"].sum()),
    "n_smdg_path_index_relative_entries": int(len(SMDG_PATH_BY_RELATIVE)),
    "n_smdg_path_index_basename_entries": int(len(SMDG_PATH_BY_BASENAME)),
    "n_smdg_metadata_rows": int(len(smdg_metadata)),
    "n_smdg_metadata_columns": int(smdg_metadata_raw.shape[1]),
    "n_smdg_sources": int(smdg_metadata[SMDG_DERIVED_SOURCE_COLUMN].nunique()),
    "n_smdg_fundus_existing": int(smdg_metadata["smdg_fundus_exists"].sum()),
    "n_smdg_native_od_masks_existing": int(smdg_metadata["smdg_od_mask_exists"].sum()),
    "n_smdg_native_oc_masks_existing": int(smdg_metadata["smdg_oc_mask_exists"].sum()),
    "n_smdg_native_od_oc_mask_pairs_existing": int(smdg_metadata["smdg_has_native_od_oc_masks"].sum()),
    "n_smdg_binary_positive_candidate": n_binary_positive_candidate,
    "n_smdg_binary_negative_candidate": n_binary_negative_candidate,
    "n_smdg_binary_excluded_candidate": n_binary_excluded_candidate,
    "n_smdg_unexpected_type_values": n_unexpected_type_values,
    "patient_column_non_null_fraction": patient_non_null_fraction,
    "patient_column_unique_count": patient_unique_count,
    "n_patient_id_available": n_patient_id_available,
    "requires_cell2_label_policy_lock": True,
    "elapsed_sec": elapsed_sec,
    "candidate_dirs_csv": str(SMDG_CANDIDATE_DIRS_CSV),
    "candidate_metadata_files_csv": str(SMDG_CANDIDATE_METADATA_FILES_CSV),
    "selected_metadata_preview_csv": str(SMDG_SELECTED_METADATA_PREVIEW_CSV),
    "metadata_column_audit_csv": str(SMDG_METADATA_COLUMN_AUDIT_CSV),
    "preliminary_label_policy_audit_csv": str(SMDG_LABEL_POLICY_PRELIMINARY_AUDIT_CSV),
    "source_distribution_csv": str(SMDG_SOURCE_DISTRIBUTION_CSV),
    "source_label_distribution_csv": str(SMDG_SOURCE_LABEL_DISTRIBUTION_CSV),
    "file_inventory_summary_csv": str(SMDG_FILE_INVENTORY_SUMMARY_CSV),
    "path_existence_audit_csv": str(SMDG_PATH_EXISTENCE_AUDIT_CSV),
    "missing_path_preview_csv": str(SMDG_MISSING_PATH_PREVIEW_CSV),
    "metadata_with_derived_columns_csv": str(SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV),
    "external_audit_contract_csv": str(SMDG_EXTERNAL_AUDIT_CONTRACT_CSV),
}

with open(SMDG_SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("AQPR frozen model panel:")
print(
    frozen_aqpr_panel_manifest[
        [
            "model_name",
            "aqpr_panel_role",
            "checkpoint_path",
            "input_size",
            "batch_size",
            "validation_selected_threshold",
            "temperature",
        ]
    ].to_string(index=False)
)

print("\nDirect SMDG dataset:")
print(f"Root     : {SMDG_SELECTED_ROOT}")
print(f"Metadata : {SMDG_SELECTED_METADATA_CSV}")

print("\nFixed SMDG schema:")
print(f"ID column              : {SMDG_ID_COLUMN}")
print(f"Derived source column  : {SMDG_DERIVED_SOURCE_COLUMN}")
print(f"Numeric label column   : {SMDG_TYPE_COLUMN}")
print(f"Expanded label column  : {SMDG_TYPE_EXPANDED_COLUMN}")
print(f"Fundus path column     : {SMDG_FUNDUS_COLUMN}")
print(f"OD mask path column    : {SMDG_OD_MASK_COLUMN}")
print(f"OC mask path column    : {SMDG_OC_MASK_COLUMN}")
print(f"Patient ID column      : {SMDG_PATIENT_ID_COLUMN}")
print(f"Eye column             : {SMDG_EYE_COLUMN}")

print("\nSMDG file inventory summary:")
print(file_inventory_summary.to_string(index=False))

print("\nSMDG candidate metadata files:")
print(candidate_metadata_files.to_string(index=False))

print("\nSMDG source distribution:")
print(source_distribution.to_string(index=False))

print("\nPreliminary SMDG label-policy audit:")
print(label_policy_audit.to_string(index=False))

print("\nSMDG source-label distribution preview:")
print(source_label_distribution.head(100).to_string(index=False))

print("\nSMDG path existence audit:")
print(path_existence_audit.to_string(index=False))

if len(missing_path_preview) > 0:
    print("\nMissing path preview:")
    print(missing_path_preview.head(60).to_string(index=False))

print("\nExternal AQPR audit contract:")
print(external_audit_contract.to_string(index=False))

print("\nCell 1 setup summary:")
print(json.dumps(setup_summary, indent=2))

print(f"\nCandidate dirs CSV: {SMDG_CANDIDATE_DIRS_CSV}")
print(f"Candidate metadata files CSV: {SMDG_CANDIDATE_METADATA_FILES_CSV}")
print(f"Selected metadata preview CSV: {SMDG_SELECTED_METADATA_PREVIEW_CSV}")
print(f"Metadata column audit CSV: {SMDG_METADATA_COLUMN_AUDIT_CSV}")
print(f"Preliminary label-policy audit CSV: {SMDG_LABEL_POLICY_PRELIMINARY_AUDIT_CSV}")
print(f"Source distribution CSV: {SMDG_SOURCE_DISTRIBUTION_CSV}")
print(f"Source-label distribution CSV: {SMDG_SOURCE_LABEL_DISTRIBUTION_CSV}")
print(f"File inventory summary CSV: {SMDG_FILE_INVENTORY_SUMMARY_CSV}")
print(f"Path existence audit CSV: {SMDG_PATH_EXISTENCE_AUDIT_CSV}")
print(f"Missing path preview CSV: {SMDG_MISSING_PATH_PREVIEW_CSV}")
print(f"Metadata with derived columns CSV: {SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV}")
print(f"External audit contract CSV: {SMDG_EXTERNAL_AUDIT_CONTRACT_CSV}")
print(f"Setup summary JSON: {SMDG_SETUP_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if len(frozen_aqpr_panel_manifest) != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Frozen AQPR panel size mismatch.")

if not SMDG_SELECTED_ROOT.exists():
    raise RuntimeError(f"SMDG root does not exist: {SMDG_SELECTED_ROOT}")

if not SMDG_SELECTED_METADATA_CSV.exists():
    raise RuntimeError(f"Selected SMDG metadata does not exist: {SMDG_SELECTED_METADATA_CSV}")

if len(smdg_metadata) == 0:
    raise RuntimeError("Selected SMDG metadata has zero rows.")

if n_unexpected_type_values > 0:
    raise RuntimeError(
        f"SMDG contains unexpected type values outside -1/0/1: {n_unexpected_type_values}"
    )

if n_binary_positive_candidate <= 0:
    raise RuntimeError("SMDG has no preliminary positive binary candidates.")

if n_binary_negative_candidate <= 0:
    raise RuntimeError("SMDG has no preliminary negative binary candidates.")

if int(smdg_metadata["smdg_fundus_exists"].sum()) != len(smdg_metadata):
    missing_fundus_preview = smdg_metadata.loc[
        ~smdg_metadata["smdg_fundus_exists"],
        [
            "smdg_sample_id",
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_FUNDUS_COLUMN,
            "smdg_fundus_abs_path",
        ],
    ].head(30)

    print("\nMissing fundus preview:")
    print(missing_fundus_preview.to_string(index=False))

    raise RuntimeError(
        "Not all SMDG fundus images referenced in metadata exist after robust path indexing. "
        "Review the missing fundus preview above."
    )

if int(smdg_metadata["smdg_has_native_od_oc_masks"].sum()) <= 0:
    missing_mask_debug = smdg_metadata.loc[
        smdg_metadata[SMDG_OD_MASK_COLUMN].notna()
        | smdg_metadata[SMDG_OC_MASK_COLUMN].notna(),
        [
            "smdg_sample_id",
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_OD_MASK_COLUMN,
            "smdg_od_mask_abs_path",
            "smdg_od_mask_exists",
            SMDG_OC_MASK_COLUMN,
            "smdg_oc_mask_abs_path",
            "smdg_oc_mask_exists",
        ],
    ].head(30)

    print("\nNative OD/OC mask debug preview:")
    print(missing_mask_debug.to_string(index=False))

    raise RuntimeError(
        "No native OD/OC mask pairs were found in SMDG after robust path indexing. "
        "This would block native anatomical-subset analysis."
    )

if source_distribution["smdg_source_dataset"].nunique() <= 1:
    raise RuntimeError(
        "SMDG source extraction failed or found only one source. "
        "Source-level domain-shift audit requires source separation."
    )

for required_output in [
    SMDG_CANDIDATE_DIRS_CSV,
    SMDG_CANDIDATE_METADATA_FILES_CSV,
    SMDG_SELECTED_METADATA_PREVIEW_CSV,
    SMDG_METADATA_COLUMN_AUDIT_CSV,
    SMDG_LABEL_POLICY_PRELIMINARY_AUDIT_CSV,
    SMDG_SOURCE_DISTRIBUTION_CSV,
    SMDG_SOURCE_LABEL_DISTRIBUTION_CSV,
    SMDG_FILE_INVENTORY_SUMMARY_CSV,
    SMDG_PATH_EXISTENCE_AUDIT_CSV,
    SMDG_MISSING_PATH_PREVIEW_CSV,
    SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV,
    SMDG_EXTERNAL_AUDIT_CONTRACT_CSV,
    SMDG_SETUP_SUMMARY_JSON,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 1 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 2: Lock SMDG label policy and build AQPR external validation manifests
# PAPILA is excluded from all analytical manifests by design.

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
RESULTS_13_DIR.mkdir(parents=True, exist_ok=True)

SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"
SMDG_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Inputs from Cell 1
# ---------------------------------------------------------------------

SMDG_SETUP_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_setup_summary.json"
)

SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_metadata_with_derived_columns.csv"
)

SMDG_PRELIMINARY_LABEL_POLICY_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_preliminary_label_policy_audit.csv"
)

SMDG_SOURCE_LABEL_DISTRIBUTION_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_label_distribution.csv"
)

required_inputs = [
    SMDG_SETUP_SUMMARY_JSON,
    SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV,
    SMDG_PRELIMINARY_LABEL_POLICY_AUDIT_CSV,
    SMDG_SOURCE_LABEL_DISTRIBUTION_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Missing Cell 2 inputs from Cell 1: {missing_inputs}"
    )


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

SMDG_LOCKED_LABEL_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_locked_label_policy.csv"
)

SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_locked_label_policy_by_source.csv"
)

SMDG_SOURCE_SCOPE_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_scope_policy.csv"
)

SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_level_metric_eligibility.csv"
)

SMDG_LOCKED_METADATA_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_locked_external_metadata.csv"
)

SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_primary_external_image_level_manifest.csv"
)

SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_training_overlap_reference_manifest.csv"
)

SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_native_anatomical_subset_manifest.csv"
)

SMDG_EXCLUSION_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_external_exclusion_audit.csv"
)

SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_fully_excluded_source_audit.csv"
)

SMDG_CELL2_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_label_policy_and_manifest_summary.json"
)

SMDG_CELL2_SUMMARY_TXT = (
    RESULTS_13_DIR
    / "notebook13_SMDG_LABEL_POLICY_AND_MANIFEST_SUMMARY.txt"
)


# ---------------------------------------------------------------------
# Fixed SMDG schema
# ---------------------------------------------------------------------

SMDG_ID_COLUMN = "names"
SMDG_TYPE_COLUMN = "types"
SMDG_TYPE_EXPANDED_COLUMN = "type_expanded"
SMDG_FUNDUS_COLUMN = "fundus"
SMDG_OD_MASK_COLUMN = "fundus_od_seg"
SMDG_OC_MASK_COLUMN = "fundus_oc_seg"
SMDG_PATIENT_ID_COLUMN = "patient_id"
SMDG_EYE_COLUMN = "eye"

SMDG_DERIVED_SOURCE_COLUMN = "smdg_source_dataset"

PRIMARY_MODEL_TRAINING_DOMAIN = "EyePACS-AIROGS-light-V2"

# PAPILA is excluded from all analysis manifests in this SMDG-focused branch.
FULLY_EXCLUDED_SOURCES = {
    "PAPILA": (
        "excluded_from_all_analysis_manifests_due_to_user_scope_no_papila_in_current_aqpr_smdg_analysis"
    ),
}

# EyePACS-Glaucoma is not treated as an external validation source because of training-domain proximity.
# It is kept only as a separate reference artifact, not as primary external evidence.
REFERENCE_ONLY_SOURCES = {
    "EyePACS-Glaucoma": (
        "excluded_from_primary_external_manifest_due_to_training_domain_overlap_or_source_proximity"
    ),
}

MIN_SOURCE_POSITIVES_FOR_SOURCE_METRICS = 10
MIN_SOURCE_NEGATIVES_FOR_SOURCE_METRICS = 10


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_text(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def require_columns(df, required_columns, table_name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def coerce_bool_series(series):
    if series.dtype == bool:
        return series.fillna(False)

    return (
        series
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "sim", "si", "pass", "passed"])
    )


def locked_binary_label(value):
    if pd.isna(value):
        return np.nan

    try:
        value_int = int(value)
    except Exception:
        return np.nan

    if value_int == 1:
        return 1

    if value_int == 0:
        return 0

    return np.nan


def locked_label_policy_status(type_value):
    if pd.isna(type_value):
        return "excluded_missing_numeric_type"

    try:
        type_int = int(type_value)
    except Exception:
        return "excluded_non_numeric_type"

    if type_int == 1:
        return "included_binary_positive"

    if type_int == 0:
        return "included_binary_negative"

    if type_int == -1:
        return "excluded_ambiguous_or_suspect"

    return "excluded_unexpected_numeric_type"


def locked_label_name(type_value):
    if pd.isna(type_value):
        return "excluded"

    try:
        type_int = int(type_value)
    except Exception:
        return "excluded"

    if type_int == 1:
        return "glaucoma_or_referable_glaucoma"

    if type_int == 0:
        return "non_glaucoma_control"

    if type_int == -1:
        return "ambiguous_or_suspect"

    return "excluded"


def label_confidence_tier(type_value, expanded_value):
    status = locked_label_policy_status(type_value)

    if status == "excluded_ambiguous_or_suspect":
        return "excluded_ambiguous_or_suspect"

    if not status.startswith("included_binary"):
        return "excluded_invalid_type"

    expanded_norm = normalize_text(expanded_value) if not pd.isna(expanded_value) else ""

    if status == "included_binary_positive":
        if "referable" in expanded_norm:
            return "positive_referable_glaucoma_high_confidence"

        if (
            "poag" in expanded_norm
            or "ntg" in expanded_norm
            or "simple_chronic" in expanded_norm
            or "chronic" in expanded_norm
        ):
            return "positive_clinical_glaucoma_high_confidence"

        if "unknown_glaucoma" in expanded_norm:
            return "positive_numeric_label_accepted_unknown_subtype_flagged"

        if expanded_norm == "":
            return "positive_numeric_label_accepted_missing_expanded_text"

        return "positive_numeric_label_accepted_other_expanded_text"

    if status == "included_binary_negative":
        if "healthy" in expanded_norm:
            return "negative_healthy_high_confidence"

        if "unknown_glaucoma" in expanded_norm:
            return "negative_numeric_label_accepted_unknown_text_flagged"

        if expanded_norm == "":
            return "negative_numeric_label_accepted_missing_expanded_text"

        return "negative_numeric_label_accepted_other_expanded_text"

    return "unclassified"


def label_policy_sensitivity_flag(type_value, expanded_value):
    tier = label_confidence_tier(type_value, expanded_value)

    flagged_tiers = {
        "positive_numeric_label_accepted_unknown_subtype_flagged",
        "positive_numeric_label_accepted_missing_expanded_text",
        "negative_numeric_label_accepted_unknown_text_flagged",
        "negative_numeric_label_accepted_missing_expanded_text",
    }

    return tier in flagged_tiers


def source_analysis_scope(source_value):
    source_str = str(source_value)

    if source_str in FULLY_EXCLUDED_SOURCES:
        return "fully_excluded_from_all_analysis_manifests"

    if source_str in REFERENCE_ONLY_SOURCES:
        return "reference_only_not_primary_external"

    return "included_in_primary_smdg_external_manifest"


def source_scope_reason(source_value):
    source_str = str(source_value)

    if source_str in FULLY_EXCLUDED_SOURCES:
        return FULLY_EXCLUDED_SOURCES[source_str]

    if source_str in REFERENCE_ONLY_SOURCES:
        return REFERENCE_ONLY_SOURCES[source_str]

    return "eligible_primary_smdg_external_source"


def safe_string_column(df, column_name):
    if column_name in df.columns:
        return df[column_name].fillna("").astype(str)

    return pd.Series([""] * len(df), index=df.index)


def build_manifest_columns(df):
    manifest = pd.DataFrame(
        {
            "external_sample_id": df["smdg_sample_id"].fillna("").astype(str),
            "external_dataset": "SMDG",
            "external_source_dataset": df[SMDG_DERIVED_SOURCE_COLUMN].fillna("").astype(str),
            "external_image_path": df["smdg_fundus_abs_path"].fillna("").astype(str),
            "external_label": df["smdg_locked_binary_label"].astype(int),
            "external_label_name": df["smdg_locked_label_name"].fillna("").astype(str),
            "external_label_policy_status": df["smdg_locked_label_policy_status"].fillna("").astype(str),
            "external_label_confidence_tier": df["smdg_label_confidence_tier"].fillna("").astype(str),
            "external_label_policy_sensitivity_flag": df[
                "smdg_label_policy_sensitivity_flag"
            ].astype(bool),
            "external_patient_id": safe_string_column(df, SMDG_PATIENT_ID_COLUMN),
            "external_patient_id_available": df["smdg_patient_id_available"].astype(bool),
            "external_eye": safe_string_column(df, SMDG_EYE_COLUMN),
            "external_eye_available": df["smdg_eye_available"].astype(bool),
            "native_od_mask_path": df["smdg_od_mask_abs_path"].fillna("").astype(str),
            "native_oc_mask_path": df["smdg_oc_mask_abs_path"].fillna("").astype(str),
            "has_native_od_mask": df["smdg_od_mask_exists"].astype(bool),
            "has_native_oc_mask": df["smdg_oc_mask_exists"].astype(bool),
            "has_native_od_oc_masks": df["smdg_has_native_od_oc_masks"].astype(bool),
            "smdg_row_index": df["smdg_row_index"].astype(int),
            "smdg_original_name": safe_string_column(df, "original_name"),
            "smdg_type": df[SMDG_TYPE_COLUMN],
            "smdg_type_expanded": safe_string_column(df, SMDG_TYPE_EXPANDED_COLUMN),
            "source_analysis_scope": df["smdg_source_analysis_scope"].fillna("").astype(str),
            "source_scope_reason": df["smdg_source_scope_reason"].fillna("").astype(str),
        }
    )

    return manifest


# ---------------------------------------------------------------------
# Load Cell 1 outputs
# ---------------------------------------------------------------------

cell1_summary = load_json(SMDG_SETUP_SUMMARY_JSON)

if str(cell1_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 1 summary does not report PASS. Found status={cell1_summary.get('status')}"
    )

smdg_metadata = pd.read_csv(SMDG_METADATA_WITH_DERIVED_COLUMNS_CSV, low_memory=False)
preliminary_label_policy_audit = pd.read_csv(SMDG_PRELIMINARY_LABEL_POLICY_AUDIT_CSV)
source_label_distribution = pd.read_csv(SMDG_SOURCE_LABEL_DISTRIBUTION_CSV)

required_metadata_columns = [
    "smdg_row_index",
    "smdg_sample_id",
    SMDG_DERIVED_SOURCE_COLUMN,
    SMDG_TYPE_COLUMN,
    SMDG_TYPE_EXPANDED_COLUMN,
    SMDG_FUNDUS_COLUMN,
    "smdg_fundus_abs_path",
    "smdg_fundus_exists",
    SMDG_OD_MASK_COLUMN,
    SMDG_OC_MASK_COLUMN,
    "smdg_od_mask_abs_path",
    "smdg_oc_mask_abs_path",
    "smdg_od_mask_exists",
    "smdg_oc_mask_exists",
    "smdg_has_native_od_oc_masks",
    SMDG_PATIENT_ID_COLUMN,
    "smdg_patient_id_available",
    SMDG_EYE_COLUMN,
    "smdg_eye_available",
]

require_columns(
    smdg_metadata,
    required_metadata_columns,
    "SMDG derived metadata from Cell 1",
)

for bool_col in [
    "smdg_fundus_exists",
    "smdg_od_mask_exists",
    "smdg_oc_mask_exists",
    "smdg_has_native_od_oc_masks",
    "smdg_patient_id_available",
    "smdg_eye_available",
]:
    smdg_metadata[bool_col] = coerce_bool_series(smdg_metadata[bool_col])


# ---------------------------------------------------------------------
# Lock SMDG label policy
# ---------------------------------------------------------------------

smdg_locked = smdg_metadata.copy()

smdg_locked["smdg_locked_binary_label"] = smdg_locked[SMDG_TYPE_COLUMN].apply(
    locked_binary_label
)

smdg_locked["smdg_locked_label_policy_status"] = smdg_locked[SMDG_TYPE_COLUMN].apply(
    locked_label_policy_status
)

smdg_locked["smdg_locked_label_name"] = smdg_locked[SMDG_TYPE_COLUMN].apply(
    locked_label_name
)

smdg_locked["smdg_label_confidence_tier"] = [
    label_confidence_tier(type_value, expanded_value)
    for type_value, expanded_value in zip(
        smdg_locked[SMDG_TYPE_COLUMN],
        smdg_locked[SMDG_TYPE_EXPANDED_COLUMN],
    )
]

smdg_locked["smdg_label_policy_sensitivity_flag"] = [
    label_policy_sensitivity_flag(type_value, expanded_value)
    for type_value, expanded_value in zip(
        smdg_locked[SMDG_TYPE_COLUMN],
        smdg_locked[SMDG_TYPE_EXPANDED_COLUMN],
    )
]

smdg_locked["smdg_binary_label_policy_locked"] = smdg_locked[
    "smdg_locked_binary_label"
].isin([0, 1])

smdg_locked["smdg_source_analysis_scope"] = smdg_locked[
    SMDG_DERIVED_SOURCE_COLUMN
].apply(source_analysis_scope)

smdg_locked["smdg_source_scope_reason"] = smdg_locked[
    SMDG_DERIVED_SOURCE_COLUMN
].apply(source_scope_reason)

smdg_locked["smdg_fully_excluded_source"] = smdg_locked[
    "smdg_source_analysis_scope"
].eq("fully_excluded_from_all_analysis_manifests")

smdg_locked["smdg_reference_only_source"] = smdg_locked[
    "smdg_source_analysis_scope"
].eq("reference_only_not_primary_external")

smdg_locked["smdg_primary_external_eligible"] = (
    smdg_locked["smdg_binary_label_policy_locked"].astype(bool)
    & smdg_locked["smdg_fundus_exists"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
    & (~smdg_locked["smdg_reference_only_source"].astype(bool))
)

smdg_locked["smdg_training_overlap_reference_eligible"] = (
    smdg_locked["smdg_binary_label_policy_locked"].astype(bool)
    & smdg_locked["smdg_fundus_exists"].astype(bool)
    & smdg_locked["smdg_reference_only_source"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
)

smdg_locked["smdg_primary_native_anatomical_eligible"] = (
    smdg_locked["smdg_primary_external_eligible"].astype(bool)
    & smdg_locked["smdg_has_native_od_oc_masks"].astype(bool)
)

smdg_locked["smdg_exclusion_reason"] = "included_primary_external_analysis"

smdg_locked.loc[
    ~smdg_locked["smdg_fundus_exists"].astype(bool),
    "smdg_exclusion_reason",
] = "excluded_missing_fundus_file"

smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & smdg_locked["smdg_fully_excluded_source"].astype(bool),
    "smdg_exclusion_reason",
] = smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & smdg_locked["smdg_fully_excluded_source"].astype(bool),
    "smdg_source_scope_reason",
]

smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
    & (~smdg_locked["smdg_binary_label_policy_locked"].astype(bool)),
    "smdg_exclusion_reason",
] = smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
    & (~smdg_locked["smdg_binary_label_policy_locked"].astype(bool)),
    "smdg_locked_label_policy_status",
]

smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & smdg_locked["smdg_binary_label_policy_locked"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
    & smdg_locked["smdg_reference_only_source"].astype(bool),
    "smdg_exclusion_reason",
] = smdg_locked.loc[
    smdg_locked["smdg_fundus_exists"].astype(bool)
    & smdg_locked["smdg_binary_label_policy_locked"].astype(bool)
    & (~smdg_locked["smdg_fully_excluded_source"].astype(bool))
    & smdg_locked["smdg_reference_only_source"].astype(bool),
    "smdg_source_scope_reason",
]


# ---------------------------------------------------------------------
# Source-level policy and eligibility
# ---------------------------------------------------------------------

source_scope_policy = (
    smdg_locked
    .groupby(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            "smdg_source_analysis_scope",
            "smdg_source_scope_reason",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_binary_locked=("smdg_binary_label_policy_locked", "sum"),
        n_primary_external_eligible=("smdg_primary_external_eligible", "sum"),
        n_training_overlap_reference_eligible=("smdg_training_overlap_reference_eligible", "sum"),
        n_native_anatomical_eligible=("smdg_primary_native_anatomical_eligible", "sum"),
        n_patient_id_available=("smdg_patient_id_available", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "smdg_source_analysis_scope",
            "n_rows",
        ],
        ascending=[True, False],
    )
)

source_scope_policy.to_csv(SMDG_SOURCE_SCOPE_POLICY_CSV, index=False)

primary_rows_for_source = smdg_locked[
    smdg_locked["smdg_primary_external_eligible"].astype(bool)
].copy()

source_level_eligibility = (
    primary_rows_for_source
    .groupby(SMDG_DERIVED_SOURCE_COLUMN, dropna=False)
    .agg(
        n_primary_external_rows=("smdg_sample_id", "count"),
        n_positive=("smdg_locked_binary_label", lambda x: int((x == 1).sum())),
        n_negative=("smdg_locked_binary_label", lambda x: int((x == 0).sum())),
        n_native_anatomical_rows=("smdg_primary_native_anatomical_eligible", "sum"),
        n_label_policy_sensitivity_flag=("smdg_label_policy_sensitivity_flag", "sum"),
        n_patient_id_available=("smdg_patient_id_available", "sum"),
    )
    .reset_index()
)

source_level_eligibility["source_specific_binary_metric_eligible"] = (
    (source_level_eligibility["n_positive"] >= MIN_SOURCE_POSITIVES_FOR_SOURCE_METRICS)
    & (source_level_eligibility["n_negative"] >= MIN_SOURCE_NEGATIVES_FOR_SOURCE_METRICS)
)

source_level_eligibility["source_specific_anatomical_subset_eligible"] = (
    (source_level_eligibility["n_native_anatomical_rows"] > 0)
    & source_level_eligibility["source_specific_binary_metric_eligible"].astype(bool)
)

source_level_eligibility["metric_eligibility_reason"] = np.where(
    source_level_eligibility["source_specific_binary_metric_eligible"].astype(bool),
    "eligible_for_source_specific_binary_metrics",
    "insufficient_positive_or_negative_cases_for_source_specific_binary_metrics",
)

source_level_eligibility = source_level_eligibility.sort_values(
    [
        "source_specific_binary_metric_eligible",
        "n_primary_external_rows",
    ],
    ascending=[False, False],
).reset_index(drop=True)

source_level_eligibility.to_csv(SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV, index=False)


# ---------------------------------------------------------------------
# Locked label-policy tables
# ---------------------------------------------------------------------

locked_label_policy = (
    smdg_locked
    .groupby(
        [
            SMDG_TYPE_COLUMN,
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_locked_label_policy_status",
            "smdg_locked_label_name",
            "smdg_label_confidence_tier",
            "smdg_label_policy_sensitivity_flag",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_sources=(SMDG_DERIVED_SOURCE_COLUMN, "nunique"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_primary_external_eligible=("smdg_primary_external_eligible", "sum"),
        n_training_overlap_reference_eligible=("smdg_training_overlap_reference_eligible", "sum"),
        n_native_anatomical_eligible=("smdg_primary_native_anatomical_eligible", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            SMDG_TYPE_COLUMN,
            "n_rows",
        ],
        ascending=[True, False],
    )
)

locked_label_policy["fraction_all_smdg"] = (
    locked_label_policy["n_rows"]
    / len(smdg_locked)
)

locked_label_policy.to_csv(SMDG_LOCKED_LABEL_POLICY_CSV, index=False)

label_policy_by_source = (
    smdg_locked
    .groupby(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_TYPE_COLUMN,
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_locked_label_policy_status",
            "smdg_locked_label_name",
            "smdg_label_confidence_tier",
            "smdg_source_analysis_scope",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_primary_external_eligible=("smdg_primary_external_eligible", "sum"),
        n_training_overlap_reference_eligible=("smdg_training_overlap_reference_eligible", "sum"),
        n_native_anatomical_eligible=("smdg_primary_native_anatomical_eligible", "sum"),
        n_patient_id_available=("smdg_patient_id_available", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_TYPE_COLUMN,
            "n_rows",
        ],
        ascending=[True, True, False],
    )
)

label_policy_by_source.to_csv(SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV, index=False)


# ---------------------------------------------------------------------
# Build external manifests
# ---------------------------------------------------------------------

smdg_locked.to_csv(SMDG_LOCKED_METADATA_CSV, index=False)

primary_manifest = build_manifest_columns(
    smdg_locked[
        smdg_locked["smdg_primary_external_eligible"].astype(bool)
    ].copy()
)

training_overlap_reference_manifest = build_manifest_columns(
    smdg_locked[
        smdg_locked["smdg_training_overlap_reference_eligible"].astype(bool)
    ].copy()
)

native_anatomical_manifest = build_manifest_columns(
    smdg_locked[
        smdg_locked["smdg_primary_native_anatomical_eligible"].astype(bool)
    ].copy()
)

sort_cols = [
    "external_source_dataset",
    "external_label",
    "external_sample_id",
]

primary_manifest = primary_manifest.sort_values(sort_cols).reset_index(drop=True)
training_overlap_reference_manifest = (
    training_overlap_reference_manifest
    .sort_values(sort_cols)
    .reset_index(drop=True)
)
native_anatomical_manifest = (
    native_anatomical_manifest
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

primary_manifest.to_csv(SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV, index=False)
training_overlap_reference_manifest.to_csv(
    SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    index=False,
)
native_anatomical_manifest.to_csv(
    SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    index=False,
)


# ---------------------------------------------------------------------
# Exclusion audits
# ---------------------------------------------------------------------

exclusion_audit = (
    smdg_locked
    .groupby(
        [
            "smdg_exclusion_reason",
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_TYPE_COLUMN,
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_locked_label_policy_status",
            "smdg_source_analysis_scope",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_native_od_oc_masks=("smdg_has_native_od_oc_masks", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "smdg_exclusion_reason",
            "n_rows",
        ],
        ascending=[True, False],
    )
)

exclusion_audit.to_csv(SMDG_EXCLUSION_AUDIT_CSV, index=False)

fully_excluded_source_audit = (
    smdg_locked[
        smdg_locked["smdg_fully_excluded_source"].astype(bool)
    ]
    .groupby(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            "smdg_source_scope_reason",
            SMDG_TYPE_COLUMN,
            SMDG_TYPE_EXPANDED_COLUMN,
            "smdg_locked_label_policy_status",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("smdg_sample_id", "count"),
        n_fundus_existing=("smdg_fundus_exists", "sum"),
        n_native_od_oc_masks=("smdg_has_native_od_oc_masks", "sum"),
        n_patient_id_available=("smdg_patient_id_available", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            SMDG_DERIVED_SOURCE_COLUMN,
            SMDG_TYPE_COLUMN,
            "n_rows",
        ],
        ascending=[True, True, False],
    )
)

fully_excluded_source_audit.to_csv(
    SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV,
    index=False,
)


# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------

n_total = int(len(smdg_locked))
n_fundus_existing = int(smdg_locked["smdg_fundus_exists"].sum())
n_binary_locked = int(smdg_locked["smdg_binary_label_policy_locked"].sum())
n_ambiguous_excluded = int(
    smdg_locked["smdg_locked_label_policy_status"].eq("excluded_ambiguous_or_suspect").sum()
)

n_fully_excluded_papila = int(
    smdg_locked["smdg_fully_excluded_source"].sum()
)

n_reference_only = int(
    smdg_locked["smdg_training_overlap_reference_eligible"].sum()
)

n_primary = int(len(primary_manifest))
n_primary_positive = int((primary_manifest["external_label"] == 1).sum())
n_primary_negative = int((primary_manifest["external_label"] == 0).sum())

n_training_overlap_reference = int(len(training_overlap_reference_manifest))
n_training_overlap_reference_positive = int(
    (training_overlap_reference_manifest["external_label"] == 1).sum()
) if n_training_overlap_reference > 0 else 0
n_training_overlap_reference_negative = int(
    (training_overlap_reference_manifest["external_label"] == 0).sum()
) if n_training_overlap_reference > 0 else 0

n_native_anatomical = int(len(native_anatomical_manifest))
n_native_anatomical_positive = int(
    (native_anatomical_manifest["external_label"] == 1).sum()
) if n_native_anatomical > 0 else 0
n_native_anatomical_negative = int(
    (native_anatomical_manifest["external_label"] == 0).sum()
) if n_native_anatomical > 0 else 0

n_source_metric_eligible = int(
    source_level_eligibility["source_specific_binary_metric_eligible"].sum()
)

n_source_anatomical_eligible = int(
    source_level_eligibility["source_specific_anatomical_subset_eligible"].sum()
)

primary_sources = sorted(primary_manifest["external_source_dataset"].unique().tolist())
training_overlap_reference_sources = sorted(
    training_overlap_reference_manifest["external_source_dataset"].unique().tolist()
) if n_training_overlap_reference > 0 else []
native_anatomical_sources = sorted(
    native_anatomical_manifest["external_source_dataset"].unique().tolist()
) if n_native_anatomical > 0 else []

summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 2",
    "status": "PASS",
    "analysis": "SMDG locked label policy and AQPR external manifest construction",
    "primary_training_dataset": PRIMARY_MODEL_TRAINING_DOMAIN,
    "external_dataset": "SMDG",
    "smdg_total_rows": n_total,
    "smdg_fundus_existing_rows": n_fundus_existing,
    "smdg_binary_label_locked_rows": n_binary_locked,
    "smdg_ambiguous_or_suspect_excluded_rows": n_ambiguous_excluded,
    "fully_excluded_sources": FULLY_EXCLUDED_SOURCES,
    "reference_only_sources": REFERENCE_ONLY_SOURCES,
    "papila_policy": (
        "PAPILA rows inside SMDG are excluded from all analytical manifests in this branch. "
        "They are retained only for source/label/exclusion audit traceability."
    ),
    "primary_manifest_rows": n_primary,
    "primary_manifest_positive_rows": n_primary_positive,
    "primary_manifest_negative_rows": n_primary_negative,
    "primary_manifest_sources": primary_sources,
    "training_overlap_reference_manifest_rows": n_training_overlap_reference,
    "training_overlap_reference_positive_rows": n_training_overlap_reference_positive,
    "training_overlap_reference_negative_rows": n_training_overlap_reference_negative,
    "training_overlap_reference_sources": training_overlap_reference_sources,
    "native_anatomical_subset_rows": n_native_anatomical,
    "native_anatomical_subset_positive_rows": n_native_anatomical_positive,
    "native_anatomical_subset_negative_rows": n_native_anatomical_negative,
    "native_anatomical_subset_sources": native_anatomical_sources,
    "n_fully_excluded_papila_rows": n_fully_excluded_papila,
    "n_source_specific_binary_metric_eligible_sources": n_source_metric_eligible,
    "n_source_specific_anatomical_eligible_sources": n_source_anatomical_eligible,
    "global_smdg_patient_level_analysis_allowed": False,
    "patient_level_scope": (
        "Global SMDG patient-level analysis is not allowed because patient_id is sparse. "
        "HYGD remains the main patient-level external audit dataset."
    ),
    "model_tuning_on_smdg_allowed": False,
    "threshold_tuning_on_smdg_allowed": False,
    "temperature_fitting_on_smdg_allowed": False,
    "label_policy_locked": True,
    "types_mapping": {
        "1": "positive_glaucoma_or_referable_glaucoma",
        "0": "negative_non_glaucoma_control",
        "-1": "excluded_ambiguous_or_suspect",
    },
    "locked_label_policy_csv": str(SMDG_LOCKED_LABEL_POLICY_CSV),
    "locked_label_policy_by_source_csv": str(SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV),
    "source_scope_policy_csv": str(SMDG_SOURCE_SCOPE_POLICY_CSV),
    "source_level_eligibility_csv": str(SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV),
    "locked_metadata_csv": str(SMDG_LOCKED_METADATA_CSV),
    "primary_external_image_manifest_csv": str(SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV),
    "training_overlap_reference_manifest_csv": str(SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV),
    "native_anatomical_subset_manifest_csv": str(SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV),
    "exclusion_audit_csv": str(SMDG_EXCLUSION_AUDIT_CSV),
    "fully_excluded_source_audit_csv": str(SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV),
}

with open(SMDG_CELL2_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(SMDG_CELL2_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("SMDG AQPR label-policy and external manifest summary\n")
    f.write("====================================================\n")
    f.write(f"Status: {summary['status']}\n")
    f.write(f"Total SMDG rows: {n_total}\n")
    f.write(f"Binary label locked rows: {n_binary_locked}\n")
    f.write(f"Ambiguous/suspect excluded rows: {n_ambiguous_excluded}\n")
    f.write(f"Fully excluded PAPILA rows: {n_fully_excluded_papila}\n")
    f.write(f"Primary external manifest rows: {n_primary}\n")
    f.write(f"Primary positives: {n_primary_positive}\n")
    f.write(f"Primary negatives: {n_primary_negative}\n")
    f.write(f"Training-overlap reference rows: {n_training_overlap_reference}\n")
    f.write(f"Native anatomical subset rows: {n_native_anatomical}\n")
    f.write(f"Native anatomical positives: {n_native_anatomical_positive}\n")
    f.write(f"Native anatomical negatives: {n_native_anatomical_negative}\n")
    f.write(f"Source-specific metric-eligible sources: {n_source_metric_eligible}\n")
    f.write("\nPolicy:\n")
    f.write("types=1 -> positive; types=0 -> negative; types=-1 -> excluded.\n")
    f.write("PAPILA is excluded from all analytical manifests.\n")
    f.write("EyePACS-Glaucoma is retained only as a training-overlap reference artifact.\n")
    f.write("SMDG is external only: no training, threshold tuning, or calibration fitting.\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("Locked SMDG label policy:")
print(locked_label_policy.to_string(index=False))

print("\nSMDG source scope policy:")
print(source_scope_policy.to_string(index=False))

print("\nSMDG source-level metric eligibility:")
print(source_level_eligibility.to_string(index=False))

print("\nPrimary SMDG external manifest summary:")
print(f"Rows      : {n_primary}")
print(f"Positive  : {n_primary_positive}")
print(f"Negative  : {n_primary_negative}")
print(f"Sources   : {primary_sources}")

print("\nTraining-overlap reference manifest summary:")
print(f"Rows      : {n_training_overlap_reference}")
print(f"Positive  : {n_training_overlap_reference_positive}")
print(f"Negative  : {n_training_overlap_reference_negative}")
print(f"Sources   : {training_overlap_reference_sources}")

print("\nNative anatomical subset summary:")
print(f"Rows      : {n_native_anatomical}")
print(f"Positive  : {n_native_anatomical_positive}")
print(f"Negative  : {n_native_anatomical_negative}")
print(f"Sources   : {native_anatomical_sources}")

print("\nFully excluded source audit:")
print(fully_excluded_source_audit.to_string(index=False))

print("\nExclusion audit preview:")
print(exclusion_audit.head(100).to_string(index=False))

print("\nCell 2 summary:")
print(json.dumps(summary, indent=2))

print(f"\nLocked label policy CSV: {SMDG_LOCKED_LABEL_POLICY_CSV}")
print(f"Locked label policy by source CSV: {SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV}")
print(f"Source scope policy CSV: {SMDG_SOURCE_SCOPE_POLICY_CSV}")
print(f"Source-level eligibility CSV: {SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV}")
print(f"Locked metadata CSV: {SMDG_LOCKED_METADATA_CSV}")
print(f"Primary external image manifest CSV: {SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV}")
print(f"Training-overlap reference manifest CSV: {SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV}")
print(f"Native anatomical subset manifest CSV: {SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV}")
print(f"Exclusion audit CSV: {SMDG_EXCLUSION_AUDIT_CSV}")
print(f"Fully excluded source audit CSV: {SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV}")
print(f"Cell 2 summary JSON: {SMDG_CELL2_SUMMARY_JSON}")
print(f"Cell 2 summary TXT: {SMDG_CELL2_SUMMARY_TXT}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if n_total <= 0:
    raise RuntimeError("SMDG locked metadata is empty.")

if n_fundus_existing != n_total:
    raise RuntimeError(
        "Not all SMDG rows have existing fundus images. "
        "This should have been resolved in Cell 1."
    )

if n_binary_locked <= 0:
    raise RuntimeError("No SMDG binary labels were locked.")

if n_ambiguous_excluded <= 0:
    raise RuntimeError(
        "No ambiguous/suspect labels were excluded. "
        "This is unexpected for SMDG and should be reviewed."
    )

if n_primary <= 0:
    raise RuntimeError("Primary SMDG external manifest is empty.")

if n_primary_positive <= 0 or n_primary_negative <= 0:
    raise RuntimeError(
        "Primary SMDG external manifest must contain both positive and negative cases."
    )

if n_source_metric_eligible <= 0:
    raise RuntimeError(
        "No SMDG source is eligible for source-specific binary metrics."
    )

if n_native_anatomical <= 0:
    raise RuntimeError(
        "Native anatomical SMDG subset is empty."
    )

if n_native_anatomical_positive <= 0 or n_native_anatomical_negative <= 0:
    raise RuntimeError(
        "Native anatomical SMDG subset must contain both positive and negative cases."
    )

for fully_excluded_source in FULLY_EXCLUDED_SOURCES:
    if fully_excluded_source in set(primary_manifest["external_source_dataset"].unique()):
        raise RuntimeError(
            f"Fully excluded source leaked into primary manifest: {fully_excluded_source}"
        )

    if fully_excluded_source in set(training_overlap_reference_manifest["external_source_dataset"].unique()):
        raise RuntimeError(
            f"Fully excluded source leaked into training-overlap reference manifest: {fully_excluded_source}"
        )

    if fully_excluded_source in set(native_anatomical_manifest["external_source_dataset"].unique()):
        raise RuntimeError(
            f"Fully excluded source leaked into native anatomical manifest: {fully_excluded_source}"
        )

for reference_only_source in REFERENCE_ONLY_SOURCES:
    if reference_only_source in set(primary_manifest["external_source_dataset"].unique()):
        raise RuntimeError(
            f"Reference-only source leaked into primary external manifest: {reference_only_source}"
        )

    if reference_only_source in set(native_anatomical_manifest["external_source_dataset"].unique()):
        raise RuntimeError(
            f"Reference-only source leaked into native anatomical manifest: {reference_only_source}"
        )

for required_output in [
    SMDG_LOCKED_LABEL_POLICY_CSV,
    SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV,
    SMDG_SOURCE_SCOPE_POLICY_CSV,
    SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV,
    SMDG_LOCKED_METADATA_CSV,
    SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV,
    SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    SMDG_EXCLUSION_AUDIT_CSV,
    SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV,
    SMDG_CELL2_SUMMARY_JSON,
    SMDG_CELL2_SUMMARY_TXT,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 2 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 3: Finalize SMDG frozen external validation package and AQPR endpoint policy

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"
MODELS_ROOT = JOURNAL_ROOT / "models"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
RESULTS_13_DIR.mkdir(parents=True, exist_ok=True)

SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"
SMDG_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_PANEL_DIR = (
    MODELS_ROOT
    / "airogs_light_glaucoma_classifiers"
    / "frozen_aqpr_model_panel_notebook07"
)

NOTEBOOK07_AQPR_MODEL_PANEL_CSV = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.csv"
)


# ---------------------------------------------------------------------
# Inputs from Cells 1 and 2
# ---------------------------------------------------------------------

SMDG_CELL1_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_setup_summary.json"
)

SMDG_CELL2_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_label_policy_and_manifest_summary.json"
)

SMDG_LOCKED_LABEL_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_locked_label_policy.csv"
)

SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_locked_label_policy_by_source.csv"
)

SMDG_SOURCE_SCOPE_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_scope_policy.csv"
)

SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_source_level_metric_eligibility.csv"
)

SMDG_LOCKED_METADATA_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_locked_external_metadata.csv"
)

SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_primary_external_image_level_manifest.csv"
)

SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_training_overlap_reference_manifest.csv"
)

SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_native_anatomical_subset_manifest.csv"
)

SMDG_EXCLUSION_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_external_exclusion_audit.csv"
)

SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_fully_excluded_source_audit.csv"
)

required_inputs = [
    NOTEBOOK07_AQPR_MODEL_PANEL_CSV,
    SMDG_CELL1_SUMMARY_JSON,
    SMDG_CELL2_SUMMARY_JSON,
    SMDG_LOCKED_LABEL_POLICY_CSV,
    SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV,
    SMDG_SOURCE_SCOPE_POLICY_CSV,
    SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV,
    SMDG_LOCKED_METADATA_CSV,
    SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV,
    SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    SMDG_EXCLUSION_AUDIT_CSV,
    SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Missing Cell 3 inputs. Run Cells 1 and 2 first. Missing: {missing_inputs}"
    )


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_external_image_level_manifest.csv"
)

SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_training_overlap_reference_manifest.csv"
)

SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_native_anatomical_subset_manifest.csv"
)

SMDG_FINAL_LOCKED_METADATA_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_locked_external_metadata.csv"
)

SMDG_FINAL_MANIFEST_REGISTRY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_manifest_registry.csv"
)

SMDG_FINAL_ENDPOINT_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_aqpr_endpoint_policy.csv"
)

SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_source_analysis_plan.csv"
)

SMDG_FINAL_DATASET_CARD_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_dataset_card.csv"
)

SMDG_FINAL_CLAIM_BOUNDARIES_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_claim_boundaries.csv"
)

SMDG_FINAL_SELECTION_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_external_dataset_selection_summary.json"
)

SMDG_FINAL_SELECTION_TXT = (
    RESULTS_13_DIR
    / "notebook13_SMDG_FINAL_EXTERNAL_DATASET_SELECTION.txt"
)

SMDG_FINAL_STATUS_JSON = (
    RESULTS_13_DIR
    / "notebook13_cell03_final_smdg_selection_status.json"
)


# ---------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------

FULLY_EXCLUDED_SOURCES = {
    "PAPILA": (
        "excluded_from_all_analysis_manifests_due_to_user_scope_no_papila_in_current_aqpr_smdg_analysis"
    ),
}

REFERENCE_ONLY_SOURCES = {
    "EyePACS-Glaucoma": (
        "excluded_from_primary_external_manifest_due_to_training_domain_overlap_or_source_proximity"
    ),
}

PRIMARY_TRAINING_DATASET = "EyePACS-AIROGS-light-V2"
EXTERNAL_DATASET = "SMDG"

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_AQPR_MODEL_NAMES = [
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]
AQPR_PANEL_MODEL_NAMES = [
    PRIMARY_MODEL_NAME,
    *SECONDARY_AQPR_MODEL_NAMES,
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def require_columns(df, required_columns, table_name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def check_paths_exist(df, path_column, table_name):
    exists = df[path_column].fillna("").astype(str).apply(lambda x: Path(x).exists())

    if not bool(exists.all()):
        missing_preview = df.loc[
            ~exists,
            [col for col in ["external_sample_id", "external_source_dataset", path_column] if col in df.columns],
        ].head(20)

        print(f"\nMissing paths in {table_name}:")
        print(missing_preview.to_string(index=False))

        raise RuntimeError(
            f"{table_name} contains missing files in column {path_column}."
        )


def summarize_manifest(manifest, manifest_name):
    if len(manifest) == 0:
        return {
            "manifest_name": manifest_name,
            "n_rows": 0,
            "n_sources": 0,
            "n_positive": 0,
            "n_negative": 0,
            "n_native_od_oc_masks": 0,
            "n_patient_id_available": 0,
            "sources": [],
        }

    return {
        "manifest_name": manifest_name,
        "n_rows": int(len(manifest)),
        "n_sources": int(manifest["external_source_dataset"].nunique()),
        "n_positive": int((manifest["external_label"] == 1).sum())
        if "external_label" in manifest.columns
        else 0,
        "n_negative": int((manifest["external_label"] == 0).sum())
        if "external_label" in manifest.columns
        else 0,
        "n_native_od_oc_masks": int(manifest["has_native_od_oc_masks"].sum())
        if "has_native_od_oc_masks" in manifest.columns
        else 0,
        "n_patient_id_available": int(manifest["external_patient_id_available"].sum())
        if "external_patient_id_available" in manifest.columns
        else 0,
        "sources": sorted(manifest["external_source_dataset"].astype(str).unique().tolist()),
    }


def count_flagged_label_policy_rows(manifest):
    if len(manifest) == 0:
        return 0

    if "external_label_policy_sensitivity_flag" not in manifest.columns:
        return 0

    return int(manifest["external_label_policy_sensitivity_flag"].astype(bool).sum())


# ---------------------------------------------------------------------
# Load inputs
# ---------------------------------------------------------------------

cell1_summary = load_json(SMDG_CELL1_SUMMARY_JSON)
cell2_summary = load_json(SMDG_CELL2_SUMMARY_JSON)

if str(cell1_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 1 did not report PASS. Found: {cell1_summary.get('status')}"
    )

if str(cell2_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 2 did not report PASS. Found: {cell2_summary.get('status')}"
    )

frozen_panel = pd.read_csv(NOTEBOOK07_AQPR_MODEL_PANEL_CSV)
locked_metadata = pd.read_csv(SMDG_LOCKED_METADATA_CSV, low_memory=False)
primary_manifest = pd.read_csv(SMDG_PRIMARY_EXTERNAL_IMAGE_MANIFEST_CSV, low_memory=False)
training_overlap_reference_manifest = pd.read_csv(
    SMDG_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    low_memory=False,
)
native_anatomical_manifest = pd.read_csv(
    SMDG_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    low_memory=False,
)
locked_label_policy = pd.read_csv(SMDG_LOCKED_LABEL_POLICY_CSV)
locked_label_policy_by_source = pd.read_csv(SMDG_LOCKED_LABEL_POLICY_BY_SOURCE_CSV)
source_scope_policy = pd.read_csv(SMDG_SOURCE_SCOPE_POLICY_CSV)
source_level_eligibility = pd.read_csv(SMDG_SOURCE_LEVEL_ELIGIBILITY_CSV)
exclusion_audit = pd.read_csv(SMDG_EXCLUSION_AUDIT_CSV)
fully_excluded_source_audit = pd.read_csv(SMDG_FULLY_EXCLUDED_SOURCE_AUDIT_CSV)

require_columns(
    frozen_panel,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
    ],
    "Notebook 07 frozen AQPR model panel",
)

require_columns(
    primary_manifest,
    [
        "external_sample_id",
        "external_dataset",
        "external_source_dataset",
        "external_image_path",
        "external_label",
        "external_label_name",
        "external_label_policy_status",
        "external_label_confidence_tier",
        "external_label_policy_sensitivity_flag",
        "has_native_od_oc_masks",
        "external_patient_id_available",
    ],
    "SMDG primary external manifest",
)

require_columns(
    native_anatomical_manifest,
    [
        "external_sample_id",
        "external_source_dataset",
        "external_image_path",
        "external_label",
        "native_od_mask_path",
        "native_oc_mask_path",
        "has_native_od_oc_masks",
    ],
    "SMDG native anatomical subset manifest",
)

if sorted(frozen_panel["model_name"].astype(str).tolist()) != sorted(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError(
        "Frozen AQPR model panel does not match expected Notebook 07 AQPR panel."
    )


# ---------------------------------------------------------------------
# Persist final immutable manifests
# ---------------------------------------------------------------------

primary_manifest.to_csv(SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV, index=False)
training_overlap_reference_manifest.to_csv(
    SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    index=False,
)
native_anatomical_manifest.to_csv(
    SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    index=False,
)
locked_metadata.to_csv(SMDG_FINAL_LOCKED_METADATA_CSV, index=False)


# ---------------------------------------------------------------------
# Manifest summaries and registry
# ---------------------------------------------------------------------

primary_summary = summarize_manifest(
    primary_manifest,
    "primary_external_image_level_manifest",
)

training_overlap_summary = summarize_manifest(
    training_overlap_reference_manifest,
    "training_overlap_reference_manifest",
)

native_anatomical_summary = summarize_manifest(
    native_anatomical_manifest,
    "native_anatomical_subset_manifest",
)

manifest_registry = pd.DataFrame(
    [
        {
            "manifest_id": "smdg_primary_external_image_level",
            "manifest_role": "primary_external_validation",
            "path": str(SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV),
            "n_rows": primary_summary["n_rows"],
            "n_sources": primary_summary["n_sources"],
            "n_positive": primary_summary["n_positive"],
            "n_negative": primary_summary["n_negative"],
            "n_native_od_oc_masks": primary_summary["n_native_od_oc_masks"],
            "contains_papila": "PAPILA" in primary_summary["sources"],
            "contains_training_overlap_source": any(
                source in REFERENCE_ONLY_SOURCES
                for source in primary_summary["sources"]
            ),
            "used_for_primary_claims": True,
            "notes": (
                "Primary SMDG external image-level manifest. PAPILA and EyePACS-Glaucoma are excluded."
            ),
        },
        {
            "manifest_id": "smdg_training_overlap_reference",
            "manifest_role": "reference_only_training_domain_overlap",
            "path": str(SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV),
            "n_rows": training_overlap_summary["n_rows"],
            "n_sources": training_overlap_summary["n_sources"],
            "n_positive": training_overlap_summary["n_positive"],
            "n_negative": training_overlap_summary["n_negative"],
            "n_native_od_oc_masks": training_overlap_summary["n_native_od_oc_masks"],
            "contains_papila": "PAPILA" in training_overlap_summary["sources"],
            "contains_training_overlap_source": any(
                source in REFERENCE_ONLY_SOURCES
                for source in training_overlap_summary["sources"]
            ),
            "used_for_primary_claims": False,
            "notes": (
                "Reference artifact only. It must not be used as external validation evidence."
            ),
        },
        {
            "manifest_id": "smdg_native_anatomical_subset",
            "manifest_role": "secondary_native_anatomical_subset",
            "path": str(SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV),
            "n_rows": native_anatomical_summary["n_rows"],
            "n_sources": native_anatomical_summary["n_sources"],
            "n_positive": native_anatomical_summary["n_positive"],
            "n_negative": native_anatomical_summary["n_negative"],
            "n_native_od_oc_masks": native_anatomical_summary["n_native_od_oc_masks"],
            "contains_papila": "PAPILA" in native_anatomical_summary["sources"],
            "contains_training_overlap_source": any(
                source in REFERENCE_ONLY_SOURCES
                for source in native_anatomical_summary["sources"]
            ),
            "used_for_primary_claims": False,
            "notes": (
                "Native OD/OC-mask subset for anatomical reliability/XAI support. "
                "It is not the primary external performance endpoint."
            ),
        },
        {
            "manifest_id": "smdg_locked_full_metadata",
            "manifest_role": "traceability_only_full_locked_metadata",
            "path": str(SMDG_FINAL_LOCKED_METADATA_CSV),
            "n_rows": int(len(locked_metadata)),
            "n_sources": int(locked_metadata["smdg_source_dataset"].nunique())
            if "smdg_source_dataset" in locked_metadata.columns
            else np.nan,
            "n_positive": int((locked_metadata["smdg_locked_binary_label"] == 1).sum())
            if "smdg_locked_binary_label" in locked_metadata.columns
            else np.nan,
            "n_negative": int((locked_metadata["smdg_locked_binary_label"] == 0).sum())
            if "smdg_locked_binary_label" in locked_metadata.columns
            else np.nan,
            "n_native_od_oc_masks": int(locked_metadata["smdg_has_native_od_oc_masks"].sum())
            if "smdg_has_native_od_oc_masks" in locked_metadata.columns
            else np.nan,
            "contains_papila": True,
            "contains_training_overlap_source": True,
            "used_for_primary_claims": False,
            "notes": (
                "Full locked metadata retained only for reproducibility, exclusions, and audit traceability."
            ),
        },
    ]
)

manifest_registry.to_csv(SMDG_FINAL_MANIFEST_REGISTRY_CSV, index=False)


# ---------------------------------------------------------------------
# Source-level analysis plan
# ---------------------------------------------------------------------

source_analysis_plan = source_level_eligibility.copy()

source_analysis_plan["global_primary_external_analysis"] = True
source_analysis_plan["source_level_metric_analysis"] = source_analysis_plan[
    "source_specific_binary_metric_eligible"
].astype(bool)

source_analysis_plan["source_level_anatomical_subset_analysis"] = source_analysis_plan[
    "source_specific_anatomical_subset_eligible"
].astype(bool)

source_analysis_plan["patient_level_analysis_allowed"] = False

source_analysis_plan["patient_level_reason"] = (
    "SMDG patient_id is sparse and not globally reliable. "
    "Patient-level external claims remain restricted to HYGD."
)

source_analysis_plan["calibration_analysis_allowed"] = source_analysis_plan[
    "source_specific_binary_metric_eligible"
].astype(bool)

source_analysis_plan["xai_or_native_mask_analysis_allowed"] = source_analysis_plan[
    "source_specific_anatomical_subset_eligible"
].astype(bool)

source_analysis_plan["analysis_priority"] = np.where(
    source_analysis_plan["source_specific_binary_metric_eligible"].astype(bool),
    "primary_source_level_metric_candidate",
    "global_only_or_descriptive_source",
)

source_analysis_plan.to_csv(SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV, index=False)


# ---------------------------------------------------------------------
# AQPR endpoint policy
# ---------------------------------------------------------------------

endpoint_rows = []

for _, model_row in frozen_panel.iterrows():
    model_name = str(model_row["model_name"])
    model_role = str(model_row["aqpr_panel_role"])
    threshold = float(model_row["validation_selected_threshold"])
    temperature = float(model_row["temperature"])
    input_size = int(model_row["input_size"])

    endpoint_rows.append(
        {
            "endpoint_id": f"SMDG_PRIMARY_IMAGE_LEVEL_{model_name}",
            "endpoint_family": "external_generalization",
            "endpoint_name": "Primary SMDG image-level external validation",
            "model_name": model_name,
            "aqpr_panel_role": model_role,
            "manifest_id": "smdg_primary_external_image_level",
            "manifest_path": str(SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV),
            "evaluation_level": "image",
            "metric_scope": "global_primary_smdg",
            "primary_or_secondary": "primary_endpoint"
            if model_name == PRIMARY_MODEL_NAME
            else "secondary_architecture_panel_endpoint",
            "allowed_metrics": (
                "AUROC;AUPRC;balanced_accuracy;sensitivity;specificity;F1;Brier;ECE;thresholded_confusion_matrix"
            ),
            "frozen_threshold": threshold,
            "temperature": temperature,
            "input_size": input_size,
            "threshold_tuning_on_smdg_allowed": False,
            "temperature_fitting_on_smdg_allowed": False,
            "training_on_smdg_allowed": False,
            "patient_level_claim_allowed": False,
            "used_for_primary_claims": True if model_name == PRIMARY_MODEL_NAME else False,
            "claim_scope": (
                "External image-level generalization under locked SMDG label policy. "
                "No patient-level SMDG claim."
            ),
        }
    )

    endpoint_rows.append(
        {
            "endpoint_id": f"SMDG_SOURCE_LEVEL_{model_name}",
            "endpoint_family": "source_level_domain_shift",
            "endpoint_name": "SMDG source-level external performance heterogeneity",
            "model_name": model_name,
            "aqpr_panel_role": model_role,
            "manifest_id": "smdg_primary_external_image_level",
            "manifest_path": str(SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV),
            "evaluation_level": "source",
            "metric_scope": "eligible_smdg_sources_only",
            "primary_or_secondary": "secondary_endpoint",
            "allowed_metrics": (
                "AUROC;AUPRC;balanced_accuracy;sensitivity;specificity;F1;Brier;ECE;source_size;class_balance"
            ),
            "frozen_threshold": threshold,
            "temperature": temperature,
            "input_size": input_size,
            "threshold_tuning_on_smdg_allowed": False,
            "temperature_fitting_on_smdg_allowed": False,
            "training_on_smdg_allowed": False,
            "patient_level_claim_allowed": False,
            "used_for_primary_claims": False,
            "claim_scope": (
                "Dataset-source heterogeneity and domain-shift audit across eligible SMDG sources."
            ),
        }
    )

    endpoint_rows.append(
        {
            "endpoint_id": f"SMDG_NATIVE_ANATOMICAL_SUBSET_{model_name}",
            "endpoint_family": "anatomical_reliability_subset",
            "endpoint_name": "SMDG native OD/OC-mask anatomical subset",
            "model_name": model_name,
            "aqpr_panel_role": model_role,
            "manifest_id": "smdg_native_anatomical_subset",
            "manifest_path": str(SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV),
            "evaluation_level": "image",
            "metric_scope": "native_od_oc_mask_subset",
            "primary_or_secondary": "secondary_endpoint",
            "allowed_metrics": (
                "AUROC;AUPRC;balanced_accuracy;Brier;ECE;mask_availability;source_distribution"
            ),
            "frozen_threshold": threshold,
            "temperature": temperature,
            "input_size": input_size,
            "threshold_tuning_on_smdg_allowed": False,
            "temperature_fitting_on_smdg_allowed": False,
            "training_on_smdg_allowed": False,
            "patient_level_claim_allowed": False,
            "used_for_primary_claims": False,
            "claim_scope": (
                "Secondary native-mask subset for anatomical/XAI support. "
                "Do not treat as full SMDG performance."
            ),
        }
    )

    endpoint_rows.append(
        {
            "endpoint_id": f"SMDG_TRAINING_OVERLAP_REFERENCE_{model_name}",
            "endpoint_family": "reference_only",
            "endpoint_name": "Training-domain-overlap source reference",
            "model_name": model_name,
            "aqpr_panel_role": model_role,
            "manifest_id": "smdg_training_overlap_reference",
            "manifest_path": str(SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV),
            "evaluation_level": "image",
            "metric_scope": "reference_only_not_external_evidence",
            "primary_or_secondary": "not_primary_evidence",
            "allowed_metrics": (
                "descriptive_probability_distribution_only"
            ),
            "frozen_threshold": threshold,
            "temperature": temperature,
            "input_size": input_size,
            "threshold_tuning_on_smdg_allowed": False,
            "temperature_fitting_on_smdg_allowed": False,
            "training_on_smdg_allowed": False,
            "patient_level_claim_allowed": False,
            "used_for_primary_claims": False,
            "claim_scope": (
                "Reference-only artifact. Must not be presented as independent external validation."
            ),
        }
    )

endpoint_policy = pd.DataFrame(endpoint_rows)
endpoint_policy.to_csv(SMDG_FINAL_ENDPOINT_POLICY_CSV, index=False)


# ---------------------------------------------------------------------
# Dataset card and claim boundaries
# ---------------------------------------------------------------------

n_total_smdg_rows = int(cell2_summary["smdg_total_rows"])
n_primary_rows = int(len(primary_manifest))
n_primary_positive = int((primary_manifest["external_label"] == 1).sum())
n_primary_negative = int((primary_manifest["external_label"] == 0).sum())
n_primary_sources = int(primary_manifest["external_source_dataset"].nunique())
n_primary_flagged_label_policy = count_flagged_label_policy_rows(primary_manifest)

n_native_rows = int(len(native_anatomical_manifest))
n_native_positive = int((native_anatomical_manifest["external_label"] == 1).sum())
n_native_negative = int((native_anatomical_manifest["external_label"] == 0).sum())
n_native_sources = int(native_anatomical_manifest["external_source_dataset"].nunique())

n_training_overlap_rows = int(len(training_overlap_reference_manifest))
n_fully_excluded_papila_rows = int(cell2_summary.get("n_fully_excluded_papila_rows", 0))
n_ambiguous_or_suspect_excluded = int(cell2_summary["smdg_ambiguous_or_suspect_excluded_rows"])

dataset_card = pd.DataFrame(
    [
        {
            "field": "external_dataset",
            "value": EXTERNAL_DATASET,
            "notes": "SMDG branch; PAPILA is excluded from all analytical manifests.",
        },
        {
            "field": "primary_training_dataset",
            "value": PRIMARY_TRAINING_DATASET,
            "notes": "Frozen classifiers were trained before SMDG evaluation.",
        },
        {
            "field": "total_locked_smdg_rows",
            "value": n_total_smdg_rows,
            "notes": "Full locked metadata retained for audit traceability only.",
        },
        {
            "field": "primary_external_manifest_rows",
            "value": n_primary_rows,
            "notes": "Primary image-level SMDG external validation cohort.",
        },
        {
            "field": "primary_external_positive_rows",
            "value": n_primary_positive,
            "notes": "types=1 under locked label policy.",
        },
        {
            "field": "primary_external_negative_rows",
            "value": n_primary_negative,
            "notes": "types=0 under locked label policy.",
        },
        {
            "field": "primary_external_sources",
            "value": n_primary_sources,
            "notes": ";".join(sorted(primary_manifest["external_source_dataset"].unique().tolist())),
        },
        {
            "field": "primary_external_label_policy_flagged_rows",
            "value": n_primary_flagged_label_policy,
            "notes": "Rows accepted by numeric label but flagged due to missing/unknown expanded text.",
        },
        {
            "field": "native_anatomical_subset_rows",
            "value": n_native_rows,
            "notes": "Rows with native OD/OC masks after exclusions.",
        },
        {
            "field": "native_anatomical_positive_rows",
            "value": n_native_positive,
            "notes": "Positive rows in native anatomical subset.",
        },
        {
            "field": "native_anatomical_negative_rows",
            "value": n_native_negative,
            "notes": "Negative rows in native anatomical subset.",
        },
        {
            "field": "native_anatomical_sources",
            "value": n_native_sources,
            "notes": ";".join(sorted(native_anatomical_manifest["external_source_dataset"].unique().tolist())),
        },
        {
            "field": "training_overlap_reference_rows",
            "value": n_training_overlap_rows,
            "notes": "Reference-only rows, not external-validation evidence.",
        },
        {
            "field": "fully_excluded_papila_rows",
            "value": n_fully_excluded_papila_rows,
            "notes": "Excluded from all analytical manifests by user-defined project scope.",
        },
        {
            "field": "ambiguous_or_suspect_excluded_rows",
            "value": n_ambiguous_or_suspect_excluded,
            "notes": "types=-1 excluded from binary evaluation.",
        },
        {
            "field": "global_patient_level_smdg_allowed",
            "value": False,
            "notes": "SMDG patient_id is sparse; patient-level claims remain HYGD-only.",
        },
        {
            "field": "model_tuning_on_smdg_allowed",
            "value": False,
            "notes": "No training, threshold tuning, or calibration fitting on SMDG.",
        },
    ]
)

dataset_card.to_csv(SMDG_FINAL_DATASET_CARD_CSV, index=False)

claim_boundaries = pd.DataFrame(
    [
        {
            "boundary_id": "SMDG_CB01",
            "topic": "external_dataset_scope",
            "claim_boundary": (
                "SMDG is used as an external image-level and source-level reliability audit dataset. "
                "It is not used for model training, model selection, threshold tuning, or temperature fitting."
            ),
            "manuscript_implication": (
                "Report SMDG results as locked external validation under a predefined label policy."
            ),
        },
        {
            "boundary_id": "SMDG_CB02",
            "topic": "papila_exclusion",
            "claim_boundary": (
                "PAPILA rows contained inside SMDG are excluded from all analytical manifests in this branch."
            ),
            "manuscript_implication": (
                "Do not report PAPILA performance, calibration, XAI, or domain-shift metrics in this SMDG branch."
            ),
        },
        {
            "boundary_id": "SMDG_CB03",
            "topic": "training_overlap_reference",
            "claim_boundary": (
                "EyePACS-Glaucoma rows are retained only as a training-domain-overlap reference artifact, "
                "not as primary external evidence."
            ),
            "manuscript_implication": (
                "Do not mix EyePACS-Glaucoma with the primary SMDG external validation cohort."
            ),
        },
        {
            "boundary_id": "SMDG_CB04",
            "topic": "patient_level_scope",
            "claim_boundary": (
                "Global SMDG patient-level analysis is not allowed because patient identifiers are sparse."
            ),
            "manuscript_implication": (
                "Patient-level external reliability claims remain restricted to HYGD."
            ),
        },
        {
            "boundary_id": "SMDG_CB05",
            "topic": "label_policy",
            "claim_boundary": (
                "SMDG binary mapping is locked as types=1 positive, types=0 negative, and types=-1 excluded."
            ),
            "manuscript_implication": (
                "Ambiguous/suspect cases must be reported in the exclusion audit, not silently merged into binary evaluation."
            ),
        },
        {
            "boundary_id": "SMDG_CB06",
            "topic": "native_anatomical_subset",
            "claim_boundary": (
                "The native OD/OC-mask subset is secondary and source-limited; it is not equivalent to full SMDG performance."
            ),
            "manuscript_implication": (
                "Use native anatomical subset analyses as reliability support, not as the main performance claim."
            ),
        },
    ]
)

claim_boundaries.to_csv(SMDG_FINAL_CLAIM_BOUNDARIES_CSV, index=False)


# ---------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------

summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 3",
    "status": "PASS",
    "analysis": "Final SMDG frozen external validation package and AQPR endpoint policy",
    "primary_training_dataset": PRIMARY_TRAINING_DATASET,
    "external_dataset": EXTERNAL_DATASET,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_aqpr_panel_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "primary_manifest": primary_summary,
    "training_overlap_reference_manifest": training_overlap_summary,
    "native_anatomical_manifest": native_anatomical_summary,
    "n_total_locked_smdg_rows": n_total_smdg_rows,
    "n_primary_external_rows": n_primary_rows,
    "n_primary_external_positive_rows": n_primary_positive,
    "n_primary_external_negative_rows": n_primary_negative,
    "n_primary_external_sources": n_primary_sources,
    "n_primary_external_label_policy_flagged_rows": n_primary_flagged_label_policy,
    "n_native_anatomical_rows": n_native_rows,
    "n_native_anatomical_positive_rows": n_native_positive,
    "n_native_anatomical_negative_rows": n_native_negative,
    "n_native_anatomical_sources": n_native_sources,
    "n_training_overlap_reference_rows": n_training_overlap_rows,
    "n_fully_excluded_papila_rows": n_fully_excluded_papila_rows,
    "n_ambiguous_or_suspect_excluded_rows": n_ambiguous_or_suspect_excluded,
    "global_patient_level_smdg_allowed": False,
    "smdg_threshold_tuning_allowed": False,
    "smdg_temperature_fitting_allowed": False,
    "smdg_training_allowed": False,
    "papila_excluded_from_all_analytical_manifests": True,
    "manifest_registry_csv": str(SMDG_FINAL_MANIFEST_REGISTRY_CSV),
    "endpoint_policy_csv": str(SMDG_FINAL_ENDPOINT_POLICY_CSV),
    "source_analysis_plan_csv": str(SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV),
    "dataset_card_csv": str(SMDG_FINAL_DATASET_CARD_CSV),
    "claim_boundaries_csv": str(SMDG_FINAL_CLAIM_BOUNDARIES_CSV),
    "final_external_image_manifest_csv": str(SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV),
    "final_training_overlap_reference_manifest_csv": str(SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV),
    "final_native_anatomical_subset_manifest_csv": str(SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV),
    "final_locked_metadata_csv": str(SMDG_FINAL_LOCKED_METADATA_CSV),
}

with open(SMDG_FINAL_SELECTION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(SMDG_FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

selection_txt_lines = [
    "SMDG final external dataset selection",
    "=====================================",
    f"Status: {summary['status']}",
    f"Primary training dataset: {PRIMARY_TRAINING_DATASET}",
    f"External dataset: {EXTERNAL_DATASET}",
    "",
    "Primary external image-level manifest:",
    f"  Rows: {n_primary_rows}",
    f"  Positives: {n_primary_positive}",
    f"  Negatives: {n_primary_negative}",
    f"  Sources: {', '.join(primary_summary['sources'])}",
    "",
    "Native anatomical subset:",
    f"  Rows: {n_native_rows}",
    f"  Positives: {n_native_positive}",
    f"  Negatives: {n_native_negative}",
    f"  Sources: {', '.join(native_anatomical_summary['sources'])}",
    "",
    "Reference-only / excluded sources:",
    f"  Training-overlap reference rows: {n_training_overlap_rows}",
    f"  Fully excluded PAPILA rows: {n_fully_excluded_papila_rows}",
    f"  Ambiguous/suspect excluded rows: {n_ambiguous_or_suspect_excluded}",
    "",
    "Policy:",
    "  No training, threshold tuning, or temperature fitting on SMDG.",
    "  No global SMDG patient-level claims.",
    "  PAPILA is excluded from all analytical manifests in this branch.",
    "  EyePACS-Glaucoma is reference-only and not primary external evidence.",
]

with open(SMDG_FINAL_SELECTION_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(selection_txt_lines) + "\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("SMDG final manifest registry:")
print(manifest_registry.to_string(index=False))

print("\nSMDG final source analysis plan:")
print(source_analysis_plan.to_string(index=False))

print("\nSMDG final endpoint policy:")
print(
    endpoint_policy[
        [
            "endpoint_id",
            "endpoint_family",
            "model_name",
            "manifest_id",
            "evaluation_level",
            "metric_scope",
            "frozen_threshold",
            "temperature",
            "used_for_primary_claims",
        ]
    ].to_string(index=False)
)

print("\nSMDG final dataset card:")
print(dataset_card.to_string(index=False))

print("\nSMDG claim boundaries:")
print(
    claim_boundaries[
        [
            "boundary_id",
            "topic",
            "manuscript_implication",
        ]
    ].to_string(index=False)
)

print("\nCell 3 final summary:")
print(json.dumps(summary, indent=2))

print(f"\nFinal external image manifest CSV: {SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV}")
print(f"Final training-overlap reference manifest CSV: {SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV}")
print(f"Final native anatomical subset manifest CSV: {SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV}")
print(f"Final locked metadata CSV: {SMDG_FINAL_LOCKED_METADATA_CSV}")
print(f"Manifest registry CSV: {SMDG_FINAL_MANIFEST_REGISTRY_CSV}")
print(f"Endpoint policy CSV: {SMDG_FINAL_ENDPOINT_POLICY_CSV}")
print(f"Source analysis plan CSV: {SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV}")
print(f"Dataset card CSV: {SMDG_FINAL_DATASET_CARD_CSV}")
print(f"Claim boundaries CSV: {SMDG_FINAL_CLAIM_BOUNDARIES_CSV}")
print(f"Selection summary JSON: {SMDG_FINAL_SELECTION_SUMMARY_JSON}")
print(f"Selection TXT: {SMDG_FINAL_SELECTION_TXT}")
print(f"Final status JSON: {SMDG_FINAL_STATUS_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if n_primary_rows <= 0:
    raise RuntimeError("Final SMDG primary external manifest is empty.")

if n_primary_positive <= 0 or n_primary_negative <= 0:
    raise RuntimeError(
        "Final SMDG primary external manifest must contain both classes."
    )

if n_native_rows <= 0:
    raise RuntimeError("Final SMDG native anatomical subset is empty.")

if n_native_positive <= 0 or n_native_negative <= 0:
    raise RuntimeError(
        "Final SMDG native anatomical subset must contain both classes."
    )

if "PAPILA" in set(primary_manifest["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("PAPILA leaked into final primary external manifest.")

if "PAPILA" in set(training_overlap_reference_manifest["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("PAPILA leaked into final training-overlap reference manifest.")

if "PAPILA" in set(native_anatomical_manifest["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("PAPILA leaked into final native anatomical manifest.")

for reference_source in REFERENCE_ONLY_SOURCES:
    if reference_source in set(primary_manifest["external_source_dataset"].astype(str).unique()):
        raise RuntimeError(
            f"Reference-only source leaked into final primary manifest: {reference_source}"
        )

    if reference_source in set(native_anatomical_manifest["external_source_dataset"].astype(str).unique()):
        raise RuntimeError(
            f"Reference-only source leaked into final native anatomical manifest: {reference_source}"
        )

check_paths_exist(
    primary_manifest,
    "external_image_path",
    "final primary SMDG external manifest",
)

check_paths_exist(
    native_anatomical_manifest,
    "external_image_path",
    "final native anatomical SMDG manifest",
)

check_paths_exist(
    native_anatomical_manifest,
    "native_od_mask_path",
    "final native anatomical SMDG manifest",
)

check_paths_exist(
    native_anatomical_manifest,
    "native_oc_mask_path",
    "final native anatomical SMDG manifest",
)

if manifest_registry["used_for_primary_claims"].sum() != 1:
    raise RuntimeError(
        "Exactly one manifest should be marked as used_for_primary_claims."
    )

if not endpoint_policy["threshold_tuning_on_smdg_allowed"].eq(False).all():
    raise RuntimeError("Endpoint policy incorrectly allows SMDG threshold tuning.")

if not endpoint_policy["temperature_fitting_on_smdg_allowed"].eq(False).all():
    raise RuntimeError("Endpoint policy incorrectly allows SMDG temperature fitting.")

if not endpoint_policy["training_on_smdg_allowed"].eq(False).all():
    raise RuntimeError("Endpoint policy incorrectly allows SMDG training.")

for required_output in [
    SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV,
    SMDG_FINAL_TRAINING_OVERLAP_REFERENCE_MANIFEST_CSV,
    SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    SMDG_FINAL_LOCKED_METADATA_CSV,
    SMDG_FINAL_MANIFEST_REGISTRY_CSV,
    SMDG_FINAL_ENDPOINT_POLICY_CSV,
    SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV,
    SMDG_FINAL_DATASET_CARD_CSV,
    SMDG_FINAL_CLAIM_BOUNDARIES_CSV,
    SMDG_FINAL_SELECTION_SUMMARY_JSON,
    SMDG_FINAL_SELECTION_TXT,
    SMDG_FINAL_STATUS_JSON,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 3 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 4: Build frozen SMDG 384x384 model inputs and native OD/OC mask inputs

import json
import re
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
RESULTS_13_DIR.mkdir(parents=True, exist_ok=True)

SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"
SMDG_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

SMDG_MODEL_INPUT_ROOT = (
    SMDG_PROJECT_DATA_DIR
    / "model_inputs_384"
)

SMDG_PRIMARY_INPUT_ROOT = SMDG_MODEL_INPUT_ROOT / "primary_external"
SMDG_NATIVE_MASK_INPUT_ROOT = SMDG_MODEL_INPUT_ROOT / "native_od_oc_masks"

SMDG_PRIMARY_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
SMDG_NATIVE_MASK_INPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Inputs from Cell 3
# ---------------------------------------------------------------------

SMDG_CELL3_FINAL_STATUS_JSON = (
    RESULTS_13_DIR
    / "notebook13_cell03_final_smdg_selection_status.json"
)

SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_external_image_level_manifest.csv"
)

SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_final_native_anatomical_subset_manifest.csv"
)

SMDG_FINAL_ENDPOINT_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_aqpr_endpoint_policy.csv"
)

SMDG_FINAL_MANIFEST_REGISTRY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_manifest_registry.csv"
)

required_inputs = [
    SMDG_CELL3_FINAL_STATUS_JSON,
    SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV,
    SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    SMDG_FINAL_ENDPOINT_POLICY_CSV,
    SMDG_FINAL_MANIFEST_REGISTRY_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Missing Cell 4 inputs. Run Cells 1-3 first. Missing: {missing_inputs}"
    )


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_primary_external_model_input_manifest_384.csv"
)

SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_native_anatomical_model_input_manifest_384.csv"
)

SMDG_MODEL_INPUT_SOURCE_SUMMARY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_source_summary_384.csv"
)

SMDG_MODEL_INPUT_QUALITY_AUDIT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_quality_audit_384.csv"
)

SMDG_MODEL_INPUT_ERROR_LOG_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_error_log_384.csv"
)

SMDG_MODEL_INPUT_CONTRACT_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_contract_384.csv"
)

SMDG_MODEL_INPUT_BUILD_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_build_summary_384.json"
)

SMDG_MODEL_INPUT_BUILD_SUMMARY_TXT = (
    RESULTS_13_DIR
    / "notebook13_SMDG_MODEL_INPUT_BUILD_SUMMARY_384.txt"
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

INPUT_SIZE = 384
PAD_VALUE_RGB = 0
FORCE_REBUILD_INPUTS = False

FULLY_EXCLUDED_SOURCES = {"PAPILA"}
REFERENCE_ONLY_SOURCES = {"EyePACS-Glaucoma"}

IMAGE_INTERPOLATION = cv2.INTER_AREA
MASK_INTERPOLATION = cv2.INTER_NEAREST


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def require_columns(df, required_columns, table_name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def normalize_file_stem(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    value = value.strip("_")

    if value == "":
        value = "sample"

    return value


def path_exists(path_value):
    if pd.isna(path_value):
        return False

    path_str = str(path_value).strip()

    if path_str == "" or path_str.lower() in ["nan", "none", "null"]:
        return False

    return Path(path_str).exists()


def read_image_rgb(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing image: {path}")

    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise RuntimeError(f"OpenCV failed to read image: {path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    return image_rgb


def read_mask_binary(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing mask: {path}")

    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise RuntimeError(f"OpenCV failed to read mask: {path}")

    mask_binary = (mask > 0).astype(np.uint8)

    return mask_binary


def pad_to_square_image(image, pad_value=0):
    height, width = image.shape[:2]

    if height == width:
        return image.copy(), {
            "pad_top": 0,
            "pad_bottom": 0,
            "pad_left": 0,
            "pad_right": 0,
            "padded_side": int(height),
            "padding_fraction": 0.0,
        }

    side = max(height, width)

    if image.ndim == 3:
        padded = np.full(
            (side, side, image.shape[2]),
            fill_value=pad_value,
            dtype=image.dtype,
        )
    else:
        padded = np.full(
            (side, side),
            fill_value=pad_value,
            dtype=image.dtype,
        )

    pad_top = (side - height) // 2
    pad_bottom = side - height - pad_top
    pad_left = (side - width) // 2
    pad_right = side - width - pad_left

    padded[
        pad_top:pad_top + height,
        pad_left:pad_left + width,
        ...
    ] = image

    original_area = float(height * width)
    padded_area = float(side * side)
    padding_fraction = float(1.0 - original_area / padded_area)

    return padded, {
        "pad_top": int(pad_top),
        "pad_bottom": int(pad_bottom),
        "pad_left": int(pad_left),
        "pad_right": int(pad_right),
        "padded_side": int(side),
        "padding_fraction": padding_fraction,
    }


def resize_rgb_to_input(image_rgb):
    return cv2.resize(
        image_rgb,
        (INPUT_SIZE, INPUT_SIZE),
        interpolation=IMAGE_INTERPOLATION,
    )


def resize_mask_to_input(mask_binary):
    resized = cv2.resize(
        mask_binary.astype(np.uint8),
        (INPUT_SIZE, INPUT_SIZE),
        interpolation=MASK_INTERPOLATION,
    )

    return (resized > 0).astype(np.uint8)


def save_rgb_png(image_rgb, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(path), image_bgr)

    if not ok:
        raise RuntimeError(f"Failed to write RGB image: {path}")


def save_mask_png(mask_binary, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    out = (mask_binary.astype(np.uint8) * 255)
    ok = cv2.imwrite(str(path), out)

    if not ok:
        raise RuntimeError(f"Failed to write mask image: {path}")


def mask_area_fraction(mask_binary):
    if mask_binary is None:
        return np.nan

    return float(mask_binary.astype(bool).mean())


def build_output_paths(sample_id, source):
    safe_source = normalize_file_stem(source)
    safe_sample = normalize_file_stem(sample_id)

    image_path = (
        SMDG_PRIMARY_INPUT_ROOT
        / safe_source
        / f"{safe_sample}__input384.png"
    )

    od_mask_path = (
        SMDG_NATIVE_MASK_INPUT_ROOT
        / safe_source
        / f"{safe_sample}__od_mask384.png"
    )

    oc_mask_path = (
        SMDG_NATIVE_MASK_INPUT_ROOT
        / safe_source
        / f"{safe_sample}__oc_mask384.png"
    )

    od_oc_union_path = (
        SMDG_NATIVE_MASK_INPUT_ROOT
        / safe_source
        / f"{safe_sample}__od_oc_union_mask384.png"
    )

    return image_path, od_mask_path, oc_mask_path, od_oc_union_path


# ---------------------------------------------------------------------
# Load inputs
# ---------------------------------------------------------------------

cell3_status = load_json(SMDG_CELL3_FINAL_STATUS_JSON)

if str(cell3_status.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 3 final status does not report PASS. Found: {cell3_status.get('status')}"
    )

primary_manifest = pd.read_csv(
    SMDG_FINAL_EXTERNAL_IMAGE_MANIFEST_CSV,
    low_memory=False,
)

native_manifest = pd.read_csv(
    SMDG_FINAL_NATIVE_ANATOMICAL_SUBSET_MANIFEST_CSV,
    low_memory=False,
)

endpoint_policy = pd.read_csv(SMDG_FINAL_ENDPOINT_POLICY_CSV)
manifest_registry = pd.read_csv(SMDG_FINAL_MANIFEST_REGISTRY_CSV)

required_primary_cols = [
    "external_sample_id",
    "external_dataset",
    "external_source_dataset",
    "external_image_path",
    "external_label",
    "external_label_name",
    "external_label_policy_status",
    "external_label_confidence_tier",
    "external_label_policy_sensitivity_flag",
    "external_patient_id",
    "external_patient_id_available",
    "external_eye",
    "external_eye_available",
    "native_od_mask_path",
    "native_oc_mask_path",
    "has_native_od_mask",
    "has_native_oc_mask",
    "has_native_od_oc_masks",
    "smdg_row_index",
    "smdg_type",
    "smdg_type_expanded",
    "source_analysis_scope",
    "source_scope_reason",
]

required_native_cols = [
    "external_sample_id",
    "external_source_dataset",
    "external_image_path",
    "external_label",
    "native_od_mask_path",
    "native_oc_mask_path",
    "has_native_od_oc_masks",
]

require_columns(
    primary_manifest,
    required_primary_cols,
    "SMDG final primary external image manifest",
)

require_columns(
    native_manifest,
    required_native_cols,
    "SMDG final native anatomical subset manifest",
)

if len(primary_manifest) == 0:
    raise RuntimeError("Primary SMDG external manifest is empty.")

if len(native_manifest) == 0:
    raise RuntimeError("Native anatomical SMDG manifest is empty.")

primary_sources = set(primary_manifest["external_source_dataset"].astype(str).unique())
native_sources = set(native_manifest["external_source_dataset"].astype(str).unique())

if "PAPILA" in primary_sources or "PAPILA" in native_sources:
    raise RuntimeError("PAPILA leaked into SMDG analytical manifests.")

if "EyePACS-Glaucoma" in primary_sources or "EyePACS-Glaucoma" in native_sources:
    raise RuntimeError("EyePACS-Glaucoma leaked into primary/native analytical manifests.")


# ---------------------------------------------------------------------
# Build standardized model inputs
# ---------------------------------------------------------------------

native_sample_ids = set(native_manifest["external_sample_id"].astype(str).tolist())

native_lookup = (
    native_manifest
    .set_index("external_sample_id")
    [
        [
            "native_od_mask_path",
            "native_oc_mask_path",
            "has_native_od_mask",
            "has_native_oc_mask",
            "has_native_od_oc_masks",
        ]
    ]
    .to_dict(orient="index")
)

rows = []
errors = []

iterator = tqdm(
    primary_manifest.itertuples(index=False),
    total=len(primary_manifest),
    desc="Building SMDG 384x384 inputs",
)

for row in iterator:
    row_dict = row._asdict()

    sample_id = str(row_dict["external_sample_id"])
    source = str(row_dict["external_source_dataset"])
    image_path = Path(str(row_dict["external_image_path"]))

    output_image_path, output_od_mask_path, output_oc_mask_path, output_union_mask_path = build_output_paths(
        sample_id=sample_id,
        source=source,
    )

    is_native_subset = sample_id in native_sample_ids

    try:
        if not image_path.exists():
            raise FileNotFoundError(f"Missing fundus image: {image_path}")

        should_build_image = FORCE_REBUILD_INPUTS or not output_image_path.exists()

        image_rgb = read_image_rgb(image_path)
        original_height, original_width = image_rgb.shape[:2]

        image_padded, pad_info = pad_to_square_image(
            image_rgb,
            pad_value=PAD_VALUE_RGB,
        )

        image_resized = resize_rgb_to_input(image_padded)

        if should_build_image:
            save_rgb_png(image_resized, output_image_path)

        od_mask_output_exists = False
        oc_mask_output_exists = False
        union_mask_output_exists = False

        od_area_fraction_384 = np.nan
        oc_area_fraction_384 = np.nan
        od_oc_union_area_fraction_384 = np.nan
        oc_inside_od_fraction_384 = np.nan

        native_od_mask_path = str(row_dict.get("native_od_mask_path", ""))
        native_oc_mask_path = str(row_dict.get("native_oc_mask_path", ""))

        if is_native_subset:
            native_info = native_lookup.get(sample_id, {})

            native_od_mask_path = str(native_info.get("native_od_mask_path", native_od_mask_path))
            native_oc_mask_path = str(native_info.get("native_oc_mask_path", native_oc_mask_path))

            if not path_exists(native_od_mask_path):
                raise FileNotFoundError(f"Missing native OD mask for {sample_id}: {native_od_mask_path}")

            if not path_exists(native_oc_mask_path):
                raise FileNotFoundError(f"Missing native OC mask for {sample_id}: {native_oc_mask_path}")

            od_mask = read_mask_binary(native_od_mask_path)
            oc_mask = read_mask_binary(native_oc_mask_path)

            if od_mask.shape[:2] != image_rgb.shape[:2]:
                od_mask = cv2.resize(
                    od_mask.astype(np.uint8),
                    (original_width, original_height),
                    interpolation=MASK_INTERPOLATION,
                )
                od_mask = (od_mask > 0).astype(np.uint8)

            if oc_mask.shape[:2] != image_rgb.shape[:2]:
                oc_mask = cv2.resize(
                    oc_mask.astype(np.uint8),
                    (original_width, original_height),
                    interpolation=MASK_INTERPOLATION,
                )
                oc_mask = (oc_mask > 0).astype(np.uint8)

            od_mask_padded, _ = pad_to_square_image(
                od_mask,
                pad_value=0,
            )

            oc_mask_padded, _ = pad_to_square_image(
                oc_mask,
                pad_value=0,
            )

            od_mask_resized = resize_mask_to_input(od_mask_padded)
            oc_mask_resized = resize_mask_to_input(oc_mask_padded)

            union_mask_resized = (
                od_mask_resized.astype(bool)
                | oc_mask_resized.astype(bool)
            ).astype(np.uint8)

            od_area_fraction_384 = mask_area_fraction(od_mask_resized)
            oc_area_fraction_384 = mask_area_fraction(oc_mask_resized)
            od_oc_union_area_fraction_384 = mask_area_fraction(union_mask_resized)

            od_area = float(od_mask_resized.astype(bool).sum())
            oc_inside_od = float(
                (
                    oc_mask_resized.astype(bool)
                    & od_mask_resized.astype(bool)
                ).sum()
            )

            if od_area > 0:
                oc_inside_od_fraction_384 = float(oc_inside_od / max(1.0, float(oc_mask_resized.astype(bool).sum())))

            should_build_masks = (
                FORCE_REBUILD_INPUTS
                or not output_od_mask_path.exists()
                or not output_oc_mask_path.exists()
                or not output_union_mask_path.exists()
            )

            if should_build_masks:
                save_mask_png(od_mask_resized, output_od_mask_path)
                save_mask_png(oc_mask_resized, output_oc_mask_path)
                save_mask_png(union_mask_resized, output_union_mask_path)

            od_mask_output_exists = output_od_mask_path.exists()
            oc_mask_output_exists = output_oc_mask_path.exists()
            union_mask_output_exists = output_union_mask_path.exists()

        output_image_exists = output_image_path.exists()

        rows.append(
            {
                **row_dict,
                "model_input_path": str(output_image_path),
                "model_input_exists": bool(output_image_exists),
                "model_input_size": INPUT_SIZE,
                "model_input_preprocessing": "pad_to_square_then_resize_384_imagenet_normalization_at_inference",
                "original_image_width": int(original_width),
                "original_image_height": int(original_height),
                "padded_side": int(pad_info["padded_side"]),
                "pad_top": int(pad_info["pad_top"]),
                "pad_bottom": int(pad_info["pad_bottom"]),
                "pad_left": int(pad_info["pad_left"]),
                "pad_right": int(pad_info["pad_right"]),
                "padding_fraction": float(pad_info["padding_fraction"]),
                "is_native_anatomical_subset": bool(is_native_subset),
                "model_input_native_od_mask_path": str(output_od_mask_path) if is_native_subset else "",
                "model_input_native_oc_mask_path": str(output_oc_mask_path) if is_native_subset else "",
                "model_input_native_od_oc_union_mask_path": str(output_union_mask_path) if is_native_subset else "",
                "model_input_native_od_mask_exists": bool(od_mask_output_exists),
                "model_input_native_oc_mask_exists": bool(oc_mask_output_exists),
                "model_input_native_od_oc_union_mask_exists": bool(union_mask_output_exists),
                "native_od_area_fraction_384": float(od_area_fraction_384)
                if np.isfinite(od_area_fraction_384)
                else np.nan,
                "native_oc_area_fraction_384": float(oc_area_fraction_384)
                if np.isfinite(oc_area_fraction_384)
                else np.nan,
                "native_od_oc_union_area_fraction_384": float(od_oc_union_area_fraction_384)
                if np.isfinite(od_oc_union_area_fraction_384)
                else np.nan,
                "native_oc_inside_od_fraction_384": float(oc_inside_od_fraction_384)
                if np.isfinite(oc_inside_od_fraction_384)
                else np.nan,
                "build_status": "PASS",
                "build_error": "",
            }
        )

    except Exception as exc:
        errors.append(
            {
                "external_sample_id": sample_id,
                "external_source_dataset": source,
                "external_image_path": str(image_path),
                "is_native_anatomical_subset": bool(is_native_subset),
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        rows.append(
            {
                **row_dict,
                "model_input_path": str(output_image_path),
                "model_input_exists": bool(output_image_path.exists()),
                "model_input_size": INPUT_SIZE,
                "model_input_preprocessing": "pad_to_square_then_resize_384_imagenet_normalization_at_inference",
                "original_image_width": np.nan,
                "original_image_height": np.nan,
                "padded_side": np.nan,
                "pad_top": np.nan,
                "pad_bottom": np.nan,
                "pad_left": np.nan,
                "pad_right": np.nan,
                "padding_fraction": np.nan,
                "is_native_anatomical_subset": bool(is_native_subset),
                "model_input_native_od_mask_path": str(output_od_mask_path) if is_native_subset else "",
                "model_input_native_oc_mask_path": str(output_oc_mask_path) if is_native_subset else "",
                "model_input_native_od_oc_union_mask_path": str(output_union_mask_path) if is_native_subset else "",
                "model_input_native_od_mask_exists": bool(output_od_mask_path.exists()) if is_native_subset else False,
                "model_input_native_oc_mask_exists": bool(output_oc_mask_path.exists()) if is_native_subset else False,
                "model_input_native_od_oc_union_mask_exists": bool(output_union_mask_path.exists()) if is_native_subset else False,
                "native_od_area_fraction_384": np.nan,
                "native_oc_area_fraction_384": np.nan,
                "native_od_oc_union_area_fraction_384": np.nan,
                "native_oc_inside_od_fraction_384": np.nan,
                "build_status": "FAIL",
                "build_error": f"{type(exc).__name__}: {str(exc)}",
            }
        )

model_input_manifest = pd.DataFrame(rows)
error_log = pd.DataFrame(errors)

model_input_manifest.to_csv(SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV, index=False)
error_log.to_csv(SMDG_MODEL_INPUT_ERROR_LOG_CSV, index=False)


# ---------------------------------------------------------------------
# Native anatomical input manifest
# ---------------------------------------------------------------------

native_model_input_manifest = model_input_manifest[
    model_input_manifest["is_native_anatomical_subset"].astype(bool)
].copy()

native_model_input_manifest.to_csv(
    SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV,
    index=False,
)


# ---------------------------------------------------------------------
# Quality and source summaries
# ---------------------------------------------------------------------

quality_audit = pd.DataFrame(
    [
        {
            "audit_item": "primary_rows",
            "value": int(len(model_input_manifest)),
            "notes": "Rows from final SMDG primary external manifest.",
        },
        {
            "audit_item": "primary_model_inputs_existing",
            "value": int(model_input_manifest["model_input_exists"].sum()),
            "notes": "Successfully written or reused 384x384 model inputs.",
        },
        {
            "audit_item": "primary_model_input_failures",
            "value": int((model_input_manifest["build_status"] != "PASS").sum()),
            "notes": "Rows with failed input construction.",
        },
        {
            "audit_item": "native_anatomical_rows",
            "value": int(len(native_model_input_manifest)),
            "notes": "Primary external rows with native OD/OC masks.",
        },
        {
            "audit_item": "native_od_masks_existing",
            "value": int(native_model_input_manifest["model_input_native_od_mask_exists"].sum())
            if len(native_model_input_manifest) > 0
            else 0,
            "notes": "Standardized 384x384 native OD masks.",
        },
        {
            "audit_item": "native_oc_masks_existing",
            "value": int(native_model_input_manifest["model_input_native_oc_mask_exists"].sum())
            if len(native_model_input_manifest) > 0
            else 0,
            "notes": "Standardized 384x384 native OC masks.",
        },
        {
            "audit_item": "native_od_oc_union_masks_existing",
            "value": int(native_model_input_manifest["model_input_native_od_oc_union_mask_exists"].sum())
            if len(native_model_input_manifest) > 0
            else 0,
            "notes": "Standardized 384x384 OD/OC union masks.",
        },
        {
            "audit_item": "mean_padding_fraction",
            "value": float(model_input_manifest["padding_fraction"].dropna().mean()),
            "notes": "Mean square-padding fraction before resizing.",
        },
        {
            "audit_item": "max_padding_fraction",
            "value": float(model_input_manifest["padding_fraction"].dropna().max()),
            "notes": "Maximum square-padding fraction before resizing.",
        },
        {
            "audit_item": "mean_native_od_area_fraction_384",
            "value": float(native_model_input_manifest["native_od_area_fraction_384"].dropna().mean())
            if len(native_model_input_manifest) > 0
            else np.nan,
            "notes": "Mean OD-mask area fraction after standardization.",
        },
        {
            "audit_item": "mean_native_oc_area_fraction_384",
            "value": float(native_model_input_manifest["native_oc_area_fraction_384"].dropna().mean())
            if len(native_model_input_manifest) > 0
            else np.nan,
            "notes": "Mean OC-mask area fraction after standardization.",
        },
    ]
)

quality_audit.to_csv(SMDG_MODEL_INPUT_QUALITY_AUDIT_CSV, index=False)

source_summary = (
    model_input_manifest
    .groupby("external_source_dataset", dropna=False)
    .agg(
        n_rows=("external_sample_id", "count"),
        n_positive=("external_label", lambda x: int((x == 1).sum())),
        n_negative=("external_label", lambda x: int((x == 0).sum())),
        n_model_inputs_existing=("model_input_exists", "sum"),
        n_native_anatomical_rows=("is_native_anatomical_subset", "sum"),
        n_native_od_masks_existing=("model_input_native_od_mask_exists", "sum"),
        n_native_oc_masks_existing=("model_input_native_oc_mask_exists", "sum"),
        mean_padding_fraction=("padding_fraction", "mean"),
        max_padding_fraction=("padding_fraction", "max"),
        mean_native_od_area_fraction_384=("native_od_area_fraction_384", "mean"),
        mean_native_oc_area_fraction_384=("native_oc_area_fraction_384", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "n_rows",
            "external_source_dataset",
        ],
        ascending=[False, True],
    )
)

source_summary.to_csv(SMDG_MODEL_INPUT_SOURCE_SUMMARY_CSV, index=False)


# ---------------------------------------------------------------------
# Input contract
# ---------------------------------------------------------------------

input_contract = pd.DataFrame(
    [
        {
            "contract_item": "image_input_size",
            "value": INPUT_SIZE,
            "status": "locked",
            "notes": "Matches frozen AQPR model-panel input size.",
        },
        {
            "contract_item": "image_preprocessing",
            "value": "pad_to_square_then_resize",
            "status": "locked",
            "notes": "Same spatial normalization family used for classifier inference.",
        },
        {
            "contract_item": "normalization",
            "value": "ImageNet mean/std at inference time",
            "status": "locked",
            "notes": "Images are saved as uint8 RGB PNG; normalization is applied by the inference dataloader.",
        },
        {
            "contract_item": "label_policy",
            "value": "locked_by_cell2",
            "status": "locked",
            "notes": "types=1 positive, types=0 negative, types=-1 excluded.",
        },
        {
            "contract_item": "papila_policy",
            "value": "fully_excluded_from_all_analytical_manifests",
            "status": "locked",
            "notes": "PAPILA rows are not present in this model-input manifest.",
        },
        {
            "contract_item": "training_overlap_policy",
            "value": "EyePACS-Glaucoma excluded from primary/native manifests",
            "status": "locked",
            "notes": "No training-domain-overlap rows are processed for primary SMDG inference.",
        },
        {
            "contract_item": "native_mask_preprocessing",
            "value": "same_pad_geometry_then_nearest_resize_384",
            "status": "locked",
            "notes": "Native OD/OC masks are standardized only for rows in the native anatomical subset.",
        },
        {
            "contract_item": "smdg_training_or_tuning",
            "value": "not_allowed",
            "status": "locked",
            "notes": "This cell only builds frozen external inputs. It does not train or tune models.",
        },
    ]
)

input_contract.to_csv(SMDG_MODEL_INPUT_CONTRACT_CSV, index=False)


# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------

n_primary_rows = int(len(model_input_manifest))
n_primary_pass = int((model_input_manifest["build_status"] == "PASS").sum())
n_primary_fail = int((model_input_manifest["build_status"] != "PASS").sum())

n_primary_positive = int((model_input_manifest["external_label"] == 1).sum())
n_primary_negative = int((model_input_manifest["external_label"] == 0).sum())
n_primary_sources = int(model_input_manifest["external_source_dataset"].nunique())

n_native_rows = int(len(native_model_input_manifest))
n_native_positive = int((native_model_input_manifest["external_label"] == 1).sum()) if n_native_rows > 0 else 0
n_native_negative = int((native_model_input_manifest["external_label"] == 0).sum()) if n_native_rows > 0 else 0
n_native_sources = int(native_model_input_manifest["external_source_dataset"].nunique()) if n_native_rows > 0 else 0

n_native_all_masks_existing = int(
    (
        native_model_input_manifest["model_input_native_od_mask_exists"].astype(bool)
        & native_model_input_manifest["model_input_native_oc_mask_exists"].astype(bool)
        & native_model_input_manifest["model_input_native_od_oc_union_mask_exists"].astype(bool)
    ).sum()
) if n_native_rows > 0 else 0

summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 4",
    "status": "PASS" if n_primary_fail == 0 else "FAIL",
    "analysis": "SMDG frozen 384x384 model-input construction",
    "input_size": INPUT_SIZE,
    "force_rebuild_inputs": FORCE_REBUILD_INPUTS,
    "primary_model_input_rows": n_primary_rows,
    "primary_model_input_pass_rows": n_primary_pass,
    "primary_model_input_fail_rows": n_primary_fail,
    "primary_positive_rows": n_primary_positive,
    "primary_negative_rows": n_primary_negative,
    "primary_sources": n_primary_sources,
    "native_anatomical_model_input_rows": n_native_rows,
    "native_anatomical_positive_rows": n_native_positive,
    "native_anatomical_negative_rows": n_native_negative,
    "native_anatomical_sources": n_native_sources,
    "native_anatomical_rows_with_all_standardized_masks": n_native_all_masks_existing,
    "papila_present_in_primary_inputs": bool(
        "PAPILA" in set(model_input_manifest["external_source_dataset"].astype(str).unique())
    ),
    "eyepacs_glaucoma_present_in_primary_inputs": bool(
        "EyePACS-Glaucoma" in set(model_input_manifest["external_source_dataset"].astype(str).unique())
    ),
    "primary_model_input_manifest_csv": str(SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV),
    "native_anatomical_model_input_manifest_csv": str(SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV),
    "source_summary_csv": str(SMDG_MODEL_INPUT_SOURCE_SUMMARY_CSV),
    "quality_audit_csv": str(SMDG_MODEL_INPUT_QUALITY_AUDIT_CSV),
    "error_log_csv": str(SMDG_MODEL_INPUT_ERROR_LOG_CSV),
    "input_contract_csv": str(SMDG_MODEL_INPUT_CONTRACT_CSV),
}

with open(SMDG_MODEL_INPUT_BUILD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(SMDG_MODEL_INPUT_BUILD_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("SMDG 384x384 model-input build summary\n")
    f.write("======================================\n")
    f.write(f"Status: {summary['status']}\n")
    f.write(f"Input size: {INPUT_SIZE}\n")
    f.write(f"Primary rows: {n_primary_rows}\n")
    f.write(f"Primary pass rows: {n_primary_pass}\n")
    f.write(f"Primary fail rows: {n_primary_fail}\n")
    f.write(f"Primary positives: {n_primary_positive}\n")
    f.write(f"Primary negatives: {n_primary_negative}\n")
    f.write(f"Primary sources: {n_primary_sources}\n")
    f.write(f"Native anatomical rows: {n_native_rows}\n")
    f.write(f"Native anatomical positives: {n_native_positive}\n")
    f.write(f"Native anatomical negatives: {n_native_negative}\n")
    f.write(f"Native rows with all standardized masks: {n_native_all_masks_existing}\n")
    f.write("\nPolicy:\n")
    f.write("PAPILA is excluded from all analytical manifests.\n")
    f.write("EyePACS-Glaucoma is excluded from primary/native analytical manifests.\n")
    f.write("No model training, threshold tuning, or calibration fitting is performed here.\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("SMDG 384x384 model-input source summary:")
print(source_summary.to_string(index=False))

print("\nSMDG 384x384 model-input quality audit:")
print(quality_audit.to_string(index=False))

print("\nSMDG 384x384 model-input contract:")
print(input_contract.to_string(index=False))

if len(error_log) > 0:
    print("\nSMDG model-input error log preview:")
    print(error_log.head(50).to_string(index=False))

print("\nCell 4 summary:")
print(json.dumps(summary, indent=2))

print(f"\nPrimary model-input manifest CSV: {SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV}")
print(f"Native anatomical model-input manifest CSV: {SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV}")
print(f"Source summary CSV: {SMDG_MODEL_INPUT_SOURCE_SUMMARY_CSV}")
print(f"Quality audit CSV: {SMDG_MODEL_INPUT_QUALITY_AUDIT_CSV}")
print(f"Error log CSV: {SMDG_MODEL_INPUT_ERROR_LOG_CSV}")
print(f"Input contract CSV: {SMDG_MODEL_INPUT_CONTRACT_CSV}")
print(f"Build summary JSON: {SMDG_MODEL_INPUT_BUILD_SUMMARY_JSON}")
print(f"Build summary TXT: {SMDG_MODEL_INPUT_BUILD_SUMMARY_TXT}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if n_primary_rows <= 0:
    raise RuntimeError("Primary SMDG model-input manifest is empty.")

if n_primary_fail > 0:
    raise RuntimeError(
        "One or more SMDG primary model inputs failed to build. "
        f"Review: {SMDG_MODEL_INPUT_ERROR_LOG_CSV}"
    )

if n_primary_positive <= 0 or n_primary_negative <= 0:
    raise RuntimeError(
        "Primary SMDG model-input manifest must contain both positive and negative cases."
    )

if n_native_rows <= 0:
    raise RuntimeError("Native anatomical model-input manifest is empty.")

if n_native_positive <= 0 or n_native_negative <= 0:
    raise RuntimeError(
        "Native anatomical model-input manifest must contain both positive and negative cases."
    )

if n_native_all_masks_existing != n_native_rows:
    raise RuntimeError(
        "Not all native anatomical rows have standardized OD, OC, and union masks."
    )

if summary["papila_present_in_primary_inputs"]:
    raise RuntimeError("PAPILA leaked into primary SMDG model-input manifest.")

if summary["eyepacs_glaucoma_present_in_primary_inputs"]:
    raise RuntimeError("EyePACS-Glaucoma leaked into primary SMDG model-input manifest.")

if not model_input_manifest["model_input_exists"].astype(bool).all():
    raise RuntimeError("Some primary model-input files do not exist.")

if not native_model_input_manifest["model_input_native_od_mask_exists"].astype(bool).all():
    raise RuntimeError("Some standardized native OD masks do not exist.")

if not native_model_input_manifest["model_input_native_oc_mask_exists"].astype(bool).all():
    raise RuntimeError("Some standardized native OC masks do not exist.")

if not native_model_input_manifest["model_input_native_od_oc_union_mask_exists"].astype(bool).all():
    raise RuntimeError("Some standardized native OD/OC union masks do not exist.")

for required_output in [
    SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV,
    SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV,
    SMDG_MODEL_INPUT_SOURCE_SUMMARY_CSV,
    SMDG_MODEL_INPUT_QUALITY_AUDIT_CSV,
    SMDG_MODEL_INPUT_ERROR_LOG_CSV,
    SMDG_MODEL_INPUT_CONTRACT_CSV,
    SMDG_MODEL_INPUT_BUILD_SUMMARY_JSON,
    SMDG_MODEL_INPUT_BUILD_SUMMARY_TXT,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 4 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 5: Frozen AQPR model-panel inference on SMDG external inputs

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"
MODELS_ROOT = JOURNAL_ROOT / "models"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"

SMDG_EXTERNAL_INFERENCE_DIR = RESULTS_13_DIR / "smdg_external_inference"
SMDG_EXTERNAL_INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

SMDG_PER_MODEL_PREDICTION_DIR = SMDG_EXTERNAL_INFERENCE_DIR / "per_model_predictions"
SMDG_PER_MODEL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------

FROZEN_PANEL_DIR = (
    MODELS_ROOT
    / "airogs_light_glaucoma_classifiers"
    / "frozen_aqpr_model_panel_notebook07"
)

NOTEBOOK07_AQPR_MODEL_PANEL_CSV = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.csv"
)

SMDG_CELL4_SUMMARY_JSON = (
    RESULTS_13_DIR
    / "notebook13_smdg_model_input_build_summary_384.json"
)

SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_primary_external_model_input_manifest_384.csv"
)

SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV = (
    SMDG_PROJECT_DATA_DIR
    / "notebook13_smdg_native_anatomical_model_input_manifest_384.csv"
)

SMDG_FINAL_ENDPOINT_POLICY_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_aqpr_endpoint_policy.csv"
)

SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_source_analysis_plan.csv"
)

required_inputs = [
    NOTEBOOK07_AQPR_MODEL_PANEL_CSV,
    SMDG_CELL4_SUMMARY_JSON,
    SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV,
    SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV,
    SMDG_FINAL_ENDPOINT_POLICY_CSV,
    SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Missing Cell 5 inputs. Run Cells 1-4 first. Missing: {missing_inputs}"
    )


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

SMDG_EXTERNAL_ALL_PREDICTIONS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_predictions_all_models.csv"
)

SMDG_EXTERNAL_GLOBAL_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_global_metrics.csv"
)

SMDG_EXTERNAL_SOURCE_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_source_level_metrics.csv"
)

SMDG_NATIVE_ANATOMICAL_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_native_anatomical_subset_metrics.csv"
)

SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_inference_progress.csv"
)

SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_inference_summary.json"
)

SMDG_EXTERNAL_INFERENCE_SUMMARY_TXT = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_SMDG_EXTERNAL_INFERENCE_SUMMARY.txt"
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"

AQPR_PANEL_MODEL_NAMES = [
    "EfficientNetB0",
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]

TIMM_MODEL_NAME_BY_AQPR_NAME = {
    "EfficientNetB0": "efficientnet_b0",
    "DenseNet121": "densenet121",
    "SwinTiny": "swin_tiny_patch4_window7_224",
    "ConvNeXtTiny": "convnext_tiny",
}

# Set to one model name for resumable per-model execution, or None to run all missing models.
MODEL_TO_RUN = None
FORCE_RECOMPUTE = False

NUM_WORKERS = 2
PIN_MEMORY = True
DEFAULT_BATCH_SIZE = 8

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

MIN_SOURCE_POSITIVES_FOR_METRICS = 10
MIN_SOURCE_NEGATIVES_FOR_METRICS = 10


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def require_columns(df, required_columns, table_name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in [
            "model_state_dict",
            "state_dict",
            "model",
            "net",
        ]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                state_dict = checkpoint[key]
                break
        else:
            state_dict = checkpoint
    else:
        raise RuntimeError("Unsupported checkpoint format.")

    cleaned = {}

    for key, value in state_dict.items():
        new_key = str(key)

        for prefix in [
            "module.",
            "model.",
            "net.",
        ]:
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]

        cleaned[new_key] = value

    return cleaned


def infer_num_classes_from_state_dict(state_dict):
    candidate_keys = [
        key
        for key in state_dict.keys()
        if (
            key.endswith(".weight")
            and state_dict[key].ndim == 2
            and any(token in key.lower() for token in ["classifier", "head", "fc"])
        )
    ]

    candidate_keys = sorted(
        candidate_keys,
        key=lambda x: state_dict[x].shape[0],
    )

    for key in candidate_keys:
        out_features = int(state_dict[key].shape[0])

        if out_features in [1, 2]:
            return out_features

    return 1


def create_timm_model(model_name, timm_model_name, num_classes, input_size):
    if "swin" in timm_model_name:
        try:
            return timm.create_model(
                timm_model_name,
                pretrained=False,
                num_classes=num_classes,
                img_size=input_size,
            )
        except TypeError:
            return timm.create_model(
                timm_model_name,
                pretrained=False,
                num_classes=num_classes,
            )

    return timm.create_model(
        timm_model_name,
        pretrained=False,
        num_classes=num_classes,
    )


def load_frozen_model(model_row, device):
    model_name = str(model_row["model_name"])
    timm_model_name = TIMM_MODEL_NAME_BY_AQPR_NAME[model_name]
    checkpoint_path = Path(str(model_row["checkpoint_path"]))
    input_size = int(model_row["input_size"])

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {model_name}: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    state_dict = extract_state_dict(checkpoint)
    num_classes = infer_num_classes_from_state_dict(state_dict)

    model = create_timm_model(
        model_name=model_name,
        timm_model_name=timm_model_name,
        num_classes=num_classes,
        input_size=input_size,
    )

    load_result = model.load_state_dict(
        state_dict,
        strict=False,
    )

    missing_keys = list(load_result.missing_keys)
    unexpected_keys = list(load_result.unexpected_keys)

    suspicious_missing = [
        key
        for key in missing_keys
        if any(token in key.lower() for token in ["classifier", "head", "fc"])
    ]

    if len(suspicious_missing) > 0:
        raise RuntimeError(
            f"Suspicious missing classifier/head keys for {model_name}: {suspicious_missing}"
        )

    model.to(device)
    model.eval()

    return model, {
        "model_name": model_name,
        "resolved_timm_model_name": timm_model_name,
        "num_classes": int(num_classes),
        "checkpoint_path": str(checkpoint_path),
        "n_missing_keys": int(len(missing_keys)),
        "n_unexpected_keys": int(len(unexpected_keys)),
        "missing_keys_preview": missing_keys[:20],
        "unexpected_keys_preview": unexpected_keys[:20],
    }


def logits_to_probability(logits, temperature=1.0):
    temperature = float(temperature)

    if temperature <= 0:
        raise ValueError(f"Temperature must be positive. Found: {temperature}")

    logits = logits / temperature

    if logits.ndim == 1:
        return torch.sigmoid(logits)

    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])

    if logits.shape[1] == 2:
        return torch.softmax(logits, dim=1)[:, 1]

    raise RuntimeError(f"Unsupported logits shape: {tuple(logits.shape)}")


def compute_ece(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for start, end in zip(bins[:-1], bins[1:]):
        if end == 1.0:
            mask = (y_prob >= start) & (y_prob <= end)
        else:
            mask = (y_prob >= start) & (y_prob < end)

        if not np.any(mask):
            continue

        bin_confidence = float(np.mean(y_prob[mask]))
        bin_accuracy = float(np.mean(y_true[mask]))
        bin_weight = float(np.mean(mask))
        ece += bin_weight * abs(bin_accuracy - bin_confidence)

    return float(ece)


def safe_metric_auc(y_true, y_prob, metric_name):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    if metric_name == "auroc":
        return float(roc_auc_score(y_true, y_prob))

    if metric_name == "auprc":
        return float(average_precision_score(y_true, y_prob))

    raise ValueError(metric_name)


def compute_metrics(df, label_col, prob_col, pred_col, group_name, model_name):
    y_true = df[label_col].astype(int).to_numpy()
    y_prob = df[prob_col].astype(float).to_numpy()
    y_pred = df[pred_col].astype(int).to_numpy()

    n_samples = int(len(df))
    n_positive = int((y_true == 1).sum())
    n_negative = int((y_true == 0).sum())

    metrics = {
        "group_name": group_name,
        "model_name": model_name,
        "n_samples": n_samples,
        "n_positive": n_positive,
        "n_negative": n_negative,
        "metric_eligible": bool(n_positive > 0 and n_negative > 0),
        "auroc": np.nan,
        "auprc": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "sensitivity": np.nan,
        "specificity": np.nan,
        "precision": np.nan,
        "f1": np.nan,
        "brier_score": np.nan,
        "ece_15": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
    }

    if n_samples == 0:
        return metrics

    metrics["auroc"] = safe_metric_auc(y_true, y_prob, "auroc")
    metrics["auprc"] = safe_metric_auc(y_true, y_prob, "auprc")

    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))

    if n_positive > 0 and n_negative > 0:
        metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    else:
        metrics["balanced_accuracy"] = np.nan

    metrics["sensitivity"] = float(recall_score(y_true, y_pred, zero_division=0))
    metrics["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    metrics["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    metrics["brier_score"] = float(brier_score_loss(y_true, y_prob))
    metrics["ece_15"] = compute_ece(y_true, y_prob, n_bins=15)

    labels = [0, 1]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)

    specificity_denominator = tn + fp

    if specificity_denominator > 0:
        metrics["specificity"] = float(tn / specificity_denominator)
    else:
        metrics["specificity"] = np.nan

    return metrics


def prediction_csv_for_model(model_name):
    return (
        SMDG_PER_MODEL_PREDICTION_DIR
        / f"notebook13_smdg_external_predictions_{model_name}.csv"
    )


class SMDGInputDataset(Dataset):
    def __init__(self, manifest_df):
        self.df = manifest_df.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(str(row["model_input_path"]))

        if not image_path.exists():
            raise FileNotFoundError(f"Missing model input image: {image_path}")

        image = Image.open(image_path).convert("RGB")
        image_np = np.asarray(image).astype(np.float32) / 255.0

        tensor = torch.from_numpy(image_np).permute(2, 0, 1)
        tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD

        return {
            "image": tensor,
            "row_index": int(idx),
        }


def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch], dim=0)
    row_indices = torch.tensor([item["row_index"] for item in batch], dtype=torch.long)

    return {
        "image": images,
        "row_index": row_indices,
    }


# ---------------------------------------------------------------------
# Load manifests and model panel
# ---------------------------------------------------------------------

cell4_summary = load_json(SMDG_CELL4_SUMMARY_JSON)

if str(cell4_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 4 does not report PASS. Found: {cell4_summary.get('status')}"
    )

model_panel = pd.read_csv(NOTEBOOK07_AQPR_MODEL_PANEL_CSV)
primary_input_manifest = pd.read_csv(SMDG_PRIMARY_MODEL_INPUT_MANIFEST_CSV, low_memory=False)
native_input_manifest = pd.read_csv(SMDG_NATIVE_ANATOMICAL_MODEL_INPUT_MANIFEST_CSV, low_memory=False)
endpoint_policy = pd.read_csv(SMDG_FINAL_ENDPOINT_POLICY_CSV)
source_analysis_plan = pd.read_csv(SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV)

require_columns(
    model_panel,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
    ],
    "Notebook 07 frozen AQPR model panel",
)

require_columns(
    primary_input_manifest,
    [
        "external_sample_id",
        "external_source_dataset",
        "external_label",
        "model_input_path",
        "model_input_exists",
        "is_native_anatomical_subset",
    ],
    "SMDG primary model-input manifest",
)

if "PAPILA" in set(primary_input_manifest["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("PAPILA leaked into primary SMDG inference manifest.")

if "EyePACS-Glaucoma" in set(primary_input_manifest["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("EyePACS-Glaucoma leaked into primary SMDG inference manifest.")

if not primary_input_manifest["model_input_exists"].astype(bool).all():
    raise RuntimeError("Some SMDG model-input files are missing.")

model_panel = model_panel[
    model_panel["model_name"].astype(str).isin(AQPR_PANEL_MODEL_NAMES)
].copy()

model_panel["model_name"] = pd.Categorical(
    model_panel["model_name"].astype(str),
    categories=AQPR_PANEL_MODEL_NAMES,
    ordered=True,
)

model_panel = model_panel.sort_values("model_name").reset_index(drop=True)

if model_panel["model_name"].astype(str).tolist() != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR panel models: {model_panel['model_name'].astype(str).tolist()}"
    )

if MODEL_TO_RUN is not None:
    if MODEL_TO_RUN not in AQPR_PANEL_MODEL_NAMES:
        raise ValueError(f"MODEL_TO_RUN must be one of {AQPR_PANEL_MODEL_NAMES} or None.")

    model_panel_to_run = model_panel[
        model_panel["model_name"].astype(str).eq(MODEL_TO_RUN)
    ].copy()
else:
    model_panel_to_run = model_panel.copy()


# ---------------------------------------------------------------------
# Run or load per-model predictions
# ---------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"SMDG primary inference rows: {len(primary_input_manifest)}")
print(f"SMDG native anatomical rows: {len(native_input_manifest)}")

progress_rows = []

for _, model_row in model_panel_to_run.iterrows():
    model_name = str(model_row["model_name"])
    output_csv = prediction_csv_for_model(model_name)

    expected_rows = int(len(primary_input_manifest))

    if output_csv.exists() and not FORCE_RECOMPUTE:
        cached = pd.read_csv(output_csv, low_memory=False)

        if len(cached) == expected_rows:
            progress_rows.append(
                {
                    "model_name": model_name,
                    "status": "cached",
                    "completed": True,
                    "n_predictions": int(len(cached)),
                    "expected_predictions": expected_rows,
                    "prediction_csv": str(output_csv),
                    "elapsed_sec": 0.0,
                    "device": str(device),
                }
            )

            print(f"Using cached predictions for {model_name}: {output_csv}")
            continue

        print(
            f"Cached prediction file has unexpected row count for {model_name}; recomputing."
        )

    start_time = time.time()

    model, load_info = load_frozen_model(
        model_row=model_row,
        device=device,
    )

    batch_size = int(model_row.get("batch_size", DEFAULT_BATCH_SIZE))
    batch_size = max(1, batch_size)

    threshold = float(model_row["validation_selected_threshold"])
    temperature = float(model_row["temperature"])

    dataset = SMDGInputDataset(primary_input_manifest)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and device.type == "cuda",
        collate_fn=collate_fn,
    )

    raw_probabilities = np.zeros(len(primary_input_manifest), dtype=np.float32)
    temperature_scaled_probabilities = np.zeros(len(primary_input_manifest), dtype=np.float32)
    logits_store = np.zeros(len(primary_input_manifest), dtype=np.float32)

    model.eval()

    with torch.no_grad():
        for batch in tqdm(
            dataloader,
            desc=f"SMDG inference | {model_name}",
        ):
            images = batch["image"].to(device, non_blocking=True)
            row_indices = batch["row_index"].cpu().numpy()

            logits = model(images)

            if logits.ndim == 2 and logits.shape[1] == 1:
                logits_vector = logits[:, 0]
            elif logits.ndim == 1:
                logits_vector = logits
            elif logits.ndim == 2 and logits.shape[1] == 2:
                logits_vector = logits[:, 1] - logits[:, 0]
            else:
                raise RuntimeError(
                    f"Unsupported logits shape for {model_name}: {tuple(logits.shape)}"
                )

            raw_probs = logits_to_probability(logits, temperature=1.0)
            temp_probs = logits_to_probability(logits, temperature=temperature)

            raw_probabilities[row_indices] = raw_probs.detach().cpu().numpy().astype(np.float32)
            temperature_scaled_probabilities[row_indices] = (
                temp_probs.detach().cpu().numpy().astype(np.float32)
            )
            logits_store[row_indices] = logits_vector.detach().cpu().numpy().astype(np.float32)

    prediction_df = primary_input_manifest.copy()
    prediction_df["model_name"] = model_name
    prediction_df["aqpr_panel_role"] = str(model_row["aqpr_panel_role"])
    prediction_df["resolved_timm_model_name"] = load_info["resolved_timm_model_name"]
    prediction_df["checkpoint_path"] = str(model_row["checkpoint_path"])
    prediction_df["frozen_threshold"] = threshold
    prediction_df["temperature"] = temperature
    prediction_df["logit"] = logits_store
    prediction_df["probability"] = raw_probabilities
    prediction_df["temperature_scaled_probability"] = temperature_scaled_probabilities
    prediction_df["predicted_label"] = (prediction_df["probability"] >= threshold).astype(int)
    prediction_df["temperature_scaled_predicted_label"] = (
        prediction_df["temperature_scaled_probability"] >= threshold
    ).astype(int)

    prediction_df["external_label"] = prediction_df["external_label"].astype(int)
    prediction_df["correct"] = (
        prediction_df["predicted_label"].astype(int)
        == prediction_df["external_label"].astype(int)
    )
    prediction_df["temperature_scaled_correct"] = (
        prediction_df["temperature_scaled_predicted_label"].astype(int)
        == prediction_df["external_label"].astype(int)
    )

    prediction_df["model_num_classes"] = load_info["num_classes"]
    prediction_df["model_missing_keys_count"] = load_info["n_missing_keys"]
    prediction_df["model_unexpected_keys_count"] = load_info["n_unexpected_keys"]

    prediction_df.to_csv(output_csv, index=False)

    elapsed_sec = round(time.time() - start_time, 3)

    progress_rows.append(
        {
            "model_name": model_name,
            "status": "computed",
            "completed": True,
            "n_predictions": int(len(prediction_df)),
            "expected_predictions": expected_rows,
            "prediction_csv": str(output_csv),
            "elapsed_sec": elapsed_sec,
            "device": str(device),
            "resolved_timm_model_name": load_info["resolved_timm_model_name"],
            "model_num_classes": load_info["num_classes"],
            "model_missing_keys_count": load_info["n_missing_keys"],
            "model_unexpected_keys_count": load_info["n_unexpected_keys"],
        }
    )

    del model

    if device.type == "cuda":
        torch.cuda.empty_cache()

progress_df = pd.DataFrame(progress_rows)

if SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV.exists():
    previous_progress = pd.read_csv(SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV)
    progress_df = (
        pd.concat([previous_progress, progress_df], axis=0, ignore_index=True)
        .drop_duplicates(subset=["model_name"], keep="last")
    )

progress_df = progress_df.sort_values("model_name").reset_index(drop=True)
progress_df.to_csv(SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV, index=False)


# ---------------------------------------------------------------------
# Consolidate available predictions
# ---------------------------------------------------------------------

available_prediction_files = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    path = prediction_csv_for_model(model_name)

    if path.exists():
        df_tmp = pd.read_csv(path, nrows=5)

        if len(pd.read_csv(path, usecols=["external_sample_id"])) == len(primary_input_manifest):
            available_prediction_files.append(path)

all_predictions = []

for path in available_prediction_files:
    all_predictions.append(pd.read_csv(path, low_memory=False))

if len(all_predictions) == 0:
    raise RuntimeError("No per-model prediction files are available.")

all_predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)
all_predictions_df.to_csv(SMDG_EXTERNAL_ALL_PREDICTIONS_CSV, index=False)

available_models = sorted(all_predictions_df["model_name"].astype(str).unique().tolist())
all_models_available = sorted(available_models) == sorted(AQPR_PANEL_MODEL_NAMES)


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

global_metric_rows = []
native_metric_rows = []
source_metric_rows = []

for model_name in available_models:
    model_predictions = all_predictions_df[
        all_predictions_df["model_name"].astype(str).eq(model_name)
    ].copy()

    global_raw_metrics = compute_metrics(
        df=model_predictions,
        label_col="external_label",
        prob_col="probability",
        pred_col="predicted_label",
        group_name="SMDG_primary_external_global_raw_probability",
        model_name=model_name,
    )
    global_raw_metrics["probability_type"] = "raw_probability"

    global_temp_metrics = compute_metrics(
        df=model_predictions,
        label_col="external_label",
        prob_col="temperature_scaled_probability",
        pred_col="temperature_scaled_predicted_label",
        group_name="SMDG_primary_external_global_temperature_scaled_probability",
        model_name=model_name,
    )
    global_temp_metrics["probability_type"] = "temperature_scaled_probability"

    global_metric_rows.extend([global_raw_metrics, global_temp_metrics])

    native_predictions = model_predictions[
        model_predictions["is_native_anatomical_subset"].astype(bool)
    ].copy()

    if len(native_predictions) > 0:
        native_raw_metrics = compute_metrics(
            df=native_predictions,
            label_col="external_label",
            prob_col="probability",
            pred_col="predicted_label",
            group_name="SMDG_native_anatomical_subset_raw_probability",
            model_name=model_name,
        )
        native_raw_metrics["probability_type"] = "raw_probability"

        native_temp_metrics = compute_metrics(
            df=native_predictions,
            label_col="external_label",
            prob_col="temperature_scaled_probability",
            pred_col="temperature_scaled_predicted_label",
            group_name="SMDG_native_anatomical_subset_temperature_scaled_probability",
            model_name=model_name,
        )
        native_temp_metrics["probability_type"] = "temperature_scaled_probability"

        native_metric_rows.extend([native_raw_metrics, native_temp_metrics])

    for source_name, source_df in model_predictions.groupby("external_source_dataset"):
        n_pos = int((source_df["external_label"] == 1).sum())
        n_neg = int((source_df["external_label"] == 0).sum())
        eligible = (
            n_pos >= MIN_SOURCE_POSITIVES_FOR_METRICS
            and n_neg >= MIN_SOURCE_NEGATIVES_FOR_METRICS
        )

        source_raw_metrics = compute_metrics(
            df=source_df,
            label_col="external_label",
            prob_col="probability",
            pred_col="predicted_label",
            group_name=str(source_name),
            model_name=model_name,
        )
        source_raw_metrics["probability_type"] = "raw_probability"
        source_raw_metrics["source_specific_metric_eligible"] = bool(eligible)

        source_temp_metrics = compute_metrics(
            df=source_df,
            label_col="external_label",
            prob_col="temperature_scaled_probability",
            pred_col="temperature_scaled_predicted_label",
            group_name=str(source_name),
            model_name=model_name,
        )
        source_temp_metrics["probability_type"] = "temperature_scaled_probability"
        source_temp_metrics["source_specific_metric_eligible"] = bool(eligible)

        source_metric_rows.extend([source_raw_metrics, source_temp_metrics])

global_metrics = pd.DataFrame(global_metric_rows)
native_metrics = pd.DataFrame(native_metric_rows)
source_metrics = pd.DataFrame(source_metric_rows)

global_metrics.to_csv(SMDG_EXTERNAL_GLOBAL_METRICS_CSV, index=False)
native_metrics.to_csv(SMDG_NATIVE_ANATOMICAL_METRICS_CSV, index=False)
source_metrics.to_csv(SMDG_EXTERNAL_SOURCE_METRICS_CSV, index=False)


# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------

primary_available = PRIMARY_MODEL_NAME in available_models

if primary_available:
    primary_global = global_metrics[
        global_metrics["model_name"].astype(str).eq(PRIMARY_MODEL_NAME)
        & global_metrics["probability_type"].eq("raw_probability")
    ].iloc[0].to_dict()
else:
    primary_global = {}

summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 5",
    "status": "PASS" if all_models_available else "PARTIAL",
    "analysis": "Frozen AQPR panel inference on SMDG external model inputs",
    "device": str(device),
    "model_to_run": MODEL_TO_RUN,
    "force_recompute": FORCE_RECOMPUTE,
    "n_primary_inference_rows": int(len(primary_input_manifest)),
    "n_native_anatomical_rows": int(len(native_input_manifest)),
    "available_models": available_models,
    "expected_models": AQPR_PANEL_MODEL_NAMES,
    "all_models_available": bool(all_models_available),
    "n_all_predictions": int(len(all_predictions_df)),
    "expected_predictions_if_complete": int(len(primary_input_manifest) * len(AQPR_PANEL_MODEL_NAMES)),
    "primary_model_available": bool(primary_available),
    "primary_model_global_raw_metrics": primary_global,
    "prediction_csv": str(SMDG_EXTERNAL_ALL_PREDICTIONS_CSV),
    "global_metrics_csv": str(SMDG_EXTERNAL_GLOBAL_METRICS_CSV),
    "source_metrics_csv": str(SMDG_EXTERNAL_SOURCE_METRICS_CSV),
    "native_anatomical_metrics_csv": str(SMDG_NATIVE_ANATOMICAL_METRICS_CSV),
    "progress_csv": str(SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV),
    "per_model_prediction_dir": str(SMDG_PER_MODEL_PREDICTION_DIR),
    "papila_present_in_predictions": bool(
        "PAPILA" in set(all_predictions_df["external_source_dataset"].astype(str).unique())
    ),
    "eyepacs_glaucoma_present_in_predictions": bool(
        "EyePACS-Glaucoma" in set(all_predictions_df["external_source_dataset"].astype(str).unique())
    ),
    "smdg_training_or_tuning_performed": False,
    "threshold_tuning_on_smdg_performed": False,
    "temperature_fitting_on_smdg_performed": False,
}

with open(SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(SMDG_EXTERNAL_INFERENCE_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("SMDG external inference summary\n")
    f.write("===============================\n")
    f.write(f"Status: {summary['status']}\n")
    f.write(f"Device: {device}\n")
    f.write(f"Primary rows: {summary['n_primary_inference_rows']}\n")
    f.write(f"Native anatomical rows: {summary['n_native_anatomical_rows']}\n")
    f.write(f"Available models: {', '.join(available_models)}\n")
    f.write(f"Expected models: {', '.join(AQPR_PANEL_MODEL_NAMES)}\n")
    f.write(f"All models available: {all_models_available}\n")
    f.write(f"All predictions: {summary['n_all_predictions']}\n")
    f.write(f"Expected complete predictions: {summary['expected_predictions_if_complete']}\n")
    f.write("\nPolicy:\n")
    f.write("No training, threshold tuning, or temperature fitting was performed on SMDG.\n")
    f.write("PAPILA and EyePACS-Glaucoma are excluded from analytical predictions.\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("SMDG external inference progress:")
print(progress_df.to_string(index=False))

print("\nSMDG global metrics:")
print(
    global_metrics.sort_values(
        ["probability_type", "model_name"]
    ).to_string(index=False)
)

print("\nSMDG native anatomical subset metrics:")
if len(native_metrics) > 0:
    print(
        native_metrics.sort_values(
            ["probability_type", "model_name"]
        ).to_string(index=False)
    )
else:
    print("No native anatomical metrics available.")

print("\nSMDG source-level metrics preview:")
print(
    source_metrics.sort_values(
        ["model_name", "probability_type", "group_name"]
    ).head(80).to_string(index=False)
)

print("\nCell 5 summary:")
print(json.dumps(summary, indent=2))

print(f"\nAll predictions CSV: {SMDG_EXTERNAL_ALL_PREDICTIONS_CSV}")
print(f"Global metrics CSV: {SMDG_EXTERNAL_GLOBAL_METRICS_CSV}")
print(f"Source metrics CSV: {SMDG_EXTERNAL_SOURCE_METRICS_CSV}")
print(f"Native anatomical metrics CSV: {SMDG_NATIVE_ANATOMICAL_METRICS_CSV}")
print(f"Progress CSV: {SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV}")
print(f"Summary JSON: {SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON}")
print(f"Summary TXT: {SMDG_EXTERNAL_INFERENCE_SUMMARY_TXT}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if summary["papila_present_in_predictions"]:
    raise RuntimeError("PAPILA leaked into SMDG external predictions.")

if summary["eyepacs_glaucoma_present_in_predictions"]:
    raise RuntimeError("EyePACS-Glaucoma leaked into SMDG external predictions.")

if len(all_predictions_df) <= 0:
    raise RuntimeError("No SMDG predictions were generated or loaded.")

for model_name in available_models:
    n_model_rows = int(
        all_predictions_df["model_name"].astype(str).eq(model_name).sum()
    )

    if n_model_rows != len(primary_input_manifest):
        raise RuntimeError(
            f"Unexpected prediction count for {model_name}: "
            f"{n_model_rows} vs {len(primary_input_manifest)}"
        )

if not all_predictions_df["probability"].between(0.0, 1.0).all():
    raise RuntimeError("Raw probabilities outside [0, 1].")

if not all_predictions_df["temperature_scaled_probability"].between(0.0, 1.0).all():
    raise RuntimeError("Temperature-scaled probabilities outside [0, 1].")

for required_output in [
    SMDG_EXTERNAL_ALL_PREDICTIONS_CSV,
    SMDG_EXTERNAL_GLOBAL_METRICS_CSV,
    SMDG_EXTERNAL_SOURCE_METRICS_CSV,
    SMDG_NATIVE_ANATOMICAL_METRICS_CSV,
    SMDG_EXTERNAL_INFERENCE_PROGRESS_CSV,
    SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON,
    SMDG_EXTERNAL_INFERENCE_SUMMARY_TXT,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 5 output: {required_output}")

if not all_models_available:
    print(
        "Status: PARTIAL. Run Cell 5 again with MODEL_TO_RUN=None or with the missing model names "
        "until all AQPR panel models are available."
    )
else:
    print("Status: PASS")

In [ ]:
# Cell 6: Journal-grade SMDG statistical consolidation with 5000 bootstrap replicates
# Single-cell execution with live progress, ETA, and persistent runtime status.

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
SMDG_EXTERNAL_INFERENCE_DIR = RESULTS_13_DIR / "smdg_external_inference"
SMDG_STATISTICS_DIR = RESULTS_13_DIR / "smdg_external_statistics"
SMDG_STATISTICS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Inputs from Cell 5
# ---------------------------------------------------------------------

SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_inference_summary.json"
)

SMDG_EXTERNAL_ALL_PREDICTIONS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_predictions_all_models.csv"
)

SMDG_EXTERNAL_GLOBAL_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_global_metrics.csv"
)

SMDG_EXTERNAL_SOURCE_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_source_level_metrics.csv"
)

SMDG_NATIVE_ANATOMICAL_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_native_anatomical_subset_metrics.csv"
)

SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV = (
    RESULTS_13_DIR
    / "notebook13_smdg_final_source_analysis_plan.csv"
)

required_inputs = [
    SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON,
    SMDG_EXTERNAL_ALL_PREDICTIONS_CSV,
    SMDG_EXTERNAL_GLOBAL_METRICS_CSV,
    SMDG_EXTERNAL_SOURCE_METRICS_CSV,
    SMDG_NATIVE_ANATOMICAL_METRICS_CSV,
    SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV,
]

missing_inputs = [str(path) for path in required_inputs if not Path(path).exists()]

if missing_inputs:
    raise FileNotFoundError(
        f"Missing Cell 6 inputs. Run Cell 5 first. Missing: {missing_inputs}"
    )


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

SMDG_BOOTSTRAP_GLOBAL_CI_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_bootstrap_global_metrics_ci_5000.csv"
)

SMDG_BOOTSTRAP_NATIVE_CI_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_bootstrap_native_anatomical_metrics_ci_5000.csv"
)

SMDG_BOOTSTRAP_SOURCE_CI_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_bootstrap_source_metrics_ci_5000.csv"
)

SMDG_PAIRED_MODEL_COMPARISON_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_paired_model_comparisons_vs_primary_5000.csv"
)

SMDG_MODEL_RANKING_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_model_ranking_summary_5000.csv"
)

SMDG_DOMAIN_SHIFT_AUDIT_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_source_domain_shift_audit_5000.csv"
)

SMDG_SOURCE_SUPPORT_AUDIT_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_source_support_audit_5000.csv"
)

SMDG_STATISTICAL_CLAIM_BOUNDARIES_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_statistical_claim_boundaries_5000.csv"
)

SMDG_RUNTIME_PROGRESS_JSON = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_statistical_runtime_progress_5000.json"
)

SMDG_CELL6_SUMMARY_JSON = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_statistical_consolidation_summary_5000.json"
)

SMDG_CELL6_SUMMARY_TXT = (
    SMDG_STATISTICS_DIR
    / "notebook13_SMDG_STATISTICAL_CONSOLIDATION_SUMMARY_5000.txt"
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"

AQPR_PANEL_MODEL_NAMES = [
    "EfficientNetB0",
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]

PROBABILITY_TYPES = [
    "raw_probability",
    "temperature_scaled_probability",
]

PROBABILITY_COLUMN_BY_TYPE = {
    "raw_probability": "probability",
    "temperature_scaled_probability": "temperature_scaled_probability",
}

PREDICTION_COLUMN_BY_TYPE = {
    "raw_probability": "predicted_label",
    "temperature_scaled_probability": "temperature_scaled_predicted_label",
}

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 1306
CI_ALPHA = 0.05

FORCE_RECOMPUTE_STATISTICS = False

MIN_SOURCE_POSITIVES_FOR_METRICS = 10
MIN_SOURCE_NEGATIVES_FOR_METRICS = 10

SOURCE_ALERT_AUROC_DROP = 0.10
SOURCE_ALERT_BACC_DROP = 0.10
SOURCE_ALERT_ECE_INCREASE = 0.05
SOURCE_ALERT_SENSITIVITY_LOW = 0.50

METRICS_FOR_BOOTSTRAP = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1",
    "brier_score",
    "ece_15",
]

PAIRED_METRICS = [
    "auroc",
    "auprc",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
    "ece_15",
]

METRICS_HIGHER_IS_BETTER = {
    "auroc": True,
    "auprc": True,
    "accuracy": True,
    "balanced_accuracy": True,
    "sensitivity": True,
    "specificity": True,
    "precision": True,
    "f1": True,
    "brier_score": False,
    "ece_15": False,
}


# ---------------------------------------------------------------------
# Cache check
# ---------------------------------------------------------------------

expected_outputs = [
    SMDG_BOOTSTRAP_GLOBAL_CI_CSV,
    SMDG_BOOTSTRAP_NATIVE_CI_CSV,
    SMDG_BOOTSTRAP_SOURCE_CI_CSV,
    SMDG_PAIRED_MODEL_COMPARISON_CSV,
    SMDG_MODEL_RANKING_CSV,
    SMDG_DOMAIN_SHIFT_AUDIT_CSV,
    SMDG_SOURCE_SUPPORT_AUDIT_CSV,
    SMDG_STATISTICAL_CLAIM_BOUNDARIES_CSV,
    SMDG_CELL6_SUMMARY_JSON,
    SMDG_CELL6_SUMMARY_TXT,
]

if (
    not FORCE_RECOMPUTE_STATISTICS
    and all(path.exists() for path in expected_outputs)
):
    with open(SMDG_CELL6_SUMMARY_JSON, "r", encoding="utf-8") as f:
        cached_summary = json.load(f)

    if (
        cached_summary.get("status") == "PASS"
        and int(cached_summary.get("n_bootstrap", -1)) == N_BOOTSTRAP
    ):
        print("Cached Cell 6 outputs already exist with PASS status and 5000 bootstrap replicates.")
        print(json.dumps(cached_summary, indent=2))
        print("Status: PASS")
        raise SystemExit


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def require_columns(df, required_columns, table_name):
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise RuntimeError(
            f"{table_name} is missing required columns: {missing}"
        )


def write_runtime_progress(payload):
    payload = dict(payload)
    payload["updated_at_unix"] = time.time()

    with open(SMDG_RUNTIME_PROGRESS_JSON, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def compute_ece(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for start, end in zip(bins[:-1], bins[1:]):
        if end == 1.0:
            mask = (y_prob >= start) & (y_prob <= end)
        else:
            mask = (y_prob >= start) & (y_prob < end)

        if not np.any(mask):
            continue

        bin_confidence = float(np.mean(y_prob[mask]))
        bin_accuracy = float(np.mean(y_true[mask]))
        bin_weight = float(np.mean(mask))
        ece += bin_weight * abs(bin_accuracy - bin_confidence)

    return float(ece)


def safe_auc(y_true, y_prob, metric_name):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    if metric_name == "auroc":
        return float(roc_auc_score(y_true, y_prob))

    if metric_name == "auprc":
        return float(average_precision_score(y_true, y_prob))

    raise ValueError(metric_name)


def compute_metric_dict(y_true, y_prob, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = np.asarray(y_pred).astype(int)

    n_samples = int(len(y_true))
    n_positive = int((y_true == 1).sum())
    n_negative = int((y_true == 0).sum())

    out = {
        "n_samples": n_samples,
        "n_positive": n_positive,
        "n_negative": n_negative,
        "metric_eligible": bool(n_positive > 0 and n_negative > 0),
        "auroc": np.nan,
        "auprc": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "sensitivity": np.nan,
        "specificity": np.nan,
        "precision": np.nan,
        "f1": np.nan,
        "brier_score": np.nan,
        "ece_15": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
    }

    if n_samples == 0:
        return out

    out["auroc"] = safe_auc(y_true, y_prob, "auroc")
    out["auprc"] = safe_auc(y_true, y_prob, "auprc")
    out["accuracy"] = float(accuracy_score(y_true, y_pred))

    if n_positive > 0 and n_negative > 0:
        out["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    out["sensitivity"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    out["brier_score"] = float(brier_score_loss(y_true, y_prob))
    out["ece_15"] = compute_ece(y_true, y_prob, n_bins=15)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    out["tn"] = int(tn)
    out["fp"] = int(fp)
    out["fn"] = int(fn)
    out["tp"] = int(tp)

    if (tn + fp) > 0:
        out["specificity"] = float(tn / (tn + fp))

    return out


def bootstrap_metric_ci_with_progress(
    df,
    probability_type,
    n_bootstrap,
    seed,
    progress_bar,
    phase_label,
):
    probability_col = PROBABILITY_COLUMN_BY_TYPE[probability_type]
    prediction_col = PREDICTION_COLUMN_BY_TYPE[probability_type]

    y_true = df["external_label"].astype(int).to_numpy()
    y_prob = df[probability_col].astype(float).to_numpy()
    y_pred = df[prediction_col].astype(int).to_numpy()

    point = compute_metric_dict(y_true, y_prob, y_pred)

    positive_indices = np.where(y_true == 1)[0]
    negative_indices = np.where(y_true == 0)[0]

    metric_values = {metric: [] for metric in METRICS_FOR_BOOTSTRAP}

    if len(positive_indices) == 0 or len(negative_indices) == 0:
        for _ in range(n_bootstrap):
            progress_bar.update(1)

        rows = []

        for metric in METRICS_FOR_BOOTSTRAP:
            rows.append(
                {
                    "metric": metric,
                    "point_estimate": point[metric],
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "bootstrap_mean": np.nan,
                    "bootstrap_std": np.nan,
                    "n_bootstrap": int(n_bootstrap),
                    "n_bootstrap_valid": 0,
                    "n_samples": int(point["n_samples"]),
                    "n_positive": int(point["n_positive"]),
                    "n_negative": int(point["n_negative"]),
                    "metric_eligible": bool(point["metric_eligible"]),
                }
            )

        return pd.DataFrame(rows)

    rng = np.random.default_rng(seed)

    for bootstrap_idx in range(n_bootstrap):
        sampled_positive = rng.choice(
            positive_indices,
            size=len(positive_indices),
            replace=True,
        )

        sampled_negative = rng.choice(
            negative_indices,
            size=len(negative_indices),
            replace=True,
        )

        sampled = np.concatenate([sampled_positive, sampled_negative])
        rng.shuffle(sampled)

        sampled_metrics = compute_metric_dict(
            y_true=y_true[sampled],
            y_prob=y_prob[sampled],
            y_pred=y_pred[sampled],
        )

        for metric in METRICS_FOR_BOOTSTRAP:
            metric_values[metric].append(sampled_metrics[metric])

        progress_bar.update(1)

        if bootstrap_idx == 0 or (bootstrap_idx + 1) % 250 == 0:
            progress_bar.set_postfix_str(
                f"{phase_label} | {bootstrap_idx + 1}/{n_bootstrap}"
            )

    rows = []

    for metric in METRICS_FOR_BOOTSTRAP:
        values = np.asarray(metric_values[metric], dtype=float)
        values = values[np.isfinite(values)]

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            bootstrap_mean = np.nan
            bootstrap_std = np.nan
        else:
            ci_low = float(np.percentile(values, 100 * CI_ALPHA / 2))
            ci_high = float(np.percentile(values, 100 * (1 - CI_ALPHA / 2)))
            bootstrap_mean = float(np.mean(values))
            bootstrap_std = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan

        rows.append(
            {
                "metric": metric,
                "point_estimate": point[metric],
                "ci_low": ci_low,
                "ci_high": ci_high,
                "bootstrap_mean": bootstrap_mean,
                "bootstrap_std": bootstrap_std,
                "n_bootstrap": int(n_bootstrap),
                "n_bootstrap_valid": int(len(values)),
                "n_samples": int(point["n_samples"]),
                "n_positive": int(point["n_positive"]),
                "n_negative": int(point["n_negative"]),
                "metric_eligible": bool(point["metric_eligible"]),
            }
        )

    return pd.DataFrame(rows)


def paired_bootstrap_all_metrics_with_progress(
    comparison_df,
    primary_model_name,
    candidate_model_name,
    probability_type,
    n_bootstrap,
    seed,
    progress_bar,
    phase_label,
):
    probability_col = PROBABILITY_COLUMN_BY_TYPE[probability_type]
    prediction_col = PREDICTION_COLUMN_BY_TYPE[probability_type]

    primary_df = (
        comparison_df[comparison_df["model_name"].astype(str).eq(primary_model_name)]
        .sort_values("external_sample_id")
        .reset_index(drop=True)
    )

    candidate_df = (
        comparison_df[comparison_df["model_name"].astype(str).eq(candidate_model_name)]
        .sort_values("external_sample_id")
        .reset_index(drop=True)
    )

    if len(primary_df) != len(candidate_df):
        raise RuntimeError(
            f"Paired comparison row-count mismatch: {primary_model_name} vs {candidate_model_name}"
        )

    if not primary_df["external_sample_id"].astype(str).equals(
        candidate_df["external_sample_id"].astype(str)
    ):
        raise RuntimeError(
            f"Paired comparison sample mismatch: {primary_model_name} vs {candidate_model_name}"
        )

    y_true = primary_df["external_label"].astype(int).to_numpy()

    primary_prob = primary_df[probability_col].astype(float).to_numpy()
    primary_pred = primary_df[prediction_col].astype(int).to_numpy()

    candidate_prob = candidate_df[probability_col].astype(float).to_numpy()
    candidate_pred = candidate_df[prediction_col].astype(int).to_numpy()

    primary_point = compute_metric_dict(y_true, primary_prob, primary_pred)
    candidate_point = compute_metric_dict(y_true, candidate_prob, candidate_pred)

    positive_indices = np.where(y_true == 1)[0]
    negative_indices = np.where(y_true == 0)[0]

    diffs_by_metric = {metric: [] for metric in PAIRED_METRICS}

    rng = np.random.default_rng(seed)

    for bootstrap_idx in range(n_bootstrap):
        sampled_positive = rng.choice(
            positive_indices,
            size=len(positive_indices),
            replace=True,
        )

        sampled_negative = rng.choice(
            negative_indices,
            size=len(negative_indices),
            replace=True,
        )

        sampled = np.concatenate([sampled_positive, sampled_negative])
        rng.shuffle(sampled)

        primary_metrics = compute_metric_dict(
            y_true=y_true[sampled],
            y_prob=primary_prob[sampled],
            y_pred=primary_pred[sampled],
        )

        candidate_metrics = compute_metric_dict(
            y_true=y_true[sampled],
            y_prob=candidate_prob[sampled],
            y_pred=candidate_pred[sampled],
        )

        for metric in PAIRED_METRICS:
            diff = candidate_metrics[metric] - primary_metrics[metric]

            if np.isfinite(diff):
                diffs_by_metric[metric].append(diff)

        progress_bar.update(1)

        if bootstrap_idx == 0 or (bootstrap_idx + 1) % 250 == 0:
            progress_bar.set_postfix_str(
                f"{phase_label} | {bootstrap_idx + 1}/{n_bootstrap}"
            )

    rows = []

    for metric in PAIRED_METRICS:
        diffs = np.asarray(diffs_by_metric[metric], dtype=float)

        point_primary = primary_point[metric]
        point_candidate = candidate_point[metric]
        point_diff = point_candidate - point_primary

        if len(diffs) == 0:
            ci_low = np.nan
            ci_high = np.nan
            p_approx = np.nan
            comparison_result = "not_estimable"
        else:
            ci_low = float(np.percentile(diffs, 100 * CI_ALPHA / 2))
            ci_high = float(np.percentile(diffs, 100 * (1 - CI_ALPHA / 2)))

            p_left = float(np.mean(diffs <= 0.0))
            p_right = float(np.mean(diffs >= 0.0))
            p_approx = float(min(1.0, 2.0 * min(p_left, p_right)))

            higher_is_better = METRICS_HIGHER_IS_BETTER[metric]

            if ci_low > 0 and higher_is_better:
                comparison_result = "candidate_better"
            elif ci_high < 0 and higher_is_better:
                comparison_result = "primary_better"
            elif ci_high < 0 and not higher_is_better:
                comparison_result = "candidate_better"
            elif ci_low > 0 and not higher_is_better:
                comparison_result = "primary_better"
            else:
                comparison_result = "inconclusive"

        rows.append(
            {
                "primary_model_name": primary_model_name,
                "candidate_model_name": candidate_model_name,
                "probability_type": probability_type,
                "metric": metric,
                "primary_point": point_primary,
                "candidate_point": point_candidate,
                "candidate_minus_primary": point_diff,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "bootstrap_two_sided_p_approx": p_approx,
                "n_bootstrap": int(n_bootstrap),
                "n_bootstrap_valid": int(len(diffs)),
                "higher_is_better": bool(METRICS_HIGHER_IS_BETTER[metric]),
                "comparison_result": comparison_result,
            }
        )

    return pd.DataFrame(rows)


def metric_row_to_dict(ci_df, metric_name):
    row = ci_df[ci_df["metric"].eq(metric_name)]

    if len(row) != 1:
        return {}

    row = row.iloc[0]

    return {
        "point_estimate": float(row["point_estimate"]),
        "ci_low": float(row["ci_low"]),
        "ci_high": float(row["ci_high"]),
    }


# ---------------------------------------------------------------------
# Load inputs
# ---------------------------------------------------------------------

cell_start_time = time.time()

cell5_summary = load_json(SMDG_EXTERNAL_INFERENCE_SUMMARY_JSON)

if str(cell5_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        f"Cell 5 does not report PASS. Found: {cell5_summary.get('status')}"
    )

predictions = pd.read_csv(SMDG_EXTERNAL_ALL_PREDICTIONS_CSV, low_memory=False)
global_metrics = pd.read_csv(SMDG_EXTERNAL_GLOBAL_METRICS_CSV)
source_metrics = pd.read_csv(SMDG_EXTERNAL_SOURCE_METRICS_CSV)
native_metrics = pd.read_csv(SMDG_NATIVE_ANATOMICAL_METRICS_CSV)
source_analysis_plan = pd.read_csv(SMDG_FINAL_SOURCE_ANALYSIS_PLAN_CSV)

require_columns(
    predictions,
    [
        "external_sample_id",
        "external_source_dataset",
        "external_label",
        "model_name",
        "probability",
        "temperature_scaled_probability",
        "predicted_label",
        "temperature_scaled_predicted_label",
        "is_native_anatomical_subset",
    ],
    "SMDG all-model predictions",
)

if "PAPILA" in set(predictions["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("PAPILA leaked into Cell 6 predictions.")

if "EyePACS-Glaucoma" in set(predictions["external_source_dataset"].astype(str).unique()):
    raise RuntimeError("EyePACS-Glaucoma leaked into Cell 6 predictions.")

available_models = sorted(predictions["model_name"].astype(str).unique().tolist())

if sorted(available_models) != sorted(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError(
        f"Expected all AQPR models {AQPR_PANEL_MODEL_NAMES}, found {available_models}"
    )


# ---------------------------------------------------------------------
# Source support and total work estimation
# ---------------------------------------------------------------------

source_support = (
    predictions[predictions["model_name"].astype(str).eq(PRIMARY_MODEL_NAME)]
    .groupby("external_source_dataset", dropna=False)
    .agg(
        n_rows=("external_sample_id", "count"),
        n_positive=("external_label", lambda x: int((x == 1).sum())),
        n_negative=("external_label", lambda x: int((x == 0).sum())),
        n_native_anatomical=("is_native_anatomical_subset", "sum"),
    )
    .reset_index()
)

source_support["source_specific_metric_eligible"] = (
    (source_support["n_positive"] >= MIN_SOURCE_POSITIVES_FOR_METRICS)
    & (source_support["n_negative"] >= MIN_SOURCE_NEGATIVES_FOR_METRICS)
)

source_support["support_status"] = np.where(
    source_support["source_specific_metric_eligible"],
    "eligible_for_source_specific_metrics",
    "insufficient_positive_or_negative_support",
)

eligible_sources = (
    source_support[source_support["source_specific_metric_eligible"]]
    ["external_source_dataset"]
    .astype(str)
    .tolist()
)

source_support.to_csv(SMDG_SOURCE_SUPPORT_AUDIT_CSV, index=False)

n_global_configs = len(AQPR_PANEL_MODEL_NAMES) * len(PROBABILITY_TYPES)
n_native_configs = len(AQPR_PANEL_MODEL_NAMES) * len(PROBABILITY_TYPES)
n_source_configs = len(AQPR_PANEL_MODEL_NAMES) * len(PROBABILITY_TYPES) * len(eligible_sources)
n_paired_configs = (len(AQPR_PANEL_MODEL_NAMES) - 1) * len(PROBABILITY_TYPES)

total_bootstrap_iterations = (
    n_global_configs
    + n_native_configs
    + n_source_configs
    + n_paired_configs
) * N_BOOTSTRAP

print("Cell 6 planned workload")
print("=======================")
print(f"Bootstrap replicates per config : {N_BOOTSTRAP}")
print(f"Global CI configs              : {n_global_configs}")
print(f"Native CI configs              : {n_native_configs}")
print(f"Eligible source CI configs     : {n_source_configs}")
print(f"Paired comparison configs      : {n_paired_configs}")
print(f"Total bootstrap iterations     : {total_bootstrap_iterations}")
print(f"Eligible source datasets       : {eligible_sources}")

write_runtime_progress(
    {
        "status": "RUNNING",
        "phase": "initialization",
        "n_bootstrap": N_BOOTSTRAP,
        "total_bootstrap_iterations": int(total_bootstrap_iterations),
        "completed_bootstrap_iterations": 0,
        "percent_complete": 0.0,
        "current_model": None,
        "current_probability_type": None,
        "current_source": None,
        "message": "Cell 6 statistical consolidation initialized.",
    }
)


# ---------------------------------------------------------------------
# Main statistical execution with one global progress bar
# ---------------------------------------------------------------------

global_ci_rows = []
native_ci_rows = []
source_ci_rows = []
paired_rows = []

completed_configs = 0

with tqdm(
    total=total_bootstrap_iterations,
    desc="Cell 6 total bootstrap progress",
    unit="bootstrap",
    dynamic_ncols=True,
) as progress_bar:

    # Global CIs
    for model_idx, model_name in enumerate(AQPR_PANEL_MODEL_NAMES):
        model_df = predictions[predictions["model_name"].astype(str).eq(model_name)].copy()

        for prob_idx, probability_type in enumerate(PROBABILITY_TYPES):
            phase_label = f"global | {model_name} | {probability_type}"
            progress_bar.set_description(phase_label)

            write_runtime_progress(
                {
                    "status": "RUNNING",
                    "phase": "global_bootstrap_ci",
                    "n_bootstrap": N_BOOTSTRAP,
                    "total_bootstrap_iterations": int(total_bootstrap_iterations),
                    "completed_bootstrap_iterations": int(progress_bar.n),
                    "percent_complete": float(100.0 * progress_bar.n / max(1, total_bootstrap_iterations)),
                    "current_model": model_name,
                    "current_probability_type": probability_type,
                    "current_source": None,
                    "message": phase_label,
                }
            )

            seed = BOOTSTRAP_SEED + 1000 * model_idx + 100 * prob_idx

            ci_df = bootstrap_metric_ci_with_progress(
                df=model_df,
                probability_type=probability_type,
                n_bootstrap=N_BOOTSTRAP,
                seed=seed,
                progress_bar=progress_bar,
                phase_label=phase_label,
            )

            ci_df.insert(0, "analysis_scope", "primary_external_global")
            ci_df.insert(1, "model_name", model_name)
            ci_df.insert(2, "probability_type", probability_type)

            global_ci_rows.append(ci_df)
            completed_configs += 1

    # Native anatomical subset CIs
    for model_idx, model_name in enumerate(AQPR_PANEL_MODEL_NAMES):
        model_df = predictions[predictions["model_name"].astype(str).eq(model_name)].copy()
        native_df = model_df[model_df["is_native_anatomical_subset"].astype(bool)].copy()

        for prob_idx, probability_type in enumerate(PROBABILITY_TYPES):
            phase_label = f"native | {model_name} | {probability_type}"
            progress_bar.set_description(phase_label)

            write_runtime_progress(
                {
                    "status": "RUNNING",
                    "phase": "native_anatomical_bootstrap_ci",
                    "n_bootstrap": N_BOOTSTRAP,
                    "total_bootstrap_iterations": int(total_bootstrap_iterations),
                    "completed_bootstrap_iterations": int(progress_bar.n),
                    "percent_complete": float(100.0 * progress_bar.n / max(1, total_bootstrap_iterations)),
                    "current_model": model_name,
                    "current_probability_type": probability_type,
                    "current_source": None,
                    "message": phase_label,
                }
            )

            seed = BOOTSTRAP_SEED + 10000 + 1000 * model_idx + 100 * prob_idx

            ci_df = bootstrap_metric_ci_with_progress(
                df=native_df,
                probability_type=probability_type,
                n_bootstrap=N_BOOTSTRAP,
                seed=seed,
                progress_bar=progress_bar,
                phase_label=phase_label,
            )

            ci_df.insert(0, "analysis_scope", "native_anatomical_subset")
            ci_df.insert(1, "model_name", model_name)
            ci_df.insert(2, "probability_type", probability_type)

            native_ci_rows.append(ci_df)
            completed_configs += 1

    # Source-level CIs
    for model_idx, model_name in enumerate(AQPR_PANEL_MODEL_NAMES):
        model_df = predictions[predictions["model_name"].astype(str).eq(model_name)].copy()

        for source_idx, source_name in enumerate(eligible_sources):
            source_df = model_df[
                model_df["external_source_dataset"].astype(str).eq(str(source_name))
            ].copy()

            for prob_idx, probability_type in enumerate(PROBABILITY_TYPES):
                phase_label = f"source | {model_name} | {probability_type} | {source_name}"
                progress_bar.set_description(phase_label)

                write_runtime_progress(
                    {
                        "status": "RUNNING",
                        "phase": "source_level_bootstrap_ci",
                        "n_bootstrap": N_BOOTSTRAP,
                        "total_bootstrap_iterations": int(total_bootstrap_iterations),
                        "completed_bootstrap_iterations": int(progress_bar.n),
                        "percent_complete": float(100.0 * progress_bar.n / max(1, total_bootstrap_iterations)),
                        "current_model": model_name,
                        "current_probability_type": probability_type,
                        "current_source": source_name,
                        "message": phase_label,
                    }
                )

                seed = (
                    BOOTSTRAP_SEED
                    + 50000
                    + 1000 * model_idx
                    + 100 * prob_idx
                    + source_idx
                )

                ci_df = bootstrap_metric_ci_with_progress(
                    df=source_df,
                    probability_type=probability_type,
                    n_bootstrap=N_BOOTSTRAP,
                    seed=seed,
                    progress_bar=progress_bar,
                    phase_label=phase_label,
                )

                ci_df.insert(0, "analysis_scope", "source_level")
                ci_df.insert(1, "external_source_dataset", str(source_name))
                ci_df.insert(2, "model_name", model_name)
                ci_df.insert(3, "probability_type", probability_type)

                source_ci_rows.append(ci_df)
                completed_configs += 1

    # Paired model comparisons
    candidate_models = [
        model_name
        for model_name in AQPR_PANEL_MODEL_NAMES
        if model_name != PRIMARY_MODEL_NAME
    ]

    for candidate_idx, candidate_model_name in enumerate(candidate_models):
        for prob_idx, probability_type in enumerate(PROBABILITY_TYPES):
            phase_label = f"paired | {candidate_model_name} vs {PRIMARY_MODEL_NAME} | {probability_type}"
            progress_bar.set_description(phase_label)

            write_runtime_progress(
                {
                    "status": "RUNNING",
                    "phase": "paired_model_comparison",
                    "n_bootstrap": N_BOOTSTRAP,
                    "total_bootstrap_iterations": int(total_bootstrap_iterations),
                    "completed_bootstrap_iterations": int(progress_bar.n),
                    "percent_complete": float(100.0 * progress_bar.n / max(1, total_bootstrap_iterations)),
                    "current_model": candidate_model_name,
                    "current_probability_type": probability_type,
                    "current_source": None,
                    "message": phase_label,
                }
            )

            seed = BOOTSTRAP_SEED + 90000 + 1000 * candidate_idx + 100 * prob_idx

            paired_df = paired_bootstrap_all_metrics_with_progress(
                comparison_df=predictions,
                primary_model_name=PRIMARY_MODEL_NAME,
                candidate_model_name=candidate_model_name,
                probability_type=probability_type,
                n_bootstrap=N_BOOTSTRAP,
                seed=seed,
                progress_bar=progress_bar,
                phase_label=phase_label,
            )

            paired_rows.append(paired_df)
            completed_configs += 1


# ---------------------------------------------------------------------
# Save bootstrap outputs
# ---------------------------------------------------------------------

global_ci = pd.concat(global_ci_rows, axis=0, ignore_index=True)
native_ci = pd.concat(native_ci_rows, axis=0, ignore_index=True)
source_ci = pd.concat(source_ci_rows, axis=0, ignore_index=True)
paired_comparisons = pd.concat(paired_rows, axis=0, ignore_index=True)

global_ci.to_csv(SMDG_BOOTSTRAP_GLOBAL_CI_CSV, index=False)
native_ci.to_csv(SMDG_BOOTSTRAP_NATIVE_CI_CSV, index=False)
source_ci.to_csv(SMDG_BOOTSTRAP_SOURCE_CI_CSV, index=False)
paired_comparisons.to_csv(SMDG_PAIRED_MODEL_COMPARISON_CSV, index=False)


# ---------------------------------------------------------------------
# Model ranking
# ---------------------------------------------------------------------

ranking_base = global_ci[
    global_ci["metric"].isin(
        [
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
            "ece_15",
        ]
    )
].copy()

ranking_pivot = ranking_base.pivot_table(
    index=["model_name", "probability_type"],
    columns="metric",
    values="point_estimate",
    aggfunc="first",
).reset_index()

for metric, higher_is_better in METRICS_HIGHER_IS_BETTER.items():
    if metric not in ranking_pivot.columns:
        continue

    ranking_pivot[f"{metric}_rank"] = ranking_pivot[metric].rank(
        ascending=not higher_is_better,
        method="min",
    )

rank_columns = [col for col in ranking_pivot.columns if col.endswith("_rank")]
ranking_pivot["mean_metric_rank"] = ranking_pivot[rank_columns].mean(axis=1)

ranking_pivot["rank_family"] = np.where(
    ranking_pivot["probability_type"].eq("raw_probability"),
    "operational_thresholded_performance",
    "probability_reliability_after_temperature_scaling",
)

ranking_pivot = ranking_pivot.sort_values(
    [
        "probability_type",
        "mean_metric_rank",
        "auroc",
    ],
    ascending=[True, True, False],
).reset_index(drop=True)

ranking_pivot.to_csv(SMDG_MODEL_RANKING_CSV, index=False)


# ---------------------------------------------------------------------
# Source-domain shift audit
# ---------------------------------------------------------------------

source_audit_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    model_global_rows = global_metrics[
        global_metrics["model_name"].astype(str).eq(model_name)
    ].copy()

    model_source_rows = source_metrics[
        source_metrics["model_name"].astype(str).eq(model_name)
    ].copy()

    for probability_type in PROBABILITY_TYPES:
        global_row = model_global_rows[
            model_global_rows["probability_type"].eq(probability_type)
        ]

        if len(global_row) != 1:
            continue

        global_row = global_row.iloc[0].to_dict()

        source_subset = model_source_rows[
            model_source_rows["probability_type"].eq(probability_type)
        ].copy()

        for _, source_row in source_subset.iterrows():
            source_dict = source_row.to_dict()
            source_name = str(source_dict["group_name"])

            n_positive = int(source_dict["n_positive"])
            n_negative = int(source_dict["n_negative"])

            eligible = (
                n_positive >= MIN_SOURCE_POSITIVES_FOR_METRICS
                and n_negative >= MIN_SOURCE_NEGATIVES_FOR_METRICS
            )

            auroc_delta = (
                float(source_dict["auroc"]) - float(global_row["auroc"])
                if np.isfinite(source_dict["auroc"]) and np.isfinite(global_row["auroc"])
                else np.nan
            )

            bacc_delta = (
                float(source_dict["balanced_accuracy"]) - float(global_row["balanced_accuracy"])
                if np.isfinite(source_dict["balanced_accuracy"]) and np.isfinite(global_row["balanced_accuracy"])
                else np.nan
            )

            ece_delta = (
                float(source_dict["ece_15"]) - float(global_row["ece_15"])
                if np.isfinite(source_dict["ece_15"]) and np.isfinite(global_row["ece_15"])
                else np.nan
            )

            sensitivity_low = (
                bool(float(source_dict["sensitivity"]) < SOURCE_ALERT_SENSITIVITY_LOW)
                if np.isfinite(source_dict["sensitivity"])
                else False
            )

            auroc_drop_flag = (
                bool(auroc_delta <= -SOURCE_ALERT_AUROC_DROP)
                if np.isfinite(auroc_delta)
                else False
            )

            bacc_drop_flag = (
                bool(bacc_delta <= -SOURCE_ALERT_BACC_DROP)
                if np.isfinite(bacc_delta)
                else False
            )

            ece_increase_flag = (
                bool(ece_delta >= SOURCE_ALERT_ECE_INCREASE)
                if np.isfinite(ece_delta)
                else False
            )

            single_class_flag = bool(n_positive == 0 or n_negative == 0)

            alert_count = int(
                auroc_drop_flag
                + bacc_drop_flag
                + ece_increase_flag
                + sensitivity_low
                + single_class_flag
            )

            if single_class_flag:
                domain_shift_category = "single_class_source_descriptive_only"
            elif not eligible:
                domain_shift_category = "low_support_source_descriptive_only"
            elif alert_count >= 2:
                domain_shift_category = "high_domain_shift_or_reliability_risk"
            elif alert_count == 1:
                domain_shift_category = "moderate_domain_shift_or_reliability_risk"
            else:
                domain_shift_category = "no_major_source_alert"

            source_audit_rows.append(
                {
                    "model_name": model_name,
                    "probability_type": probability_type,
                    "external_source_dataset": source_name,
                    "n_samples": int(source_dict["n_samples"]),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "source_specific_metric_eligible": bool(eligible),
                    "source_auroc": source_dict["auroc"],
                    "global_auroc": global_row["auroc"],
                    "auroc_delta_vs_global": auroc_delta,
                    "source_auprc": source_dict["auprc"],
                    "global_auprc": global_row["auprc"],
                    "source_balanced_accuracy": source_dict["balanced_accuracy"],
                    "global_balanced_accuracy": global_row["balanced_accuracy"],
                    "balanced_accuracy_delta_vs_global": bacc_delta,
                    "source_sensitivity": source_dict["sensitivity"],
                    "global_sensitivity": global_row["sensitivity"],
                    "source_specificity": source_dict["specificity"],
                    "global_specificity": global_row["specificity"],
                    "source_f1": source_dict["f1"],
                    "global_f1": global_row["f1"],
                    "source_brier_score": source_dict["brier_score"],
                    "global_brier_score": global_row["brier_score"],
                    "source_ece_15": source_dict["ece_15"],
                    "global_ece_15": global_row["ece_15"],
                    "ece_delta_vs_global": ece_delta,
                    "single_class_flag": single_class_flag,
                    "auroc_drop_flag": auroc_drop_flag,
                    "balanced_accuracy_drop_flag": bacc_drop_flag,
                    "ece_increase_flag": ece_increase_flag,
                    "sensitivity_low_flag": sensitivity_low,
                    "alert_count": alert_count,
                    "domain_shift_category": domain_shift_category,
                }
            )

domain_shift_audit = pd.DataFrame(source_audit_rows)
domain_shift_audit.to_csv(SMDG_DOMAIN_SHIFT_AUDIT_CSV, index=False)


# ---------------------------------------------------------------------
# Statistical claim boundaries
# ---------------------------------------------------------------------

claim_boundaries = pd.DataFrame(
    [
        {
            "boundary_id": "SMDG_STAT_01",
            "topic": "confidence_intervals",
            "claim_boundary": (
                "Report SMDG external metrics with 5000-replicate stratified bootstrap confidence intervals, "
                "not only point estimates."
            ),
            "journal_relevance": (
                "Supports uncertainty-aware external validation rather than single-number reporting."
            ),
        },
        {
            "boundary_id": "SMDG_STAT_02",
            "topic": "paired_model_comparison",
            "claim_boundary": (
                "Architecture-panel differences are paired by image and interpreted through bootstrap CIs. "
                "They are not treated as independent-sample comparisons."
            ),
            "journal_relevance": (
                "Prevents overstatement of model differences and supports rigorous panel-level comparison."
            ),
        },
        {
            "boundary_id": "SMDG_STAT_03",
            "topic": "probability_type",
            "claim_boundary": (
                "Raw probabilities with frozen validation thresholds define operational external performance; "
                "temperature-scaled probabilities support probability reliability and calibration analysis."
            ),
            "journal_relevance": (
                "Separates discrimination/thresholded performance from calibration behavior."
            ),
        },
        {
            "boundary_id": "SMDG_STAT_04",
            "topic": "source_heterogeneity",
            "claim_boundary": (
                "Source-level results are interpreted as domain-shift and reliability heterogeneity, "
                "not as independent dataset rankings when support is small or class-incomplete."
            ),
            "journal_relevance": (
                "Avoids overclaiming on small or single-class SMDG sources."
            ),
        },
        {
            "boundary_id": "SMDG_STAT_05",
            "topic": "native_anatomical_subset",
            "claim_boundary": (
                "The native OD/OC-mask subset is a secondary anatomical-reliability subset and must not replace "
                "the full SMDG primary external endpoint."
            ),
            "journal_relevance": (
                "Maintains clear endpoint hierarchy for AQPR."
            ),
        },
        {
            "boundary_id": "SMDG_STAT_06",
            "topic": "patient_level_claims",
            "claim_boundary": (
                "No global patient-level claim is made on SMDG. Patient-level external reliability remains HYGD-only."
            ),
            "journal_relevance": (
                "Prevents invalid patient-level generalization claims."
            ),
        },
    ]
)

claim_boundaries.to_csv(SMDG_STATISTICAL_CLAIM_BOUNDARIES_CSV, index=False)


# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------

primary_raw_global = global_ci[
    global_ci["model_name"].eq(PRIMARY_MODEL_NAME)
    & global_ci["probability_type"].eq("raw_probability")
].copy()

primary_temp_global = global_ci[
    global_ci["model_name"].eq(PRIMARY_MODEL_NAME)
    & global_ci["probability_type"].eq("temperature_scaled_probability")
].copy()

primary_summary_metrics = {
    "raw_auroc": metric_row_to_dict(primary_raw_global, "auroc"),
    "raw_auprc": metric_row_to_dict(primary_raw_global, "auprc"),
    "raw_balanced_accuracy": metric_row_to_dict(primary_raw_global, "balanced_accuracy"),
    "raw_sensitivity": metric_row_to_dict(primary_raw_global, "sensitivity"),
    "raw_specificity": metric_row_to_dict(primary_raw_global, "specificity"),
    "raw_brier_score": metric_row_to_dict(primary_raw_global, "brier_score"),
    "raw_ece_15": metric_row_to_dict(primary_raw_global, "ece_15"),
    "temperature_scaled_brier_score": metric_row_to_dict(primary_temp_global, "brier_score"),
    "temperature_scaled_ece_15": metric_row_to_dict(primary_temp_global, "ece_15"),
}

high_risk_sources_primary_raw = domain_shift_audit[
    domain_shift_audit["model_name"].eq(PRIMARY_MODEL_NAME)
    & domain_shift_audit["probability_type"].eq("raw_probability")
    & domain_shift_audit["domain_shift_category"].isin(
        [
            "high_domain_shift_or_reliability_risk",
            "single_class_source_descriptive_only",
            "low_support_source_descriptive_only",
        ]
    )
].copy()

best_raw_rank = ranking_pivot[
    ranking_pivot["probability_type"].eq("raw_probability")
].sort_values("mean_metric_rank").head(1)

best_temp_rank = ranking_pivot[
    ranking_pivot["probability_type"].eq("temperature_scaled_probability")
].sort_values("mean_metric_rank").head(1)

elapsed_sec = round(time.time() - cell_start_time, 3)

summary = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 6",
    "status": "PASS",
    "analysis": "SMDG statistical consolidation for journal-grade external validation",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_bootstrap": int(N_BOOTSTRAP),
    "n_prediction_rows": int(len(predictions)),
    "n_unique_images": int(predictions["external_sample_id"].nunique()),
    "n_models": int(predictions["model_name"].nunique()),
    "n_sources": int(predictions["external_source_dataset"].nunique()),
    "n_eligible_sources_for_source_bootstrap": int(len(eligible_sources)),
    "eligible_sources_for_source_bootstrap": eligible_sources,
    "total_bootstrap_iterations": int(total_bootstrap_iterations),
    "primary_model_global_ci_summary": primary_summary_metrics,
    "best_raw_operational_rank_model": best_raw_rank["model_name"].iloc[0] if len(best_raw_rank) > 0 else None,
    "best_temperature_scaled_rank_model": best_temp_rank["model_name"].iloc[0] if len(best_temp_rank) > 0 else None,
    "n_high_or_descriptive_risk_sources_primary_raw": int(len(high_risk_sources_primary_raw)),
    "high_or_descriptive_risk_sources_primary_raw": sorted(
        high_risk_sources_primary_raw["external_source_dataset"].astype(str).unique().tolist()
    ),
    "papila_present": bool("PAPILA" in set(predictions["external_source_dataset"].astype(str).unique())),
    "eyepacs_glaucoma_present": bool("EyePACS-Glaucoma" in set(predictions["external_source_dataset"].astype(str).unique())),
    "elapsed_sec": elapsed_sec,
    "runtime_progress_json": str(SMDG_RUNTIME_PROGRESS_JSON),
    "bootstrap_global_ci_csv": str(SMDG_BOOTSTRAP_GLOBAL_CI_CSV),
    "bootstrap_native_ci_csv": str(SMDG_BOOTSTRAP_NATIVE_CI_CSV),
    "bootstrap_source_ci_csv": str(SMDG_BOOTSTRAP_SOURCE_CI_CSV),
    "paired_model_comparison_csv": str(SMDG_PAIRED_MODEL_COMPARISON_CSV),
    "model_ranking_csv": str(SMDG_MODEL_RANKING_CSV),
    "domain_shift_audit_csv": str(SMDG_DOMAIN_SHIFT_AUDIT_CSV),
    "source_support_audit_csv": str(SMDG_SOURCE_SUPPORT_AUDIT_CSV),
    "statistical_claim_boundaries_csv": str(SMDG_STATISTICAL_CLAIM_BOUNDARIES_CSV),
}

with open(SMDG_CELL6_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(SMDG_CELL6_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("SMDG statistical consolidation summary\n")
    f.write("======================================\n")
    f.write(f"Status: {summary['status']}\n")
    f.write(f"Primary model: {PRIMARY_MODEL_NAME}\n")
    f.write(f"Bootstrap replicates: {N_BOOTSTRAP}\n")
    f.write(f"Total bootstrap iterations: {total_bootstrap_iterations}\n")
    f.write(f"Prediction rows: {summary['n_prediction_rows']}\n")
    f.write(f"Unique images: {summary['n_unique_images']}\n")
    f.write(f"Models: {summary['n_models']}\n")
    f.write(f"Sources: {summary['n_sources']}\n")
    f.write(f"Eligible sources for source bootstrap: {len(eligible_sources)}\n")
    f.write(f"Elapsed seconds: {elapsed_sec}\n")
    f.write("\nPrimary EfficientNetB0 metrics with 95% CI:\n")

    for metric_name, metric_payload in primary_summary_metrics.items():
        if metric_payload:
            f.write(
                f"  {metric_name}: "
                f"{metric_payload['point_estimate']:.6f} "
                f"[{metric_payload['ci_low']:.6f}, {metric_payload['ci_high']:.6f}]\n"
            )

    f.write("\nHigh/descriptive-risk sources for primary raw endpoint:\n")
    for source_name in summary["high_or_descriptive_risk_sources_primary_raw"]:
        f.write(f"  - {source_name}\n")

    f.write("\nClaim boundary:\n")
    f.write("SMDG results are image-level/source-level external validation evidence only.\n")
    f.write("No SMDG training, threshold tuning, temperature fitting, PAPILA analysis, or patient-level claim is made.\n")

write_runtime_progress(
    {
        "status": "PASS",
        "phase": "completed",
        "n_bootstrap": N_BOOTSTRAP,
        "total_bootstrap_iterations": int(total_bootstrap_iterations),
        "completed_bootstrap_iterations": int(total_bootstrap_iterations),
        "percent_complete": 100.0,
        "current_model": None,
        "current_probability_type": None,
        "current_source": None,
        "elapsed_sec": elapsed_sec,
        "message": "Cell 6 completed successfully.",
        "summary_json": str(SMDG_CELL6_SUMMARY_JSON),
    }
)


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("\nSMDG global bootstrap CI:")
print(
    global_ci[
        global_ci["metric"].isin(
            [
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "ece_15",
            ]
        )
    ]
    .sort_values(["probability_type", "model_name", "metric"])
    .to_string(index=False)
)

print("\nSMDG native anatomical bootstrap CI:")
print(
    native_ci[
        native_ci["metric"].isin(
            [
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "ece_15",
            ]
        )
    ]
    .sort_values(["probability_type", "model_name", "metric"])
    .to_string(index=False)
)

print("\nSMDG paired model comparisons vs EfficientNetB0:")
print(
    paired_comparisons
    .sort_values(["probability_type", "candidate_model_name", "metric"])
    .to_string(index=False)
)

print("\nSMDG model ranking summary:")
print(ranking_pivot.to_string(index=False))

print("\nSMDG source support audit:")
print(source_support.to_string(index=False))

print("\nSMDG source-domain audit for primary model, raw probability:")
print(
    domain_shift_audit[
        domain_shift_audit["model_name"].eq(PRIMARY_MODEL_NAME)
        & domain_shift_audit["probability_type"].eq("raw_probability")
    ]
    .sort_values(["domain_shift_category", "external_source_dataset"])
    .to_string(index=False)
)

print("\nSMDG statistical claim boundaries:")
print(claim_boundaries.to_string(index=False))

print("\nCell 6 summary:")
print(json.dumps(summary, indent=2))

print(f"\nRuntime progress JSON: {SMDG_RUNTIME_PROGRESS_JSON}")
print(f"Global bootstrap CI CSV: {SMDG_BOOTSTRAP_GLOBAL_CI_CSV}")
print(f"Native bootstrap CI CSV: {SMDG_BOOTSTRAP_NATIVE_CI_CSV}")
print(f"Source bootstrap CI CSV: {SMDG_BOOTSTRAP_SOURCE_CI_CSV}")
print(f"Paired model comparison CSV: {SMDG_PAIRED_MODEL_COMPARISON_CSV}")
print(f"Model ranking CSV: {SMDG_MODEL_RANKING_CSV}")
print(f"Domain-shift audit CSV: {SMDG_DOMAIN_SHIFT_AUDIT_CSV}")
print(f"Source support audit CSV: {SMDG_SOURCE_SUPPORT_AUDIT_CSV}")
print(f"Statistical claim boundaries CSV: {SMDG_STATISTICAL_CLAIM_BOUNDARIES_CSV}")
print(f"Cell 6 summary JSON: {SMDG_CELL6_SUMMARY_JSON}")
print(f"Cell 6 summary TXT: {SMDG_CELL6_SUMMARY_TXT}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if summary["papila_present"]:
    raise RuntimeError("PAPILA leaked into Cell 6 statistical consolidation.")

if summary["eyepacs_glaucoma_present"]:
    raise RuntimeError("EyePACS-Glaucoma leaked into Cell 6 statistical consolidation.")

if len(global_ci) == 0:
    raise RuntimeError("Global bootstrap CI table is empty.")

if len(native_ci) == 0:
    raise RuntimeError("Native anatomical bootstrap CI table is empty.")

if len(source_ci) == 0:
    raise RuntimeError("Source bootstrap CI table is empty.")

if len(paired_comparisons) == 0:
    raise RuntimeError("Paired model comparison table is empty.")

if len(ranking_pivot) == 0:
    raise RuntimeError("Model ranking table is empty.")

if len(domain_shift_audit) == 0:
    raise RuntimeError("Source-domain audit table is empty.")

primary_raw_auroc = primary_summary_metrics["raw_auroc"]

if not primary_raw_auroc:
    raise RuntimeError("Primary EfficientNetB0 raw AUROC CI was not generated.")

if primary_raw_auroc["ci_low"] > primary_raw_auroc["point_estimate"]:
    raise RuntimeError("Invalid AUROC CI: lower bound exceeds point estimate.")

if primary_raw_auroc["ci_high"] < primary_raw_auroc["point_estimate"]:
    raise RuntimeError("Invalid AUROC CI: upper bound below point estimate.")

for required_output in expected_outputs:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 6 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 7: Expanded SMDG source-level calibration, reliability diagrams, and Notebook 16 handoff
#
# Purpose:
# - Do not recompute inference.
# - Do not recompute bootstrap.
# - Read Cell 5 predictions and Cell 6 bootstrap CIs.
# - Generate reviewer-facing per-source calibration tables:
#     raw vs temperature-scaled Brier score
#     raw vs temperature-scaled ECE-15
#     raw vs temperature-scaled AUROC where eligible
# - Generate reliability diagram bin data and figures:
#     pooled SMDG raw vs scaled
#     representative sources raw vs scaled
# - Produce a clean handoff JSON for Notebook 16 / manuscript v3.2.
#
# Reviewer concerns addressed:
# - "Calibration analyses are pooled; per-source calibration diagnostics would illuminate how calibration shifts with domain."
# - "Could you include per-source calibration plots before and after temperature scaling?"
# - "Report ECE/Brier per source before and after temperature scaling."

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = JOURNAL_ROOT / "results"
PROJECT_DATA_ROOT = JOURNAL_ROOT / "project_data"

RESULTS_13_DIR = RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation"
SMDG_PROJECT_DATA_DIR = PROJECT_DATA_ROOT / "smdg_aqpr_external_validation"

SMDG_EXTERNAL_INFERENCE_DIR = RESULTS_13_DIR / "smdg_external_inference"
SMDG_STATISTICS_DIR = RESULTS_13_DIR / "smdg_external_statistics"

SMDG_JOURNAL_PACKAGE_DIR = RESULTS_13_DIR / "smdg_journal_ready_package"
SMDG_TABLE_DIR = SMDG_JOURNAL_PACKAGE_DIR / "tables"
SMDG_FIGURE_DIR = SMDG_JOURNAL_PACKAGE_DIR / "figures"
SMDG_FIGURE_DATA_DIR = SMDG_JOURNAL_PACKAGE_DIR / "figure_data"

for d in [SMDG_JOURNAL_PACKAGE_DIR, SMDG_TABLE_DIR, SMDG_FIGURE_DIR, SMDG_FIGURE_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Inputs from Cell 5 and Cell 6
# ---------------------------------------------------------------------

SMDG_EXTERNAL_ALL_PREDICTIONS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_predictions_all_models.csv"
)

SMDG_EXTERNAL_SOURCE_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_source_level_metrics.csv"
)

SMDG_EXTERNAL_GLOBAL_METRICS_CSV = (
    SMDG_EXTERNAL_INFERENCE_DIR
    / "notebook13_smdg_external_global_metrics.csv"
)

SMDG_BOOTSTRAP_GLOBAL_CI_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_bootstrap_global_metrics_ci_5000.csv"
)

SMDG_BOOTSTRAP_SOURCE_CI_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_bootstrap_source_metrics_ci_5000.csv"
)

SMDG_DOMAIN_SHIFT_AUDIT_CSV = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_source_domain_shift_audit_5000.csv"
)

SMDG_CELL6_SUMMARY_JSON = (
    SMDG_STATISTICS_DIR
    / "notebook13_smdg_statistical_consolidation_summary_5000.json"
)


# ---------------------------------------------------------------------
# New outputs
# ---------------------------------------------------------------------

SMDG_TABLE_GLOBAL_CALIBRATION_CSV = (
    SMDG_TABLE_DIR
    / "table_smdg_global_calibration_raw_vs_scaled_all_models.csv"
)

SMDG_TABLE_GLOBAL_CALIBRATION_MD = (
    SMDG_TABLE_DIR
    / "table_smdg_global_calibration_raw_vs_scaled_all_models.md"
)

SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_CSV = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_all_models.csv"
)

SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_MD = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_all_models.md"
)

SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_CSV = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_primary_model.csv"
)

SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_MD = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_primary_model.md"
)

SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_COMPACT_CSV = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_primary_model_compact.csv"
)

SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_COMPACT_MD = (
    SMDG_TABLE_DIR
    / "table_smdg_source_calibration_raw_vs_scaled_primary_model_compact.md"
)

SMDG_TABLE_SOURCE_METRIC_ELIGIBILITY_CSV = (
    SMDG_TABLE_DIR
    / "table_smdg_source_metric_eligibility_and_calibration_support.csv"
)

SMDG_TABLE_SOURCE_METRIC_ELIGIBILITY_MD = (
    SMDG_TABLE_DIR
    / "table_smdg_source_metric_eligibility_and_calibration_support.md"
)

SMDG_FIGURE_DATA_RELIABILITY_BINS_CSV = (
    SMDG_FIGURE_DATA_DIR
    / "figure_data_smdg_reliability_diagram_bins_all_models.csv"
)

SMDG_FIGURE_DATA_SOURCE_CALIBRATION_DELTA_PRIMARY_CSV = (
    SMDG_FIGURE_DATA_DIR
    / "figure_data_smdg_source_calibration_delta_primary_model.csv"
)

SMDG_FIGURE_DATA_GLOBAL_CALIBRATION_DELTA_CSV = (
    SMDG_FIGURE_DATA_DIR
    / "figure_data_smdg_global_calibration_delta_all_models.csv"
)

SMDG_FIGURE_POOLED_RELIABILITY_PNG = (
    SMDG_FIGURE_DIR
    / "figure_smdg_reliability_pooled_raw_vs_scaled_primary_model.png"
)

SMDG_FIGURE_POOLED_RELIABILITY_PDF = (
    SMDG_FIGURE_DIR
    / "figure_smdg_reliability_pooled_raw_vs_scaled_primary_model.pdf"
)

SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PNG = (
    SMDG_FIGURE_DIR
    / "figure_smdg_reliability_representative_sources_raw_vs_scaled_primary_model.png"
)

SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PDF = (
    SMDG_FIGURE_DIR
    / "figure_smdg_reliability_representative_sources_raw_vs_scaled_primary_model.pdf"
)

SMDG_CALIBRATION_HANDOFF_JSON = (
    SMDG_JOURNAL_PACKAGE_DIR
    / "notebook13_smdg_calibration_handoff_for_notebook16.json"
)

SMDG_CALIBRATION_HANDOFF_TXT = (
    SMDG_JOURNAL_PACKAGE_DIR
    / "notebook13_SMDG_CALIBRATION_HANDOFF_FOR_NOTEBOOK16.txt"
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"

PRIMARY_PROBABILITY_TYPE = "raw_probability"
CALIBRATION_PROBABILITY_TYPE = "temperature_scaled_probability"

PROBABILITY_COLUMN_BY_TYPE = {
    PRIMARY_PROBABILITY_TYPE: "probability",
    CALIBRATION_PROBABILITY_TYPE: "temperature_scaled_probability",
}

PREDICTION_COLUMN_BY_TYPE = {
    PRIMARY_PROBABILITY_TYPE: "predicted_label",
    CALIBRATION_PROBABILITY_TYPE: "temperature_scaled_predicted_label",
}

PROBABILITY_LABEL_BY_TYPE = {
    PRIMARY_PROBABILITY_TYPE: "Raw",
    CALIBRATION_PROBABILITY_TYPE: "Temperature-scaled",
}

N_RELIABILITY_BINS = 15

MIN_SOURCE_POSITIVES_FOR_AUROC = 10
MIN_SOURCE_NEGATIVES_FOR_AUROC = 10

REPRESENTATIVE_SOURCE_PRIORITY = [
    "CRFO-v4",
    "G1020",
    "BEH",
    "OIA-ODIR-TRAIN",
    "REFUGE1-train",
]

# Use a serif font visually compatible with Elsevier/Times-like manuscript text.
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 200,
})


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required {label}: {path}\n"
            "If this file does not exist, run Cells 1-6 first."
        )
    if path.stat().st_size == 0:
        raise RuntimeError(f"Required {label} exists but is empty: {path}")


def read_csv(path, label):
    require_file(path, label)
    return pd.read_csv(path, low_memory=False)


def require_columns(df, required, label):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def save_csv_and_markdown(df, csv_path, md_path=None, max_markdown_rows=80):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

    if md_path is not None:
        md_path = Path(md_path)
        md_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            df.head(max_markdown_rows).to_markdown(md_path, index=False)
        except Exception:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(df.head(max_markdown_rows).to_string(index=False))

    return csv_path


def compute_ece(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(y_true) == 0:
        return np.nan

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for bin_start, bin_end in zip(bins[:-1], bins[1:]):
        if bin_end == 1.0:
            mask = (y_prob >= bin_start) & (y_prob <= bin_end)
        else:
            mask = (y_prob >= bin_start) & (y_prob < bin_end)

        if not np.any(mask):
            continue

        bin_confidence = float(np.mean(y_prob[mask]))
        bin_accuracy = float(np.mean(y_true[mask]))
        bin_weight = float(np.mean(mask))
        ece += bin_weight * abs(bin_accuracy - bin_confidence)

    return float(ece)


def brier_score(y_true, y_prob):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)

    if len(y_true) == 0:
        return np.nan

    return float(np.mean((y_prob - y_true) ** 2))


def safe_auroc(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    return float(roc_auc_score(y_true, y_prob))


def safe_auprc(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    return float(average_precision_score(y_true, y_prob))


def safe_balanced_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    if len(np.unique(y_true)) < 2:
        return np.nan

    return float(balanced_accuracy_score(y_true, y_pred))


def compute_point_metrics(df, probability_type):
    prob_col = PROBABILITY_COLUMN_BY_TYPE[probability_type]
    pred_col = PREDICTION_COLUMN_BY_TYPE[probability_type]

    y_true = df["external_label"].astype(int).to_numpy()
    y_prob = df[prob_col].astype(float).to_numpy()
    y_pred = df[pred_col].astype(int).to_numpy()

    n = int(len(df))
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    prevalence = float(n_pos / n) if n > 0 else np.nan

    metric_eligible = bool(
        n_pos >= MIN_SOURCE_POSITIVES_FOR_AUROC
        and n_neg >= MIN_SOURCE_NEGATIVES_FOR_AUROC
    )

    return {
        "n_images": n,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "prevalence": prevalence,
        "metric_eligible": metric_eligible,
        "auroc": safe_auroc(y_true, y_prob) if n > 0 else np.nan,
        "auprc": safe_auprc(y_true, y_prob) if n > 0 else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)) if n > 0 else np.nan,
        "balanced_accuracy": safe_balanced_accuracy(y_true, y_pred) if n > 0 else np.nan,
        "sensitivity": float(recall_score(y_true, y_pred, zero_division=0)) if n > 0 else np.nan,
        "specificity": _specificity(y_true, y_pred) if n > 0 else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)) if n > 0 else np.nan,
        "f1": float(f1_score(y_true, y_pred, zero_division=0)) if n > 0 else np.nan,
        "brier_score": brier_score(y_true, y_prob),
        "ece_15": compute_ece(y_true, y_prob, n_bins=N_RELIABILITY_BINS),
    }


def _specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    denom = tn + fp
    return float(tn / denom) if denom > 0 else np.nan


def reliability_bins(df, probability_type, source_scope, source_name, model_name, n_bins=15):
    prob_col = PROBABILITY_COLUMN_BY_TYPE[probability_type]

    y_true = df["external_label"].astype(int).to_numpy()
    y_prob = df[prob_col].astype(float).to_numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []

    for bin_id, (bin_start, bin_end) in enumerate(zip(bins[:-1], bins[1:])):
        if bin_end == 1.0:
            mask = (y_prob >= bin_start) & (y_prob <= bin_end)
        else:
            mask = (y_prob >= bin_start) & (y_prob < bin_end)

        n_bin = int(mask.sum())

        if n_bin > 0:
            mean_pred = float(np.mean(y_prob[mask]))
            observed = float(np.mean(y_true[mask]))
            gap = float(abs(observed - mean_pred))
        else:
            mean_pred = np.nan
            observed = np.nan
            gap = np.nan

        rows.append({
            "model_name": model_name,
            "source_scope": source_scope,
            "external_source_dataset": source_name,
            "probability_type": probability_type,
            "probability_label": PROBABILITY_LABEL_BY_TYPE[probability_type],
            "bin_id": int(bin_id),
            "bin_low": float(bin_start),
            "bin_high": float(bin_end),
            "n_bin": n_bin,
            "mean_predicted_probability": mean_pred,
            "observed_positive_fraction": observed,
            "absolute_calibration_gap": gap,
        })

    return rows


def get_ci_row(ci_df, model_name, probability_type, metric, source=None, analysis_scope=None):
    if ci_df is None or len(ci_df) == 0:
        return None

    query = (
        ci_df["model_name"].astype(str).eq(str(model_name))
        & ci_df["probability_type"].astype(str).eq(str(probability_type))
        & ci_df["metric"].astype(str).eq(str(metric))
    )

    if source is not None and "external_source_dataset" in ci_df.columns:
        query &= ci_df["external_source_dataset"].astype(str).eq(str(source))

    if analysis_scope is not None and "analysis_scope" in ci_df.columns:
        query &= ci_df["analysis_scope"].astype(str).eq(str(analysis_scope))

    block = ci_df[query].copy()

    if len(block) == 0:
        return None

    return block.iloc[0].to_dict()


def ci_value(ci_df, model_name, probability_type, metric, source=None, analysis_scope=None, field="point_estimate"):
    row = get_ci_row(
        ci_df=ci_df,
        model_name=model_name,
        probability_type=probability_type,
        metric=metric,
        source=source,
        analysis_scope=analysis_scope,
    )

    if row is None:
        return np.nan

    return row.get(field, np.nan)


def ci_string(ci_df, model_name, probability_type, metric, source=None, analysis_scope=None, decimals=3):
    row = get_ci_row(
        ci_df=ci_df,
        model_name=model_name,
        probability_type=probability_type,
        metric=metric,
        source=source,
        analysis_scope=analysis_scope,
    )

    if row is None:
        return "NA"

    point = row.get("point_estimate", np.nan)
    lo = row.get("ci_low", np.nan)
    hi = row.get("ci_high", np.nan)

    if not np.isfinite(point):
        return "NA"

    if np.isfinite(lo) and np.isfinite(hi):
        return f"{point:.{decimals}f} [{lo:.{decimals}f}, {hi:.{decimals}f}]"

    return f"{point:.{decimals}f} [CI NA]"


def fmt_float(x, decimals=3):
    try:
        x = float(x)
    except Exception:
        return "NA"

    if not np.isfinite(x):
        return "NA"

    return f"{x:.{decimals}f}"


def plot_reliability(ax, bins_df, title):
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0, label="Perfect")

    for probability_type in [PRIMARY_PROBABILITY_TYPE, CALIBRATION_PROBABILITY_TYPE]:
        block = bins_df[
            bins_df["probability_type"].astype(str).eq(probability_type)
            & bins_df["n_bin"].fillna(0).astype(int).gt(0)
        ].copy()

        if len(block) == 0:
            continue

        block = block.sort_values("bin_id")
        ax.plot(
            block["mean_predicted_probability"],
            block["observed_positive_fraction"],
            marker="o",
            linewidth=1.0,
            markersize=3.0,
            label=PROBABILITY_LABEL_BY_TYPE[probability_type],
        )

    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed GON+ fraction")
    ax.grid(True, linewidth=0.3, alpha=0.5)
    ax.legend(frameon=False)


# ---------------------------------------------------------------------
# Input validation and loading
# ---------------------------------------------------------------------

required_inputs = [
    (SMDG_EXTERNAL_ALL_PREDICTIONS_CSV, "Cell 5 all-model predictions"),
    (SMDG_BOOTSTRAP_GLOBAL_CI_CSV, "Cell 6 global bootstrap CI"),
    (SMDG_BOOTSTRAP_SOURCE_CI_CSV, "Cell 6 source-level bootstrap CI"),
]

for path, label in required_inputs:
    require_file(path, label)

predictions = read_csv(SMDG_EXTERNAL_ALL_PREDICTIONS_CSV, "SMDG predictions")
global_ci = read_csv(SMDG_BOOTSTRAP_GLOBAL_CI_CSV, "SMDG global bootstrap CI")
source_ci = read_csv(SMDG_BOOTSTRAP_SOURCE_CI_CSV, "SMDG source bootstrap CI")

source_metrics = (
    pd.read_csv(SMDG_EXTERNAL_SOURCE_METRICS_CSV, low_memory=False)
    if SMDG_EXTERNAL_SOURCE_METRICS_CSV.exists()
    else pd.DataFrame()
)

domain_shift_audit = (
    pd.read_csv(SMDG_DOMAIN_SHIFT_AUDIT_CSV, low_memory=False)
    if SMDG_DOMAIN_SHIFT_AUDIT_CSV.exists()
    else pd.DataFrame()
)

require_columns(
    predictions,
    [
        "model_name",
        "external_sample_id",
        "external_source_dataset",
        "external_label",
        "probability",
        "temperature_scaled_probability",
        "predicted_label",
        "temperature_scaled_predicted_label",
    ],
    "predictions",
)

require_columns(
    global_ci,
    [
        "analysis_scope",
        "model_name",
        "probability_type",
        "metric",
        "point_estimate",
        "ci_low",
        "ci_high",
    ],
    "global_ci",
)

require_columns(
    source_ci,
    [
        "analysis_scope",
        "external_source_dataset",
        "model_name",
        "probability_type",
        "metric",
        "point_estimate",
        "ci_low",
        "ci_high",
    ],
    "source_ci",
)

predictions["model_name"] = predictions["model_name"].astype(str)
predictions["external_source_dataset"] = predictions["external_source_dataset"].astype(str)
predictions["external_label"] = predictions["external_label"].astype(int)

available_models = sorted(predictions["model_name"].unique().tolist())

if PRIMARY_MODEL_NAME not in available_models:
    warnings.warn(
        f"Primary model {PRIMARY_MODEL_NAME} not found. "
        f"Using first available model: {available_models[0]}"
    )
    PRIMARY_MODEL_NAME = available_models[0]

available_sources = sorted(predictions["external_source_dataset"].unique().tolist())


# ---------------------------------------------------------------------
# Direct global point metrics raw vs scaled
# ---------------------------------------------------------------------

global_metric_rows = []

for model_name in available_models:
    model_df = predictions[predictions["model_name"].eq(model_name)].copy()

    for probability_type in [PRIMARY_PROBABILITY_TYPE, CALIBRATION_PROBABILITY_TYPE]:
        point = compute_point_metrics(model_df, probability_type)

        row = {
            "model_name": model_name,
            "source_scope": "pooled_smdg",
            "external_source_dataset": "pooled_smdg",
            "probability_type": probability_type,
            "probability_label": PROBABILITY_LABEL_BY_TYPE[probability_type],
            **point,
        }

        for metric in ["auroc", "balanced_accuracy", "brier_score", "ece_15"]:
            row[f"{metric}_ci_string"] = ci_string(
                global_ci,
                model_name=model_name,
                probability_type=probability_type,
                metric=metric,
                analysis_scope="primary_external_global",
                decimals=4 if metric in ["brier_score", "ece_15"] else 3,
            )
            row[f"{metric}_ci_low"] = ci_value(
                global_ci,
                model_name=model_name,
                probability_type=probability_type,
                metric=metric,
                analysis_scope="primary_external_global",
                field="ci_low",
            )
            row[f"{metric}_ci_high"] = ci_value(
                global_ci,
                model_name=model_name,
                probability_type=probability_type,
                metric=metric,
                analysis_scope="primary_external_global",
                field="ci_high",
            )

        global_metric_rows.append(row)

global_calibration_long = pd.DataFrame(global_metric_rows)


# Global raw vs scaled deltas by model
global_delta_rows = []

for model_name in available_models:
    raw = global_calibration_long[
        global_calibration_long["model_name"].eq(model_name)
        & global_calibration_long["probability_type"].eq(PRIMARY_PROBABILITY_TYPE)
    ]

    scaled = global_calibration_long[
        global_calibration_long["model_name"].eq(model_name)
        & global_calibration_long["probability_type"].eq(CALIBRATION_PROBABILITY_TYPE)
    ]

    if len(raw) != 1 or len(scaled) != 1:
        continue

    raw = raw.iloc[0]
    scaled = scaled.iloc[0]

    global_delta_rows.append({
        "model_name": model_name,
        "n_images": int(raw["n_images"]),
        "prevalence": float(raw["prevalence"]),
        "raw_auroc": float(raw["auroc"]) if np.isfinite(raw["auroc"]) else np.nan,
        "scaled_auroc": float(scaled["auroc"]) if np.isfinite(scaled["auroc"]) else np.nan,
        "delta_auroc_scaled_minus_raw": float(scaled["auroc"] - raw["auroc"]) if np.isfinite(raw["auroc"]) and np.isfinite(scaled["auroc"]) else np.nan,
        "raw_brier_score": float(raw["brier_score"]),
        "scaled_brier_score": float(scaled["brier_score"]),
        "delta_brier_scaled_minus_raw": float(scaled["brier_score"] - raw["brier_score"]),
        "raw_ece_15": float(raw["ece_15"]),
        "scaled_ece_15": float(scaled["ece_15"]),
        "delta_ece_15_scaled_minus_raw": float(scaled["ece_15"] - raw["ece_15"]),
        "calibration_effect": (
            "improved_brier_and_ece"
            if (scaled["brier_score"] < raw["brier_score"] and scaled["ece_15"] < raw["ece_15"])
            else "mixed_or_worse"
        ),
    })

global_calibration_delta = pd.DataFrame(global_delta_rows)

save_csv_and_markdown(
    global_calibration_delta,
    SMDG_TABLE_GLOBAL_CALIBRATION_CSV,
    SMDG_TABLE_GLOBAL_CALIBRATION_MD,
)

global_calibration_delta.to_csv(
    SMDG_FIGURE_DATA_GLOBAL_CALIBRATION_DELTA_CSV,
    index=False,
)


# ---------------------------------------------------------------------
# Source-level point metrics raw vs scaled for all models
# ---------------------------------------------------------------------

source_metric_rows = []

for model_name in available_models:
    model_df = predictions[predictions["model_name"].eq(model_name)].copy()

    for source_name, source_df in model_df.groupby("external_source_dataset", dropna=False):
        source_name = str(source_name)

        for probability_type in [PRIMARY_PROBABILITY_TYPE, CALIBRATION_PROBABILITY_TYPE]:
            point = compute_point_metrics(source_df, probability_type)

            row = {
                "model_name": model_name,
                "external_source_dataset": source_name,
                "probability_type": probability_type,
                "probability_label": PROBABILITY_LABEL_BY_TYPE[probability_type],
                **point,
            }

            for metric in ["auroc", "balanced_accuracy", "brier_score", "ece_15"]:
                row[f"{metric}_ci_string"] = ci_string(
                    source_ci,
                    model_name=model_name,
                    probability_type=probability_type,
                    metric=metric,
                    source=source_name,
                    analysis_scope="source_level",
                    decimals=4 if metric in ["brier_score", "ece_15"] else 3,
                )
                row[f"{metric}_ci_low"] = ci_value(
                    source_ci,
                    model_name=model_name,
                    probability_type=probability_type,
                    metric=metric,
                    source=source_name,
                    analysis_scope="source_level",
                    field="ci_low",
                )
                row[f"{metric}_ci_high"] = ci_value(
                    source_ci,
                    model_name=model_name,
                    probability_type=probability_type,
                    metric=metric,
                    source=source_name,
                    analysis_scope="source_level",
                    field="ci_high",
                )

            source_metric_rows.append(row)

source_calibration_long = pd.DataFrame(source_metric_rows)

save_csv_and_markdown(
    source_calibration_long,
    SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_CSV,
    SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_MD,
)


# ---------------------------------------------------------------------
# Source-level raw vs scaled deltas
# ---------------------------------------------------------------------

source_delta_rows = []

for (model_name, source_name), group in source_calibration_long.groupby(
    ["model_name", "external_source_dataset"],
    dropna=False,
):
    raw = group[group["probability_type"].eq(PRIMARY_PROBABILITY_TYPE)]
    scaled = group[group["probability_type"].eq(CALIBRATION_PROBABILITY_TYPE)]

    if len(raw) != 1 or len(scaled) != 1:
        continue

    raw = raw.iloc[0]
    scaled = scaled.iloc[0]

    source_delta_rows.append({
        "model_name": model_name,
        "external_source_dataset": source_name,
        "n_images": int(raw["n_images"]),
        "n_positive": int(raw["n_positive"]),
        "n_negative": int(raw["n_negative"]),
        "prevalence": float(raw["prevalence"]),
        "metric_eligible": bool(raw["metric_eligible"]),
        "raw_auroc": float(raw["auroc"]) if np.isfinite(raw["auroc"]) else np.nan,
        "scaled_auroc": float(scaled["auroc"]) if np.isfinite(scaled["auroc"]) else np.nan,
        "delta_auroc_scaled_minus_raw": (
            float(scaled["auroc"] - raw["auroc"])
            if np.isfinite(raw["auroc"]) and np.isfinite(scaled["auroc"])
            else np.nan
        ),
        "raw_balanced_accuracy": float(raw["balanced_accuracy"]) if np.isfinite(raw["balanced_accuracy"]) else np.nan,
        "scaled_balanced_accuracy": float(scaled["balanced_accuracy"]) if np.isfinite(scaled["balanced_accuracy"]) else np.nan,
        "raw_brier_score": float(raw["brier_score"]),
        "scaled_brier_score": float(scaled["brier_score"]),
        "delta_brier_scaled_minus_raw": float(scaled["brier_score"] - raw["brier_score"]),
        "raw_ece_15": float(raw["ece_15"]),
        "scaled_ece_15": float(scaled["ece_15"]),
        "delta_ece_15_scaled_minus_raw": float(scaled["ece_15"] - raw["ece_15"]),
        "raw_auroc_ci": raw["auroc_ci_string"],
        "scaled_auroc_ci": scaled["auroc_ci_string"],
        "raw_brier_ci": raw["brier_score_ci_string"],
        "scaled_brier_ci": scaled["brier_score_ci_string"],
        "raw_ece_15_ci": raw["ece_15_ci_string"],
        "scaled_ece_15_ci": scaled["ece_15_ci_string"],
        "calibration_effect": (
            "improved_brier_and_ece"
            if (scaled["brier_score"] < raw["brier_score"] and scaled["ece_15"] < raw["ece_15"])
            else "improved_brier_only"
            if (scaled["brier_score"] < raw["brier_score"] and scaled["ece_15"] >= raw["ece_15"])
            else "improved_ece_only"
            if (scaled["brier_score"] >= raw["brier_score"] and scaled["ece_15"] < raw["ece_15"])
            else "not_improved"
        ),
    })

source_calibration_delta = pd.DataFrame(source_delta_rows)

save_csv_and_markdown(
    source_calibration_delta,
    SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_CSV,
    SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_MD,
)

primary_source_calibration = source_calibration_delta[
    source_calibration_delta["model_name"].astype(str).eq(PRIMARY_MODEL_NAME)
].copy()

primary_source_calibration = primary_source_calibration.sort_values(
    by=["metric_eligible", "raw_auroc", "delta_ece_15_scaled_minus_raw"],
    ascending=[False, True, False],
    na_position="last",
).reset_index(drop=True)

save_csv_and_markdown(
    primary_source_calibration,
    SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_CSV,
    SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_MD,
)

primary_compact = primary_source_calibration.copy()
primary_compact["Prev."] = primary_compact["prevalence"].apply(lambda x: fmt_float(x, 3))
primary_compact["Raw AUROC"] = primary_compact["raw_auroc_ci"]
primary_compact["Raw Brier"] = primary_compact["raw_brier_ci"]
primary_compact["Scaled Brier"] = primary_compact["scaled_brier_ci"]
primary_compact["Delta Brier"] = primary_compact["delta_brier_scaled_minus_raw"].apply(lambda x: fmt_float(x, 4))
primary_compact["Raw ECE-15"] = primary_compact["raw_ece_15_ci"]
primary_compact["Scaled ECE-15"] = primary_compact["scaled_ece_15_ci"]
primary_compact["Delta ECE-15"] = primary_compact["delta_ece_15_scaled_minus_raw"].apply(lambda x: fmt_float(x, 4))
primary_compact["Eligible"] = primary_compact["metric_eligible"].astype(bool)

primary_compact = primary_compact[
    [
        "external_source_dataset",
        "n_images",
        "Prev.",
        "Eligible",
        "Raw AUROC",
        "Raw Brier",
        "Scaled Brier",
        "Delta Brier",
        "Raw ECE-15",
        "Scaled ECE-15",
        "Delta ECE-15",
        "calibration_effect",
    ]
].rename(
    columns={
        "external_source_dataset": "Source",
        "n_images": "Images",
        "calibration_effect": "Calibration effect",
    }
)

save_csv_and_markdown(
    primary_compact,
    SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_COMPACT_CSV,
    SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_COMPACT_MD,
)

primary_source_calibration.to_csv(
    SMDG_FIGURE_DATA_SOURCE_CALIBRATION_DELTA_PRIMARY_CSV,
    index=False,
)


# ---------------------------------------------------------------------
# Source eligibility / support table
# ---------------------------------------------------------------------

eligibility_rows = []

for source_name, source_df in predictions[
    predictions["model_name"].eq(PRIMARY_MODEL_NAME)
].groupby("external_source_dataset", dropna=False):
    y = source_df["external_label"].astype(int).to_numpy()

    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())

    eligible_for_auc = bool(
        n_pos >= MIN_SOURCE_POSITIVES_FOR_AUROC
        and n_neg >= MIN_SOURCE_NEGATIVES_FOR_AUROC
    )

    has_bootstrap_ci = bool(
        len(
            source_ci[
                source_ci["external_source_dataset"].astype(str).eq(str(source_name))
                & source_ci["model_name"].astype(str).eq(str(PRIMARY_MODEL_NAME))
            ]
        )
        > 0
    )

    eligibility_rows.append({
        "external_source_dataset": str(source_name),
        "n_images": n,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "prevalence": float(n_pos / n) if n > 0 else np.nan,
        "eligible_for_auroc_and_bootstrap_ci": eligible_for_auc,
        "has_source_bootstrap_ci": has_bootstrap_ci,
        "calibration_points_available": bool(n > 0),
        "interpretation": (
            "two-class source with bootstrap CI"
            if eligible_for_auc and has_bootstrap_ci
            else "descriptive calibration only; source metric CI limited by class count"
            if n > 0
            else "no usable prediction rows"
        ),
    })

source_eligibility = pd.DataFrame(eligibility_rows).sort_values(
    by=["eligible_for_auroc_and_bootstrap_ci", "n_images"],
    ascending=[False, False],
).reset_index(drop=True)

save_csv_and_markdown(
    source_eligibility,
    SMDG_TABLE_SOURCE_METRIC_ELIGIBILITY_CSV,
    SMDG_TABLE_SOURCE_METRIC_ELIGIBILITY_MD,
)


# ---------------------------------------------------------------------
# Reliability diagram bin data
# ---------------------------------------------------------------------

bin_rows = []

for model_name in available_models:
    model_df = predictions[predictions["model_name"].eq(model_name)].copy()

    for probability_type in [PRIMARY_PROBABILITY_TYPE, CALIBRATION_PROBABILITY_TYPE]:
        bin_rows.extend(
            reliability_bins(
                df=model_df,
                probability_type=probability_type,
                source_scope="pooled_smdg",
                source_name="pooled_smdg",
                model_name=model_name,
                n_bins=N_RELIABILITY_BINS,
            )
        )

        for source_name, source_df in model_df.groupby("external_source_dataset", dropna=False):
            bin_rows.extend(
                reliability_bins(
                    df=source_df,
                    probability_type=probability_type,
                    source_scope="source_level",
                    source_name=str(source_name),
                    model_name=model_name,
                    n_bins=N_RELIABILITY_BINS,
                )
            )

reliability_bin_data = pd.DataFrame(bin_rows)
reliability_bin_data.to_csv(SMDG_FIGURE_DATA_RELIABILITY_BINS_CSV, index=False)


# ---------------------------------------------------------------------
# Reliability diagram figures
# ---------------------------------------------------------------------

primary_bins = reliability_bin_data[
    reliability_bin_data["model_name"].astype(str).eq(PRIMARY_MODEL_NAME)
].copy()

# Pooled SMDG raw vs temperature-scaled reliability diagram.
fig, ax = plt.subplots(figsize=(3.25, 3.10))
pooled_bins = primary_bins[
    primary_bins["source_scope"].eq("pooled_smdg")
    & primary_bins["external_source_dataset"].eq("pooled_smdg")
].copy()

plot_reliability(
    ax=ax,
    bins_df=pooled_bins,
    title=f"{PRIMARY_MODEL_NAME}: pooled SMDG",
)

fig.tight_layout()
fig.savefig(SMDG_FIGURE_POOLED_RELIABILITY_PNG, bbox_inches="tight")
fig.savefig(SMDG_FIGURE_POOLED_RELIABILITY_PDF, bbox_inches="tight")
plt.show()


# Representative source reliability diagrams.
present_sources = set(primary_bins["external_source_dataset"].astype(str).unique())

representative_sources = [
    s for s in REPRESENTATIVE_SOURCE_PRIORITY
    if s in present_sources
]

# If fewer than 3 priority sources are present, add sources with largest absolute ECE delta.
if len(representative_sources) < 3 and len(primary_source_calibration) > 0:
    candidates = (
        primary_source_calibration.assign(
            abs_delta_ece=lambda d: d["delta_ece_15_scaled_minus_raw"].abs()
        )
        .sort_values("abs_delta_ece", ascending=False)
        ["external_source_dataset"]
        .astype(str)
        .tolist()
    )

    for s in candidates:
        if s not in representative_sources and s in present_sources:
            representative_sources.append(s)
        if len(representative_sources) >= 5:
            break

representative_sources = representative_sources[:5]

if len(representative_sources) > 0:
    n_sources = len(representative_sources)
    fig_height = max(2.4, 2.05 * n_sources)
    fig, axes = plt.subplots(
        nrows=n_sources,
        ncols=1,
        figsize=(3.35, fig_height),
        squeeze=False,
    )

    for row_idx, source_name in enumerate(representative_sources):
        ax = axes[row_idx, 0]
        source_bins = primary_bins[
            primary_bins["source_scope"].eq("source_level")
            & primary_bins["external_source_dataset"].astype(str).eq(str(source_name))
        ].copy()

        n_source = int(
            predictions[
                predictions["model_name"].eq(PRIMARY_MODEL_NAME)
                & predictions["external_source_dataset"].astype(str).eq(str(source_name))
            ].shape[0]
        )

        plot_reliability(
            ax=ax,
            bins_df=source_bins,
            title=f"{source_name} (n={n_source})",
        )

    fig.tight_layout()
    fig.savefig(SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PNG, bbox_inches="tight")
    fig.savefig(SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PDF, bbox_inches="tight")
    plt.show()
else:
    print("No representative source reliability figure was generated because no source bins were available.")


# ---------------------------------------------------------------------
# Console summaries for reviewer-facing interpretation
# ---------------------------------------------------------------------

print("\nGlobal calibration raw vs temperature-scaled")
display(global_calibration_delta)

print(f"\nPrimary model source-level calibration: {PRIMARY_MODEL_NAME}")
display(primary_compact)

print("\nSource metric eligibility/support")
display(source_eligibility)

print("\nRepresentative reliability sources")
print(representative_sources)


# ---------------------------------------------------------------------
# Handoff for Notebook 16 and manuscript v3.2
# ---------------------------------------------------------------------

primary_global_row = global_calibration_delta[
    global_calibration_delta["model_name"].astype(str).eq(PRIMARY_MODEL_NAME)
]

if len(primary_global_row) == 1:
    primary_global_row = primary_global_row.iloc[0].to_dict()
else:
    primary_global_row = {}

high_domain_shift_sources = []
if len(domain_shift_audit) > 0:
    if "domain_shift_category" in domain_shift_audit.columns:
        block = domain_shift_audit[
            domain_shift_audit["model_name"].astype(str).eq(str(PRIMARY_MODEL_NAME))
            & domain_shift_audit["probability_type"].astype(str).eq(PRIMARY_PROBABILITY_TYPE)
        ].copy()

        risky = block[
            block["domain_shift_category"]
            .astype(str)
            .str.contains("risk|single_class|metric_unavailable|high", case=False, na=False)
        ]

        high_domain_shift_sources = sorted(
            risky["external_source_dataset"].astype(str).unique().tolist()
        )

handoff = {
    "notebook": "13_external_validation_dataset_selection_and_preparation",
    "cell": "Cell 7",
    "status": "PASS",
    "purpose": (
        "Expanded SMDG source-level calibration reporting with raw vs temperature-scaled "
        "Brier/ECE, reliability diagram bins, and representative reliability figures."
    ),
    "primary_model_name": PRIMARY_MODEL_NAME,
    "available_models": available_models,
    "n_prediction_rows": int(len(predictions)),
    "n_unique_images": int(predictions["external_sample_id"].nunique()),
    "n_sources": int(len(available_sources)),
    "available_sources": available_sources,
    "representative_reliability_sources": representative_sources,
    "global_primary_calibration": {
        "raw_brier_score": primary_global_row.get("raw_brier_score", None),
        "scaled_brier_score": primary_global_row.get("scaled_brier_score", None),
        "delta_brier_scaled_minus_raw": primary_global_row.get("delta_brier_scaled_minus_raw", None),
        "raw_ece_15": primary_global_row.get("raw_ece_15", None),
        "scaled_ece_15": primary_global_row.get("scaled_ece_15", None),
        "delta_ece_15_scaled_minus_raw": primary_global_row.get("delta_ece_15_scaled_minus_raw", None),
        "calibration_effect": primary_global_row.get("calibration_effect", None),
    },
    "source_calibration_summary": {
        "n_primary_source_rows": int(len(primary_source_calibration)),
        "n_sources_improved_brier_and_ece": int(
            primary_source_calibration["calibration_effect"].eq("improved_brier_and_ece").sum()
        ) if len(primary_source_calibration) else 0,
        "n_sources_not_improved": int(
            primary_source_calibration["calibration_effect"].eq("not_improved").sum()
        ) if len(primary_source_calibration) else 0,
        "mean_delta_brier_scaled_minus_raw": float(
            primary_source_calibration["delta_brier_scaled_minus_raw"].mean()
        ) if len(primary_source_calibration) else None,
        "mean_delta_ece_scaled_minus_raw": float(
            primary_source_calibration["delta_ece_15_scaled_minus_raw"].mean()
        ) if len(primary_source_calibration) else None,
    },
    "high_or_descriptive_risk_sources_primary_raw": high_domain_shift_sources,
    "input_tables": {
        "predictions": str(SMDG_EXTERNAL_ALL_PREDICTIONS_CSV),
        "global_bootstrap_ci": str(SMDG_BOOTSTRAP_GLOBAL_CI_CSV),
        "source_bootstrap_ci": str(SMDG_BOOTSTRAP_SOURCE_CI_CSV),
        "source_metrics": str(SMDG_EXTERNAL_SOURCE_METRICS_CSV),
        "domain_shift_audit": str(SMDG_DOMAIN_SHIFT_AUDIT_CSV),
    },
    "output_tables": {
        "global_calibration": str(SMDG_TABLE_GLOBAL_CALIBRATION_CSV),
        "source_calibration_all_models": str(SMDG_TABLE_SOURCE_CALIBRATION_ALL_MODELS_CSV),
        "source_calibration_primary": str(SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_CSV),
        "source_calibration_primary_compact": str(SMDG_TABLE_SOURCE_CALIBRATION_PRIMARY_COMPACT_CSV),
        "source_metric_eligibility": str(SMDG_TABLE_SOURCE_METRIC_ELIGIBILITY_CSV),
        "reliability_bins": str(SMDG_FIGURE_DATA_RELIABILITY_BINS_CSV),
        "source_calibration_delta_primary": str(SMDG_FIGURE_DATA_SOURCE_CALIBRATION_DELTA_PRIMARY_CSV),
        "global_calibration_delta": str(SMDG_FIGURE_DATA_GLOBAL_CALIBRATION_DELTA_CSV),
    },
    "output_figures": {
        "pooled_reliability_png": str(SMDG_FIGURE_POOLED_RELIABILITY_PNG),
        "pooled_reliability_pdf": str(SMDG_FIGURE_POOLED_RELIABILITY_PDF),
        "representative_source_reliability_png": str(SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PNG),
        "representative_source_reliability_pdf": str(SMDG_FIGURE_REPRESENTATIVE_SOURCE_RELIABILITY_PDF),
    },
    "claim_boundary": (
        "Temperature scaling was fitted only on AIROGS-light validation and applied frozen to SMDG. "
        "Per-source calibration is a source-level transportability diagnostic, not source-specific recalibration."
    ),
}

with open(SMDG_CALIBRATION_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

with open(SMDG_CALIBRATION_HANDOFF_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 13 SMDG calibration handoff for Notebook 16\n")
    f.write("=" * 72 + "\n\n")
    f.write(f"Status: {handoff['status']}\n")
    f.write(f"Primary model: {PRIMARY_MODEL_NAME}\n")
    f.write(f"Prediction rows: {handoff['n_prediction_rows']}\n")
    f.write(f"Unique images: {handoff['n_unique_images']}\n")
    f.write(f"Sources: {handoff['n_sources']}\n\n")
    f.write("Global primary calibration:\n")
    for key, value in handoff["global_primary_calibration"].items():
        f.write(f"  - {key}: {value}\n")
    f.write("\nSource calibration summary:\n")
    for key, value in handoff["source_calibration_summary"].items():
        f.write(f"  - {key}: {value}\n")
    f.write("\nRepresentative reliability sources:\n")
    for source in representative_sources:
        f.write(f"  - {source}\n")
    f.write("\nClaim boundary:\n")
    f.write(handoff["claim_boundary"] + "\n")

print("\nCell 7 completed.")
print(json.dumps(handoff, indent=2))